<a id="ace-finqa-notebook-intro"></a>
# ACE-FinQA playbook experiment

Builds and evaluates an adaptive ACE playbook for Qwen3-8B on FinQA.

> **Status:** Colab-first experiment. Read [`docs/reproducibility.md`](../docs/reproducibility.md) before running. Notebook metrics are run diagnostics. The canonical project result is published in [`results/report.md`](../results/report.md) from the thesis record.

Run cells from top to bottom in a fresh runtime. Never commit outputs, widget state, credentials, or downloaded weights.


## Environment setup


In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv

if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo>=2025.10.12" \
        "unsloth==2025.10.11"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq "unsloth==2025.10.11"

!uv pip install --upgrade --no-deps tokenizers trl==0.22.2

# Keep Transformers aligned with the baseline environment.
!uv pip install "transformers==4.55.4"

# Keep vLLM aligned with the Unsloth environment.
!uv pip install "vllm==0.10.2"

!uv pip install sentence-transformers openai rank-bm25==0.2.2
!uv pip uninstall -qqq torchcodec 2>/dev/null || true


## Experiment configuration


In [ ]:
import os, json, re, time, glob, random, shutil
import numpy as np
from datetime import datetime
from typing import List, Dict, Any, Tuple, Optional
from collections import Counter

from google.colab import drive
drive.mount('/content/drive', force_remount=False)


### MODEL


In [ ]:
MODEL_TAG  = "qwen3_8b"
MODEL_NAME = "unsloth/Qwen3-8B-bnb-4bit"
RANDOM_SEED = 42

# 2. HYBRID CONFIG
USE_HYBRID_REFLECTOR    = True
USE_HYBRID_CURATOR      = False
HYBRID_MODEL_REFLECTOR  = "gpt-4o-mini"

if USE_HYBRID_REFLECTOR or USE_HYBRID_CURATOR:
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        print("[HYBRID] ✅ API key from Colab secrets")
    except:
        os.environ["OPENAI_API_KEY"] = ""
    HYBRID_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    if not HYBRID_API_KEY:
        raise RuntimeError("[ERROR] OPENAI_API_KEY missing")
    print(f"[HYBRID] Reflector={USE_HYBRID_REFLECTOR} | Curator={USE_HYBRID_CURATOR}")

# 3. GPU CONFIG
import subprocess
_gpu_name = subprocess.getoutput(
    "nvidia-smi --query-gpu=name --format=csv,noheader").strip().upper()

GPU_MEM_UTIL   = 0.90
MAX_SEQ_LENGTH = 4096
_BATCH_HINT    = 128
print(f"[GPU] Detected: {_gpu_name}")
print(f"[GPU] MEM_UTIL={GPU_MEM_UTIL} | MAX_SEQ={MAX_SEQ_LENGTH} | BATCH_HINT={_BATCH_HINT}")


### INFERENCE PARAMS


In [ ]:
TEMPERATURE        = 0.0
MAX_TOKENS         = 1024  # generator only needs ~50-700 tok
REPETITION_PENALTY = 1.1

# 5. ACE PIPELINE PARAMS
TOP_K_RETRIEVAL_TIER1 = 4  # Tier 1 always-retrieved
TOP_K_RETRIEVAL_TIER2 = 3  # Tier 2 dynamic top-3
TOP_K_RETRIEVAL       = TOP_K_RETRIEVAL_TIER1 + TOP_K_RETRIEVAL_TIER2  # = 7
CURATOR_EVERY         = 1
NUM_EPOCHS            = 1
EA_DECIMAL_PLACES     = 5

QG_MIN_LEN         = 30
QG_MAX_LEN         = 220
QG_DEDUP_THRESH    = 0.85
QG_OVERLAP_THRESH  = 0.80

# 6. EVAL
EVAL_EVERY    = 60
EVAL_DEV_SIZE = 883

# 7. DATA + CHECKPOINT
CHECKPOINT_EVERY = 10
TRAIN_SUBSET     = 600
DEV_SUBSET       = 883

# 8. COMPOSITE SELECTION (EA-priority)
USE_PA_GUARD       = True
PA_GUARD_TOLERANCE = 0.02
EA_IMPROVEMENT_MIN = 0.0

USE_PA_TIEBREAK     = True
COMPOSITE_SELECTION = True
COMPOSITE_EA_WEIGHT = 0.60
COMPOSITE_PA_WEIGHT = 0.40

# 9. PLAYBOOK CAPACITY
MAX_PLAYBOOK_BULLETS  = 30  # 5 Tier1 + 25 Tier2
ENFORCE_BULLET_BUDGET = True
BULLET_MIN_AGE_STEPS  = 80  # for budget eviction
BULLET_EVICT_SCORE    = 0

# 10. PA-AWARE / OUTCOME FLAGS
REFLECTOR_TEMP = 0.0

USE_LUCKY_GUESS_REFLECT    = False  # skip Reflector entirely
USE_LUCKY_GUESS_ADD_BULLET = False  # don't add lucky bullets
USE_EXEC_MISMATCH_BRANCH   = False
USE_CLOSE_BUT_WRONG        = False
USE_SEMANTIC_DEDUP         = True
USE_OVERLAP_DEDUP          = True


### TIER 1/2 SYSTEM


In [ ]:
USE_TIER_SYSTEM        = True
TIER_1_MAX             = 5
TIER_1_PROMOTION_AGE   = 200  # bullet must live ≥ 200 steps
TIER_1_PROMOTION_LIFT  = 0.01  # AND have dev pa_lift > 0.01
TIER_1_FALLBACK_AGE    = 350  # if Tier1 < 3 by step 350, relax
TIER_1_FALLBACK_LIFT   = 0.0  # fallback: lift > 0

# DEV-BASED LIFT TRACKING
USE_DEV_LIFT_TRACKING  = True
LIFT_EVAL_EVERY        = 60  # = EVAL_EVERY
LIFT_DEV_SET_SIZE      = 100  # stratified by n_ops
LIFT_EMA_ALPHA         = 0.4  # smoothing factor

# STAGE 2 STRENGTHEN
VALIDATION_N_SAMPLES         = 40
COMMON_ERRORS_THRESHOLD      = 0.005
RARE_ERRORS_THRESHOLD        = 0.0

# VERIFY-ITERATE
USE_VERIFY_ITERATE        = True
MAX_VERIFY_ROUNDS = 3
VERIFY_DEDUP_JACCARD      = 0.90  # allow more variation
VERIFY_REQUIRE_PA = os.environ.get("ACE_FINQA_VERIFY_REQUIRE_PA", "1") == "1"
VERIFY_SKIP_LUCKY_GUESS   = True  # already covered by USE_LUCKY_GUESS_*

# AUTO-ABLATE (DEV-BASED)
ABLATE_DATA_SOURCE       = 'dev'
ABLATE_MIN_USES          = 10
ABLATE_PA_LIFT_THR       = -0.02
ABLATE_MIN_PLAYBOOK_SIZE = 10
ABLATE_WINDOW            = 100  # for fallback history-based mode

# QUARANTINE
QUARANTINE_COOLDOWN = 200

# CLUSTER MATCHING
CLUSTER_MATCH_MODE       = 'highest_score'  # 'AND_strict' | 'highest_score'
CLUSTER_NSTEPS_WEIGHT    = 2
CLUSTER_FIRSTOP_WEIGHT   = 1
CLUSTER_REGEX_WEIGHT     = 2
CLUSTER_MATCH_THRESHOLD = 2
STRICT_REGEX_CLUSTERS    = {
    'C6_unit_no_conversion',
    'C8_sub_div_2step_other',
    'C12_misc_other',
    'C16_multistep_5plus',
}
MAX_BULLETS_PER_CLUSTER  = 2


### BEST-CHECKPOINT (3 SNAPSHOTS)


In [ ]:
USE_3SNAPSHOT_BEST       = True
EA_STRICT_PA_FLOOR_DELTA = 0.02  # best_ea_strict requires PA ≥ baseline_PA - 0.02

# 19. POST-TRAINING PRUNING
USE_POST_TRAINING_PRUNE  = True
PRUNE_LIFT_THRESHOLD     = -0.01
PRUNE_DEV_SUBSET         = 200  # saves ~30 min
SKIP_PRUNE_EA            = True  # saves 30-45 min

# 20. STRATIFIED TRAINING
STRATIFIED_RATIOS = (140, 170, 170, 100, 20)  # 5-bucket (1/2/3/4/5+)

# 21. EARLY STOPPING
EARLY_STOP_PATIENCE = 8

print("\n[CONFIGURATION SUMMARY]")
print(f"  COMPOSITE_EA_WEIGHT  : {COMPOSITE_EA_WEIGHT} ")
print(f"  COMPOSITE_PA_WEIGHT  : {COMPOSITE_PA_WEIGHT}")
print(f"  PA_GUARD_TOLERANCE   : {PA_GUARD_TOLERANCE}")
print(f"  MAX_TOKENS           : {MAX_TOKENS}")
print(f"  MAX_PLAYBOOK_BULLETS : {MAX_PLAYBOOK_BULLETS} (T1 max={TIER_1_MAX} + T2)")
print(f"  TOP_K_RETRIEVAL      : {TOP_K_RETRIEVAL} (T1={TOP_K_RETRIEVAL_TIER1} + T2={TOP_K_RETRIEVAL_TIER2})")
print(f"  VERIFY_REQUIRE_PA    : {VERIFY_REQUIRE_PA}")
print(f"  MAX_VERIFY_ROUNDS    : {MAX_VERIFY_ROUNDS}")
print(f"  VERIFY_DEDUP_JACCARD : {VERIFY_DEDUP_JACCARD}")
print(f"  VALIDATION_N_SAMPLES : {VALIDATION_N_SAMPLES}")
print(f"  COMMON_ERR_THRESHOLD : {COMMON_ERRORS_THRESHOLD}")
print(f"  RARE_ERR_THRESHOLD   : {RARE_ERRORS_THRESHOLD}")
print(f"  ABLATE_DATA_SOURCE   : {ABLATE_DATA_SOURCE}")
print(f"  ABLATE_MIN_USES      : {ABLATE_MIN_USES}")
print(f"  ABLATE_PA_LIFT_THR   : {ABLATE_PA_LIFT_THR}")
print(f"  QUARANTINE_COOLDOWN  : {QUARANTINE_COOLDOWN}")
print(f"  CLUSTER_MATCH_MODE   : {CLUSTER_MATCH_MODE}")
print(f"  STRATIFIED_RATIOS    : {STRATIFIED_RATIOS})")
print(f"  USE_LUCKY_REFLECT    : {USE_LUCKY_GUESS_REFLECT}")
print(f"  PRUNE_DEV_SUBSET     : {PRUNE_DEV_SUBSET}")
print(f"  SKIP_PRUNE_EA        : {SKIP_PRUNE_EA}")
print()
print("[ENABLED MODULES]")
print(f"  Tier 1/2 system      : ✅ (lock {TIER_1_MAX} after age={TIER_1_PROMOTION_AGE})")
print(f"  Dev lift tracking    : ✅ (every {LIFT_EVAL_EVERY} steps × {LIFT_DEV_SET_SIZE} samples)")
print(f"  3-snapshot best      : ✅ (composite + ea-strict + pa)")
print(f"  Cluster highest-score: ✅ (threshold={CLUSTER_MATCH_THRESHOLD}/5)")


### PATHS


In [ ]:
DRIVE_BASE = os.environ.get("ACE_FINQA_DRIVE_BASE", "/content/drive/MyDrive")
DATA_DIR = os.environ.get("ACE_FINQA_DATA_DIR", f"{DRIVE_BASE}/datasets")
TRAIN_PATH = os.path.join(DATA_DIR, "train.json")
DEV_PATH = os.path.join(DATA_DIR, "dev.json")
TEST_PATH = os.path.join(DATA_DIR, "test.json")
OUTPUT_DIR = os.environ.get(
    "ACE_FINQA_OUTPUT_DIR", f"{DRIVE_BASE}/ace-finqa-runs/ace/{MODEL_TAG}"
)
PLAYBOOK_DIR  = f"{OUTPUT_DIR}/playbooks"
CHUNK_DIR     = f"{OUTPUT_DIR}/chunks"
LOG_DIR       = f"{OUTPUT_DIR}/logs"
PROGRESS_PATH = f"{OUTPUT_DIR}/progress.json"

for d in [OUTPUT_DIR, PLAYBOOK_DIR, CHUNK_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

for path, name in [(TRAIN_PATH, "train"), (DEV_PATH, "dev")]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"[ERROR] Cannot find {name}: {path}")
    with open(path) as f: _n = len(json.load(f))
    print(f"[DATA] {name}: {_n} samples | {os.path.getsize(path)/1024**2:.1f} MB")

# Resume/reset controls; state is never deleted implicitly.
START_STEP = int(os.environ.get("ACE_FINQA_START_STEP", "0"))
RESET_RUN_STATE = os.environ.get("ACE_FINQA_RESET_RUN_STATE", "0") == "1"
if os.path.exists(PROGRESS_PATH):
    print("[CONFIG] Existing progress detected; choose resume, reset, or a new output directory")

print(f"[CONFIG] {MODEL_TAG} → {OUTPUT_DIR}")
print(f"[CONFIG] TRAIN={TRAIN_SUBSET} | TOP_K={TOP_K_RETRIEVAL} | MAX_SEQ={MAX_SEQ_LENGTH}")


## Model loading


In [ ]:
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
import torch
import random
import numpy as np
from unsloth import FastLanguageModel
from vllm import SamplingParams
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"[MODEL] Đang tải {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name             = MODEL_NAME,
    max_seq_length         = MAX_SEQ_LENGTH,
    load_in_4bit           = True,
    fast_inference         = True,
    gpu_memory_utilization = GPU_MEM_UTIL,
    enforce_eager          = True,
    max_num_seqs           = 256,
)
SAMPLING_PARAMS = SamplingParams(
    temperature         = TEMPERATURE,
    max_tokens          = MAX_TOKENS,
    repetition_penalty  = REPETITION_PENALTY,
    skip_special_tokens = True,
    seed                = RANDOM_SEED,
)
_total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
_used_vram  = (_total_vram - torch.cuda.mem_get_info()[0] / 1024**3)
print(f"[MODEL] ✅ {MODEL_TAG} sẵn sàng | VRAM {_used_vram:.1f}/{_total_vram:.1f} GB")


## Notebook evaluator


In [ ]:
import math
import re
FINQA_CONSTANTS = {
    "const_1": 1.0, "const_2": 2.0, "const_3": 3.0, "const_4": 4.0,
    "const_5": 5.0, "const_6": 6.0, "const_7": 7.0, "const_8": 8.0,
    "const_9": 9.0, "const_10": 10.0, "const_12": 12.0,
    "const_100": 100.0, "const_1000": 1000.0, "const_10000": 10000.0,
    "const_100000": 100000.0, "const_1000000": 1000000.0,
    "const_1000000000": 1000000000.0, "const_m1": -1.0,
}

def _split_dsl_items(text):
    """Split comma-separated DSL items while respecting quotes and parentheses."""
    items, start, depth, quote = [], 0, 0, None
    for index, char in enumerate(text):
        if quote:
            if char == quote and (index == 0 or text[index - 1] != "\\"):
                quote = None
        elif char in {"'", '"'}:
            quote = char
        elif char == '(':
            depth += 1
        elif char == ')':
            depth -= 1
            if depth < 0:
                raise ValueError("unbalanced parentheses")
        elif char == ',' and depth == 0:
            items.append(text[start:index].strip())
            start = index + 1
    if quote or depth:
        raise ValueError("unterminated quote or parenthesis")
    items.append(text[start:].strip())
    return items

def _parse_numeric_literal(token):
    text = str(token).strip().replace('−', '-').replace('–', '-')
    negative = text.startswith('(') and text.endswith(')')
    if negative:
        text = text[1:-1].strip()
    text = re.sub(r'^[\$€£¥]\s*', '', text).replace(',', '')
    is_percent = text.endswith('%')
    if is_percent:
        text = text[:-1].strip()
    value = float(text)
    if not math.isfinite(value):
        raise ValueError("non-finite number")
    if negative:
        value = -abs(value)
    return value / 100.0 if is_percent else value

def _table_row_values(table, label):
    if not isinstance(table, list) or len(table) < 2:
        raise ValueError("table is missing")
    normalized = ' '.join(str(label).casefold().split())
    matches = [row for row in table[1:] if isinstance(row, list) and row
               and ' '.join(str(row[0]).casefold().split()) == normalized]
    if len(matches) != 1:
        raise ValueError("table row label must match exactly once")
    values = []
    for cell in matches[0][1:]:
        text = str(cell).strip()
        if not text or text.casefold() in {'-', '--', '—', 'n/a', 'na', 'none'}:
            continue
        match = re.match(r'^[\s\$€£¥(]*[-+]?\d[\d,]*(?:\.\d+)?%?', text)
        if not match:
            continue
        values.append(_parse_numeric_literal(match.group(0).strip()))
    if not values:
        raise ValueError("table row has no numeric values")
    return values

def execute_program(program, table):
    """Execute a FinQA DSL program and fail closed on malformed input."""
    if not isinstance(program, str) or not program.strip():
        return None
    results = []

    def resolve(token):
        token = token.strip()
        if re.fullmatch(r'#\d+', token):
            index = int(token[1:])
            if index >= len(results) or isinstance(results[index], str):
                raise ValueError("invalid numeric reference")
            return float(results[index])
        if token.casefold() in FINQA_CONSTANTS:
            return FINQA_CONSTANTS[token.casefold()]
        return _parse_numeric_literal(token)

    try:
        for command in _split_dsl_items(program.strip()):
            match = re.fullmatch(r'([a-z_]+)\s*\((.*)\)', command, re.I | re.S)
            if not match:
                raise ValueError("malformed operation")
            operation = match.group(1).casefold()
            arguments = _split_dsl_items(match.group(2))
            if len(arguments) != 2:
                raise ValueError("operations require two arguments")
            left, right = arguments
            if operation.startswith('table_'):
                if right.strip().casefold() != 'none':
                    raise ValueError("table operations require the none sentinel")
                label = left.strip().strip('"').strip("'")
                values = _table_row_values(table, label)
                functions = {
                    'table_max': max,
                    'table_min': min,
                    'table_sum': sum,
                    'table_average': lambda items: sum(items) / len(items),
                }
                if operation not in functions:
                    raise ValueError("unsupported table operation")
                result = float(functions[operation](values))
            else:
                a, b = resolve(left), resolve(right)
                if operation == 'add':
                    result = a + b
                elif operation == 'subtract':
                    result = a - b
                elif operation == 'multiply':
                    result = a * b
                elif operation == 'divide':
                    if b == 0:
                        raise ValueError("division by zero")
                    result = a / b
                elif operation == 'exp':
                    result = a ** b
                elif operation == 'greater':
                    result = 'yes' if a > b else 'no'
                else:
                    raise ValueError("unsupported operation")
            if isinstance(result, float) and not math.isfinite(result):
                raise ValueError("non-finite result")
            results.append(result)
    except (ArithmeticError, OverflowError, TypeError, ValueError):
        return None
    return results[-1] if results else None

# DSL ops detection regex (used by extract_program permissive mode)
_DSL_OP_DETECT = re.compile(
    r'\b(add|subtract|multiply|divide|greater|exp|table_max|table_min|'
    r'table_sum|table_average)\s*\(',
    re.I
)

def extract_program(text):
    """Extract the final FinQA program from supported response formats.

    Priority:
      1. ```plaintext block with 'program: <ops>' prefix
      2. ```plaintext block with DSL ops directly (no prefix needed)
      3. 'program:' anywhere in text outside block (fallback)
      4. JSON {"program": "..."} format
    """
    if not text: return None
    clean = re.sub(r'<think>.*?</think>', '', text, flags=re.S).strip()

    # Match ```plaintext / ```text / ``` (any code block)
    blocks = re.findall(r'```(?:plaintext|text)?\n?(.*?)\n?```', clean, re.S | re.I)

    if blocks:
        # Use LAST block (model often shows multiple, last is usually final answer)
        last = blocks[-1].strip()

        # Priority 1: Has explicit "program:" prefix
        p = re.search(r'program:\s*(.+?)(?:\n|$)', last, re.S | re.I)
        if p:
            return re.sub(r'[`\n]+$', '', p.group(1)).strip()

        # Priority 2: Block content IS a DSL program (no prefix)
        # Check if block contains DSL operations
        if _DSL_OP_DETECT.search(last):
            # Take first non-empty line that has DSL ops
            for line in last.split('\n'):
                line = line.strip()
                if line and _DSL_OP_DETECT.search(line):
                    return re.sub(r'[`\n]+$', '', line).strip()

    # Priority 3: "program:" anywhere outside block (fallback)
    p = re.search(r'program:\s*(.+?)(?:\n|```|$)', clean, re.S | re.I)
    if p:
        prog = re.sub(r'[`\n]+$', '', p.group(1)).strip()
        if prog: return prog

    # Priority 4: JSON format
    j = re.search(r'"program"\s*:\s*"([^"]+)"', clean)
    if j: return j.group(1).strip()

    return None

def _canonical_operand(token, step_index):
    token = token.strip()
    if re.fullmatch(r'#\d+', token):
        reference = int(token[1:])
        if reference >= step_index:
            raise ValueError("forward or out-of-range reference")
        return f'#{reference}'
    if token.casefold() in FINQA_CONSTANTS:
        value = FINQA_CONSTANTS[token.casefold()]
    else:
        value = _parse_numeric_literal(token)
    return str(int(value)) if value == int(value) else f'{value:.10f}'.rstrip('0').rstrip('.')

def normalize_program(program):
    """Return a semantics-preserving canonical program or an empty string."""
    if not isinstance(program, str) or not program.strip():
        return ''
    canonical = []
    try:
        for step_index, command in enumerate(_split_dsl_items(program.strip())):
            match = re.fullmatch(r'([a-z_]+)\s*\((.*)\)', command, re.I | re.S)
            if not match:
                raise ValueError("malformed operation")
            operation = match.group(1).casefold()
            arguments = _split_dsl_items(match.group(2))
            if len(arguments) != 2:
                raise ValueError("operations require two arguments")
            if operation.startswith('table_'):
                if operation not in {'table_max', 'table_min', 'table_sum', 'table_average'}:
                    raise ValueError("unsupported table operation")
                left = ' '.join(arguments[0].strip().strip('"').strip("'").casefold().split())
                right = 'none'
            else:
                if operation not in {'add', 'subtract', 'multiply', 'divide', 'exp', 'greater'}:
                    raise ValueError("unsupported operation")
                left = _canonical_operand(arguments[0], step_index)
                right = _canonical_operand(arguments[1], step_index)
                if operation in {'add', 'multiply'}:
                    left, right = sorted((left, right))
            canonical.append(f'{operation}({left},{right})')
    except (TypeError, ValueError):
        return ''
    return ','.join(canonical)

def check_ea(predicted, gold, decimal_places=5):
    """Match the FinQA evaluator by comparing values rounded to five places."""
    if isinstance(predicted, str) or isinstance(gold, str):
        left, right = str(predicted).strip().casefold(), str(gold).strip().casefold()
        return left in {'yes', 'no'} and left == right
    if predicted is None or gold is None or isinstance(predicted, bool) or isinstance(gold, bool):
        return False
    try:
        left, right = float(predicted), float(gold)
        return math.isfinite(left) and math.isfinite(right) and round(left, decimal_places) == round(right, decimal_places)
    except (TypeError, ValueError, OverflowError):
        return False

def check_pa(predicted_program, gold_program):
    predicted = normalize_program(predicted_program)
    gold = normalize_program(gold_program)
    return bool(predicted) and predicted == gold


### SELF-TESTS


In [ ]:
# Evaluator regression tests
assert execute_program("greater(100, 50)", []) == 'yes'
assert execute_program("greater(100, 50), add(#0, const_1)", []) is None
assert check_ea('yes', 'yes') == True
assert check_ea(0.25, 0.250001, 5) is True

# Constant equivalence
assert check_pa("divide(637, 5.0)", "divide(637, const_5)")
assert check_pa("subtract(193.5, 100.0), divide(#0, 100.0)",
                 "subtract(193.5, const_100), divide(#0, const_100)")
assert not check_pa("divide(60, 243)", "divide(60, 243), multiply(#0, const_100)")

# Extraction tests

# Standard "program:" prefix
_t1 = """```plaintext
program: divide(637, 5.0)
```"""
assert extract_program(_t1) == "divide(637, 5.0)", \
    f"Standard prefix fail: {extract_program(_t1)}"

# Accept fenced blocks without a program prefix
_t2 = """```plaintext
subtract(193.5, const_100), divide(#0, const_100)
```"""
result_t2 = extract_program(_t2)
assert result_t2 == "subtract(193.5, const_100), divide(#0, const_100)", \
    f"extraction failed: {result_t2}"

# Block with prose explanation before DSL (model output verbose)
_t3 = """To calculate this, we need to divide.

```plaintext
program: divide(60, 243), multiply(#0, const_100)
```"""
assert extract_program(_t3) == "divide(60, 243), multiply(#0, const_100)"

# Block with prose, no "program:" prefix
_t4 = """Here is the calculation:

```plaintext
divide(60, 243), multiply(#0, const_100)
```"""
result_t4 = extract_program(_t4)
assert result_t4 == "divide(60, 243), multiply(#0, const_100)", \
    f"Prose + permissive fail: {result_t4}"

# Block without language tag (just ``` ... ```)
_t5 = """```
add(18.9, 0.3)
```"""
assert extract_program(_t5) == "add(18.9, 0.3)", \
    f"Bare block fail: {extract_program(_t5)}"

# Empty block
_t6 = """```plaintext
```"""
assert extract_program(_t6) is None, \
    f"Empty block should return None: {extract_program(_t6)}"

# Block with non-DSL content
_t7 = """```plaintext
This is just some text with no operations.
```"""
assert extract_program(_t7) is None, \
    f"Non-DSL block should return None: {extract_program(_t7)}"

# Multiple blocks — use LAST (final answer)
_t8 = """First attempt:
```plaintext
divide(60, 243)
```
Actually, the correct answer is:
```plaintext
divide(60, 243), multiply(#0, const_100)
```"""
assert extract_program(_t8) == "divide(60, 243), multiply(#0, const_100)", \
    f"Multiple blocks should use last: {extract_program(_t8)}"

# Fallback "program:" outside block (newline terminates match)
_t9 = "The program: divide(637, 5.0)\nis the answer."
assert extract_program(_t9) == "divide(637, 5.0)"

# PA equivalence still works after extract
_extracted = extract_program("""```plaintext
subtract(193.5, const_100), divide(#0, const_100)
```""")
_gold = "subtract(193.5, const_100), divide(#0, const_100)"
assert check_pa(_extracted, _gold)

# Boolean comparison example
# Real output: just DSL inside block, no prefix
_real_sample1 = """```plaintext
subtract(193.5, const_100), divide(#0, const_100)
```"""
result_real = extract_program(_real_sample1)
assert result_real is not None, \
    f"REAL Sample 1 must extract: got {result_real}"
assert "subtract" in result_real and "divide" in result_real

# Fail-closed FinQA semantics
_STRICT_TABLE_FIXTURE = [
    ["", "2019", "2020"],
    ["revenue", "10", "20"],
]
assert execute_program("divide(32%, const_100)", []) == 0.0032
assert execute_program("table_sum(revenue, none)", _STRICT_TABLE_FIXTURE) == 30.0
assert execute_program("add(#9, const_1)", []) is None
assert check_ea(17290, 17447, 5) is False

print("[DSL] ✅ Evaluator regression tests passed")
print("         - extraction, constants, references, table rows, and strict EA")
print("         - Commutative add/multiply (existing)")


## Data and prompt construction


In [ ]:
import json, re

RUN_NAME           = "FULL_thesis"
USE_THINKING_TRACE = False
USE_BARE_PLAYBOOK  = True


### Full system prompt


In [ ]:
SYSTEM_PROMPT = """You are a financial analyst. Given context from SEC filings, write a DSL program to answer the question.

=== OPERATIONS ===
- add(a, b), subtract(a, b), multiply(a, b), divide(a, b)
- greater(a, b): returns "yes" if a > b, else "no"
- exp(a, b): a^b
- table_max(column_name, none), table_min(column_name, none), table_average(column_name, none), table_sum(column_name, none)
- Constants: const_1, const_2, const_3, const_4, const_5, const_6, const_7, const_8, const_9, const_10, const_12, const_100, const_1000, const_10000, const_100000, const_1000000, const_1000000000, const_m1
- Step references: #0 = result of 1st op, #1 = 2nd, etc.

=== CRITICAL RULES ===

1. ANSWER FORMAT: Results are ALWAYS decimal, NEVER percentage.
   "what percent of X is Y" → divide(Y, X) → returns 0.41, NOT 41.
   "percentage change" → subtract(new, old), divide(#0, old) → returns 0.15, NOT 15.
   DO NOT multiply by const_100 unless the question explicitly asks to convert units.

2. SUBTRACT ORDER: subtract(a, b) means a − b. For "change from Y1 to Y2", ALWAYS use subtract(Y2_value, Y1_value), even if the result is negative.
   "increase from 2010 to 2011" with 2010=411, 2011=403 → subtract(403, 411) → result is -8 (a decrease).
   NEVER swap the order to avoid negative numbers.

3. UNIT CONVERSION: Use const_ ONLY when question asks "in millions" but data is in raw numbers (or vice versa).
   Data is 4840000, question asks "in millions" → divide(4840000, const_1000000)
   Data already in millions, question asks "in millions" → no conversion needed.

4. TABLE FUNCTIONS: Use the ROW LABEL (first column) as column_name, not the year.
   Table: ["", "2008", "2007"] / ["interest income", "653", "647"]
   → table_max(interest income, none), NOT table_max(2008, none)

5. RANGE: For "range of X" or "difference between max and min":
   → table_min(column, none), table_max(column, none), subtract(#1, #0)
   Always 3 steps: min first, max second, then subtract.

6. MINIMUM OPERATIONS: Use the fewest steps possible. Do not add unnecessary operations.
   "what percent of total is X" → divide(X, total). That's it. One step.

7. FLAT FORMAT: Operations must be sequential, never nested.

=== MULTI-STEP PATTERNS ===

Pattern A — Percentage change (2 steps):
subtract(new, old), divide(#0, old)

Pattern B — Sum then divide (3 steps):
add(a, b), add(#0, c), divide(#1, const_3)

Pattern C — Value-weighted comparison (4 steps):
multiply(shares_A, price_A), multiply(shares_B, price_B), subtract(#1, #0), divide(#2, #0)

Pattern D — Cumulative return from index (2 steps, uses const_100):
subtract(index_value, const_100), divide(#0, const_100)

Pattern E — Comparison with direct values (1 step):
greater(value_2017, value_2016)
Do NOT use table_max to extract values first. Use the actual numbers directly.

=== EXAMPLES ===

Q: percentage increase in revenue from 2018 ($450M) to 2019 ($500M)?
```plaintext
program: subtract(500, 450), divide(#0, 450)
```

Q: what percent of total goodwill ($2217.6M) was public ($911.3M)?
```plaintext
program: divide(911.3, 2217.6)
```

Q: total expenses of $120M, $135M, $142M over three years?
```plaintext
program: add(120, 135), add(#0, 142)
```

Q: average of 29, 24, 21 over three years?
```plaintext
program: add(29, 24), add(#0, 21), divide(#1, const_3)
```

Q: par value in millions? (2200000 units at $2.2 each)
```plaintext
program: multiply(2200000, 2.2), divide(#0, const_1000000)
```

Q: value-weighted difference between A (100 shares at $20) and B (120 shares at $25), relative to A?
```plaintext
program: multiply(100, 20), multiply(120, 25), subtract(#1, #0), divide(#2, #0)
```

Q: was rate greater in 2017 (4.70%) than 2016 (4.50%)?
```plaintext
program: greater(4.70, 4.50)
```

Q: range of volatility? (column "expected volatility")
```plaintext
program: table_min(expected volatility, none), table_max(expected volatility, none), subtract(#1, #0)
```

Q: cumulative return? (index ended at 221.91 from 100 base)
```plaintext
program: subtract(221.91, const_100), divide(#0, const_100)
```"""

print(f"[PROMPT] RUN_NAME={RUN_NAME} | thinking={USE_THINKING_TRACE} | "
      f"prompt_chars={len(SYSTEM_PROMPT)}")


### Context builder


In [ ]:
def table_to_str(table):
    if not table or not isinstance(table, list):
        return ""
    lines = []
    for row in table:
        if isinstance(row, list):
            lines.append(" | ".join(str(c).replace("$","").replace(",","").strip() for c in row))
    if not lines:
        return ""
    sep = " | ".join(["---"] * len(lines[0].split("|")))
    return "\n".join([lines[0], sep] + lines[1:])

def build_oracle_context(sample):
    gold_inds = sample.get("qa", {}).get("gold_inds", {})
    if not gold_inds:
        pre  = " ".join(sample.get("pre_text", [])).strip()
        post = " ".join(sample.get("post_text", [])).strip()
        tbl  = table_to_str(sample.get("table", []))
        return f"Text: {pre} {post}\nTable:\n{tbl}" if tbl else f"Text: {pre} {post}"
    text_facts, table_rows = [], []
    for key, value in gold_inds.items():
        if key.startswith("text_"):
            text_facts.append(value)
        elif key.startswith("table_"):
            try:
                table_rows.append(int(key.split("_")[1]))
            except:
                pass
    parts = []
    if text_facts:
        parts.append("Text: " + " ".join(text_facts))
    if table_rows and sample.get("table"):
        table = sample["table"]
        selected = [table[0]] + [table[i] for i in sorted(set(table_rows)) if 0 < i < len(table)]
        if len(selected) > 1:
            parts.append(f"Table:\n{table_to_str(selected)}")
    if not parts:
        pre = " ".join(sample.get("pre_text",[])).strip()
        post = " ".join(sample.get("post_text",[])).strip()
        return f"Text: {pre} {post}"
    return "\n".join(parts)


### Prompt builders


In [ ]:
def build_baseline_prompt(sample):
    """Baseline: no playbook, no fewshot."""
    q = sample["qa"]["question"]
    ctx = build_oracle_context(sample)
    user = (f"Context:\n{ctx}\n\n"
            f"Question: {q}\n\n"
            f"Write the program in a ```plaintext block.")
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": user}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

def build_baseline_prompt_short(sample, ctx_max=400):
    """Baseline with thinking enabled — for trace capture."""
    q = sample["qa"]["question"]
    ctx = build_oracle_context(sample)
    if len(ctx) > ctx_max:
        ctx = ctx[:ctx_max] + "..."
    user = (f"Context:\n{ctx}\n\n"
            f"Question: {q}\n\n"
            f"Solve step by step, then write the program in a ```plaintext block.")
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": user}],
        tokenize=False, add_generation_prompt=True,
        enable_thinking=True,
    )

def build_ace_prompt(sample, playbook_bullets="", fewshot_examples=None):
    """ACE prompt: system + (memory if bullets) + (fewshot if provided) + user."""
    q = sample["qa"]["question"]
    ctx = build_oracle_context(sample)

    user_parts = [f"Context:\n{ctx}\n"]

    if fewshot_examples:
        user_parts.append("Here are similar problems I solved before:\n")
        for i, ex in enumerate(fewshot_examples, 1):
            ex_q = ex['qa']['question']
            ex_prog = ex['qa']['program']
            ex_ans = ex['qa'].get('exe_ans', '')
            user_parts.append(f"Example {i}: {ex_q}")
            user_parts.append("```plaintext")
            user_parts.append(f"program: {ex_prog}")
            user_parts.append("```")
            user_parts.append(f"Answer: {ex_ans}\n")

    user_parts.append(f"Question: {q}\n")
    user_parts.append("Write the program in a ```plaintext block.")
    user_msg = "\n".join(user_parts)

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if playbook_bullets and playbook_bullets.strip():
        memory_msg = (
            "Here are strategies I learned from solving similar financial QA problems before. "
            "I will use them as secondary hints, but the rules in my system prompt take priority.\n\n"
            f"{playbook_bullets}\n\n"
            "Now I'm ready for the question."
        )
        messages.append({"role": "user", "content":
            "Before the actual question, can you recall what strategies you've learned from past problems?"})
        messages.append({"role": "assistant", "content": memory_msg})

    messages.append({"role": "user", "content": user_msg})

    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )


### Thinking trace capture (optional, OFF by default)


In [ ]:
_THINK_RE = re.compile(r'<think>(.*?)</think>', re.S)

def _truncate_thinking(trace, max_tokens=600):
    if not trace:
        return ""
    try:
        tokens = tokenizer.encode(trace)
        if len(tokens) <= max_tokens:
            return trace
        head = tokenizer.decode(tokens[:300])
        tail = tokenizer.decode(tokens[-300:])
        return f"{head}\n... [truncated middle] ...\n{tail}"
    except Exception:
        if len(trace) <= max_tokens * 4:
            return trace
        head_chars = max_tokens * 2
        return f"{trace[:head_chars]}\n... [truncated] ...\n{trace[-head_chars:]}"

def capture_slm_thinking(sample, max_tokens=2000):
    if not USE_THINKING_TRACE:
        return ""
    try:
        prompt = build_baseline_prompt_short(sample, ctx_max=400)
        sp = SamplingParams(
            temperature=0.0,
            max_tokens=max_tokens,
            repetition_penalty=REPETITION_PENALTY,
            skip_special_tokens=False,
            seed=RANDOM_SEED,
        )
        out = model.fast_generate([prompt], sampling_params=sp)
        raw = out[0].outputs[0].text
        m = _THINK_RE.search(raw)
        if not m:
            cut = raw.find('```plaintext')
            if cut > 50:
                trace = raw[:cut].strip()
            else:
                return ""
        else:
            trace = m.group(1).strip()
        return _truncate_thinking(trace, max_tokens=600)
    except Exception as e:
        print(f"[THINKING] ⚠ capture failed: {type(e).__name__}: {e}")
        return ""


### Fewshot example selection


In [ ]:
_OP_RE_FEWSHOT = re.compile(
    r'\b(add|subtract|multiply|divide|greater|exp|table_\w+)\s*\(', re.I)

def _n_ops_program(prog):
    if not prog:
        return 0
    return len(_OP_RE_FEWSHOT.findall(prog))

_train_q_embs = globals().get('_train_q_embs', {})
_train_by_complexity = globals().get('_train_by_complexity', {})

def _build_fewshot_cache_from_train(train_sub):
    global _train_q_embs, _train_by_complexity
    if _train_q_embs is None or not isinstance(_train_q_embs, dict):
        _train_q_embs = {}
    _train_by_complexity = {}
    for s in train_sub:
        n = _n_ops_program(s.get('qa', {}).get('program', ''))
        _train_by_complexity.setdefault(n, []).append(s)
    if _EMBED_MODEL is None:
        print("[FEWSHOT] ⚠ No embedding model — fewshot will use random fallback")
        return
    todo, todo_keys = [], []
    for s in train_sub:
        q = s['qa']['question']
        key = q[:80]
        if key not in _train_q_embs:
            todo.append(_BGE_QUERY_PREFIX + q)
            todo_keys.append(key)
    if todo:
        embs = _EMBED_MODEL.encode(todo, show_progress_bar=False,
                                    batch_size=64, normalize_embeddings=True)
        for k, e in zip(todo_keys, embs):
            _train_q_embs[k] = e
        print(f"[FEWSHOT] ✓ Cached {len(todo)} new train embeddings "
              f"(total: {len(_train_q_embs)})")
    else:
        print(f"[FEWSHOT] ✓ All {len(train_sub)} train embeddings already cached")
    print(f"[FEWSHOT] ✓ Complexity index: " +
          ", ".join(f"{k}-step={len(v)}" for k, v in sorted(_train_by_complexity.items())))

def _select_fewshot_for_query(query_sample, n_examples=2,
                                exclude_question=None, require_correct_history=True):
    if not _train_by_complexity:
        return []
    q = query_sample['qa']['question']
    n_steps_query = _n_ops_program(query_sample['qa'].get('program', ''))
    candidates = list(_train_by_complexity.get(n_steps_query, []))
    if len(candidates) < n_examples * 3:
        for delta in (-1, 1):
            candidates.extend(_train_by_complexity.get(n_steps_query + delta, []))
    if not candidates:
        return []
    if exclude_question is None:
        exclude_question = q
    candidates = [c for c in candidates if c['qa']['question'] != exclude_question]
    if require_correct_history:
        history_list = globals().get('history', [])
        if history_list:
            correct_qs = {h['q'] for h in history_list
                           if h.get('outcome') == 'correct' and h.get('q')}
            filtered = [c for c in candidates if c['qa']['question'][:80] in correct_qs]
            if len(filtered) >= n_examples:
                candidates = filtered
    if not candidates:
        return []
    if _EMBED_MODEL is not None:
        q_emb = _train_q_embs.get(q[:80])
        if q_emb is None:
            q_emb = _EMBED_MODEL.encode([_BGE_QUERY_PREFIX + q],
                                         normalize_embeddings=True)[0]
        scored = []
        for c in candidates:
            c_emb = _train_q_embs.get(c['qa']['question'][:80])
            if c_emb is None:
                continue
            sim = float(c_emb @ q_emb)
            scored.append((c, sim))
        scored.sort(key=lambda x: -x[1])
        return [c for c, sim in scored[:n_examples] if sim > 0.4]
    return candidates[:n_examples]

def _fewshot_count_for_query(query_sample):
    n = _n_ops_program(query_sample['qa'].get('program', ''))
    if n <= 2:
        return 0
    if n == 3:
        return 2
    return 3


### Load data


In [ ]:
with open(TRAIN_PATH) as f:
    train_raw = json.load(f)
with open(DEV_PATH) as f:
    dev_raw = json.load(f)

train_valid = [s for s in train_raw if s.get("qa", {}).get("program", "").strip()]
dev_valid   = [s for s in dev_raw   if s.get("qa", {}).get("program", "").strip()]
print(f"[DATA] Train: {len(train_valid)} | Dev: {len(dev_valid)}")


### Sanity checks


In [ ]:
_test_base = build_baseline_prompt(train_valid[0])
_test_ace  = build_ace_prompt(train_valid[0],
    playbook_bullets="- Strategy 1\n- Strategy 2")
_test_fs   = build_ace_prompt(train_valid[0],
    playbook_bullets="- Strategy 1",
    fewshot_examples=train_valid[1:4])

print(f"[PROMPT] Baseline    : {len(tokenizer.encode(_test_base))} tokens")
print(f"[PROMPT] ACE+memory  : {len(tokenizer.encode(_test_ace))} tokens")
print(f"[PROMPT] ACE+3fewshot: {len(tokenizer.encode(_test_fs))} tokens")
print(f"[PROMPT] Budget      : {MAX_SEQ_LENGTH - MAX_TOKENS} tokens for input")

assert _n_ops_program("subtract(500, 450), divide(#0, 450)") == 2
assert _n_ops_program("divide(165000, 254000)") == 1
assert _n_ops_program("") == 0

print("[PROMPT] Loaded")

# Model warmup
print("[WARMUP] ", end="")
_wp = tokenizer.apply_chat_template(
    [{"role": "user", "content": "test"}],
    tokenize=False, add_generation_prompt=True)
_ = model.fast_generate([_wp],
                          sampling_params=SamplingParams(temperature=0, max_tokens=5, seed=RANDOM_SEED))
print("✅")


## ACE retrieval, quality gate, and reflection


In [ ]:
import re, hashlib, subprocess, os, json
import numpy as np
from collections import deque, defaultdict
import glob as _glob


### Lexical retrieval dependency


In [ ]:
try:
    from rank_bm25 import BM25Okapi
    _BM25_AVAILABLE = True
except ImportError:
    _BM25_AVAILABLE = False
print(f"[BM25] {'✅ available' if _BM25_AVAILABLE else '⚠ disabled'}")


### Cluster definitions


In [ ]:
_PHASE0_CLUSTER_PATHS = [
    f"{OUTPUT_DIR}/phase0_clusters.json",
    f"{DRIVE_BASE}/ACE_test/{MODEL_TAG}/phase0_clusters.json",
    f"{DRIVE_BASE}/ACE_vip/{MODEL_TAG}/phase0_clusters.json",
    f"{DRIVE_BASE}/ACE_vip2/{MODEL_TAG}/phase0_clusters.json",
    f"{DRIVE_BASE}/{MODEL_TAG}/phase0_clusters.json",
    "/content/phase0_clusters.json",
    "/content/drive/MyDrive/phase0_clusters.json",
]
try:
    _PHASE0_CLUSTER_PATHS += _glob.glob(
        "/content/drive/MyDrive/**/phase0_clusters.json", recursive=True)
except Exception:
    pass
_seen = set()
_PHASE0_CLUSTER_PATHS = [p for p in _PHASE0_CLUSTER_PATHS
                          if not (p in _seen or _seen.add(p))]

_PHASE0_CLUSTERS_DATA = None
_PHASE0_LOADED_FROM = None
for _p in _PHASE0_CLUSTER_PATHS:
    if os.path.exists(_p):
        try:
            with open(_p) as _f:
                _PHASE0_CLUSTERS_DATA = json.load(_f)
            _PHASE0_LOADED_FROM = _p
            print(f"[CLUSTERS] ✅ Loaded {_p}")
            print(f"           {_PHASE0_CLUSTERS_DATA['n_clusters']} clusters, "
                   f"version: {_PHASE0_CLUSTERS_DATA.get('version', 'n/a')}")
            break
        except Exception as _e:
            print(f"[CLUSTERS] ⚠ Found {_p} but failed to parse: {_e}")

if _PHASE0_CLUSTERS_DATA is None:
    print(f"\n{'='*70}")
    print(f"  ❌ phase0_clusters.json NOT FOUND")
    print(f"{'='*70}")
    print(f"  Searched paths:")
    for _p in _PHASE0_CLUSTER_PATHS:
        print(f"    - {_p}  {'EXISTS' if os.path.exists(_p) else 'missing'}")
    print(f"\n  Upload phase0_clusters.json to {OUTPUT_DIR}/ in Drive,")
    print(f"  then re-run this cell.")
    print(f"{'='*70}\n")
    raise FileNotFoundError(
        f"phase0_clusters.json not found. Upload to {OUTPUT_DIR}/ and re-run.")

if _PHASE0_CLUSTERS_DATA['n_clusters'] < 5:
    print(f"\n[CLUSTERS] ⚠ Only {_PHASE0_CLUSTERS_DATA['n_clusters']} clusters loaded "
          f"(expected 12-16). File may be a stub.\n")

_CLUSTER_KW_RE = {
    c['id']: re.compile(c.get('kw_regex', '.*'), re.I)
    for c in _PHASE0_CLUSTERS_DATA['clusters']
}
_CLUSTER_BY_ID = {c['id']: c for c in _PHASE0_CLUSTERS_DATA['clusters']}
_CLUSTER_PRIORITY = _PHASE0_CLUSTERS_DATA.get(
    'matching_priority_order',
    [c['id'] for c in _PHASE0_CLUSTERS_DATA['clusters']])

_OP_RE_CLUSTER = re.compile(
    r'\b(add|subtract|multiply|divide|greater|exp|table_\w+)\s*\(', re.I)

def _first_op_of(prog):
    if not prog:
        return ''
    m = _OP_RE_CLUSTER.search(prog.lower())
    return m.group(1).lower() if m else ''

def _n_ops_of(prog):
    if not prog:
        return 0
    return len(_OP_RE_CLUSTER.findall(prog))


### Cluster classifier


In [ ]:
def cluster_id_for_sample(sample):
    """Highest-score cluster matching: each cluster scored by
       n_steps_match * CLUSTER_NSTEPS_WEIGHT
     + first_op_match * CLUSTER_FIRSTOP_WEIGHT
     + regex_match    * CLUSTER_REGEX_WEIGHT.

    Default weights: n=2, op=1, regex=2 → max=5.
    Threshold: ≥2 to count as a match.

    For STRICT_REGEX_CLUSTERS (catch-all regex), fall back to AND-strict
    matching to avoid over-population.

    Returns 'C12_misc_other' if no cluster meets threshold."""
    q = sample.get('qa', {}).get('question', '').lower()
    gold_prog = sample.get('qa', {}).get('program', '')
    n_steps = _n_ops_of(gold_prog)
    first_op = _first_op_of(gold_prog)

    mode = globals().get('CLUSTER_MATCH_MODE', 'highest_score')
    strict_set = globals().get('STRICT_REGEX_CLUSTERS', set())
    w_n = globals().get('CLUSTER_NSTEPS_WEIGHT', 2)
    w_op = globals().get('CLUSTER_FIRSTOP_WEIGHT', 1)
    w_re = globals().get('CLUSTER_REGEX_WEIGHT', 2)
    thr = globals().get('CLUSTER_MATCH_THRESHOLD', 2)

    if mode == 'AND_strict':
        # Strict conjunction mode
        for cid in _CLUSTER_PRIORITY:
            cdef = _CLUSTER_BY_ID.get(cid)
            if cdef is None:
                continue
            target_n = cdef.get('n_steps_gold', 0)
            if target_n != 0 and target_n != n_steps:
                continue
            target_op = cdef.get('first_op', '*')
            if target_op != '*' and target_op != first_op:
                continue
            if _CLUSTER_KW_RE[cid].search(q):
                return cid
        return 'C12_misc_other'

    # highest-score with strict-regex whitelist
    scores = {}
    priority_idx = {cid: i for i, cid in enumerate(_CLUSTER_PRIORITY)}

    for cid in _CLUSTER_PRIORITY:
        cdef = _CLUSTER_BY_ID.get(cid)
        if cdef is None:
            continue

        target_n = cdef.get('n_steps_gold', 0)
        target_op = cdef.get('first_op', '*')

        n_match = (target_n == 0) or (target_n == n_steps)
        op_match = (target_op == '*') or (target_op == first_op)
        re_match = bool(_CLUSTER_KW_RE[cid].search(q))

        # Strict regex whitelist: regex must match AND any other condition
        if cid in strict_set:
            if not re_match:
                continue
            # for strict, treat as binary: regex required, others bonus
            if not (n_match or op_match):
                continue
            # always score: regex+n+op all checked
            score = w_n * (1 if n_match else 0) + w_op * (1 if op_match else 0) + w_re
            if score >= thr:
                scores[cid] = score
            continue

        score = (w_n * (1 if n_match else 0)
               + w_op * (1 if op_match else 0)
               + w_re * (1 if re_match else 0))
        if score >= thr:
            scores[cid] = score

    if not scores:
        return 'C12_misc_other'

    # Tie-break: highest score, then highest priority (lowest index)
    best_cid = max(scores.items(),
                   key=lambda kv: (kv[1], -priority_idx.get(kv[0], 999)))[0]
    return best_cid

def log_cluster_distribution(samples, label='samples'):
    counts = defaultdict(int)
    for s in samples:
        counts[cluster_id_for_sample(s)] += 1
    print(f"\n[CLUSTERS] Distribution on {label} ({len(samples)} total):")
    total = sum(counts.values())
    for cid in _CLUSTER_PRIORITY:
        c = counts.get(cid, 0)
        pct = c / total * 100 if total else 0
        bar = '█' * int(pct / 2)
        print(f"  {cid:<28} {c:>4} ({pct:>5.1f}%) {bar}")
    return dict(counts)


### Section definitions


In [ ]:
SECTION_SLUGS = {
    'numerical_strategies': 'ns', 'table_reading_rules': 'tr',
    'program_generation':   'pg', 'common_errors':       'ce',
    'context_tips':         'ct',
}

ERROR_TO_SECTION = {
    'wrong_number': 'table_reading_rules', 'wrong_row_col': 'table_reading_rules',
    'wrong_operation': 'program_generation', 'unit_mismatch': 'numerical_strategies',
    'wrong_formula': 'program_generation', 'missing_step': 'program_generation',
    'no_program': 'program_generation', 'lucky_guess': 'program_generation',
    'exec_mismatch': 'numerical_strategies', 'close_but_wrong': 'program_generation',
    'wrong_reasoning': 'common_errors', 'other': 'common_errors',
    'wrong_value': 'table_reading_rules', 'magnitude_error': 'numerical_strategies',
    'sign_error': 'numerical_strategies',
    'format_ok_but_no_program': 'program_generation',
    'no_program_at_all': 'program_generation',
    'program_extracted_but_exec_fail': 'numerical_strategies',
    'missed_step': 'program_generation', 'extra_step': 'program_generation',
    'wrong_aggregate': 'table_reading_rules',
    'wrong_direct_value': 'table_reading_rules',
}

SECTION_QUOTA_PCT         = 0.50
SECTION_QUOTA_MIN_BULLETS = 8
SECTION_FALLBACK_ORDER    = ['numerical_strategies', 'table_reading_rules',
                              'program_generation', 'common_errors', 'context_tips']


### Initial playbook


In [ ]:
INITIAL_PLAYBOOK_FULL = """## Numerical Strategies

## Table Reading Rules

## Program Generation

## Common Errors
- [ce-00001] When the question asks for percentage or ratio, return decimal (0.41, not 41); apply divide(part, total) without const_100 unless unit conversion is explicit.

## Context Tips
- [ct-00001] When the question references a year or category not in the table, extract the value from pre_text or post_text and use the literal number directly in subtract/divide/add operations.
"""

INITIAL_PLAYBOOK_BARE = """## Numerical Strategies

## Table Reading Rules

## Program Generation

## Common Errors

## Context Tips
"""

INITIAL_PLAYBOOK = INITIAL_PLAYBOOK_BARE if USE_BARE_PLAYBOOK else INITIAL_PLAYBOOK_FULL
print(f"[PLAYBOOK] INITIAL_PLAYBOOK: "
      f"{'BARE (empty)' if USE_BARE_PLAYBOOK else 'FULL (2 bullets)'}")


### Playbook parsing


In [ ]:
def _section_header(s):
    for full, slug in SECTION_SLUGS.items():
        if s in (full, slug):
            return f"## {full.replace('_',' ').title()}"
    return "## Common Errors"

def _bullet_line(bid, content):
    return f"- [{bid}] {content}"

def _parse_bullet_line(line):
    m = re.match(r'^\s*-\s*\[([^\]]+)\]\s*(.+?)\s*(?:\(h=(\d+),\s*harm=(\d+)\))?\s*$', line)
    if not m:
        return None, None, 0, 0
    return m.group(1), m.group(2), int(m.group(3) or 0), int(m.group(4) or 0)

def _all_bullets(pb_str):
    bullets, current = [], None
    for line in pb_str.splitlines():
        line = line.rstrip()
        if line.startswith('##'):
            current = line[2:].strip().lower().replace(' ', '_')
        elif line.strip().startswith('-'):
            bid, content, h, harm = _parse_bullet_line(line)
            if bid:
                bullets.append({'id': bid, 'content': content, 'section': current,
                                 'helpful': h, 'harmful': harm})
    return bullets

def _pb_stats(pb_str):
    bullets = _all_bullets(pb_str)
    by_sec = {}
    for b in bullets:
        sec = (b['section'] or 'unknown').upper()
        by_sec[sec] = by_sec.get(sec, 0) + 1
    return {'total': len(bullets), 'by_section': by_sec}

def _next_id(pb_str):
    out = {}
    for b in _all_bullets(pb_str):
        parts = b['id'].split('-')
        if len(parts) == 2:
            slug, num = parts
            try:
                out[slug] = max(out.get(slug, 0), int(num))
            except:
                pass
    return out

def _section_counts(pb_str):
    counts = {full: 0 for full in SECTION_SLUGS}
    for b in _all_bullets(pb_str):
        sec = b.get('section') or 'common_errors'
        if sec in counts:
            counts[sec] += 1
    return counts

def _section_over_quota(pb_str, sec):
    counts = _section_counts(pb_str)
    total = sum(counts.values())
    if total < SECTION_QUOTA_MIN_BULLETS:
        return False
    return counts.get(sec, 0) >= total * SECTION_QUOTA_PCT

def _pick_fallback_section(pb_str, exclude):
    counts = _section_counts(pb_str)
    cands = [s for s in SECTION_FALLBACK_ORDER if s != exclude]
    cands.sort(key=lambda s: (counts.get(s, 0), SECTION_FALLBACK_ORDER.index(s)))
    return cands[0] if cands else 'common_errors'


### Semantic retrieval model and cache


In [ ]:
_BGE_QUERY_PREFIX = ("Represent this question for retrieving relevant "
                     "financial calculation strategies: ")

try:
    from sentence_transformers import SentenceTransformer
    _EMBED_MODEL = SentenceTransformer('BAAI/bge-base-en-v1.5')
    _EMBED_DIM   = 768
    print("[EMBED] ✅ BAAI/bge-base-en-v1.5")
except Exception:
    try:
        _EMBED_MODEL = SentenceTransformer('all-MiniLM-L6-v2')
        _EMBED_DIM   = 384
        print(f"[EMBED] ⚠ BGE failed, fallback MiniLM-L6")
    except Exception:
        _EMBED_MODEL = None
        _EMBED_DIM = 0
        print(f"[EMBED] ⚠ No embedding")

_BULLET_EMB_CACHE = {}

def _get_bullet_embeddings(bullets):
    if _EMBED_MODEL is None:
        return None
    contents = [b['content'] for b in bullets]
    hashes   = [hashlib.md5(c.encode()).hexdigest()[:12] for c in contents]
    embs     = [None] * len(bullets)
    todo, todo_idx = [], []
    for i, (b, h) in enumerate(zip(bullets, hashes)):
        cached = _BULLET_EMB_CACHE.get(b['id'])
        if cached and cached[0] == h:
            embs[i] = cached[1]
        else:
            todo.append(contents[i])
            todo_idx.append(i)
    if todo:
        new = _EMBED_MODEL.encode(todo, show_progress_bar=False, normalize_embeddings=True)
        for j, idx in enumerate(todo_idx):
            embs[idx] = new[j]
            _BULLET_EMB_CACHE[bullets[idx]['id']] = (hashes[idx], new[j])
    return np.array(embs)


### Stopwords + tokenization


In [ ]:
_DEDUP_STOPWORDS = {
    'the','a','an','is','are','was','were','be','been','to','for','in','on','of',
    'and','or','but','use','with','by','from','at','as','this','that','it','its',
    'when','if','then','should','must','will','can','do','does','than',
    'i','you','we','they','them','their','your','our',
}

def _tokenize(text):
    return [w for w in re.findall(r'\b[a-z_]+\b', text.lower())
            if w not in _DEDUP_STOPWORDS and len(w) > 2]

def _content_words(text):
    return set(_tokenize(text))


### BM25


In [ ]:
def _bm25_scores(question, bullets):
    if not _BM25_AVAILABLE or not bullets:
        return [0.5] * len(bullets)
    try:
        q_tokens = _tokenize(question)
        if not q_tokens:
            return [0.5] * len(bullets)
        corpus = [_tokenize(b['content']) for b in bullets]
        bm25 = BM25Okapi(corpus)
        scores = np.array(bm25.get_scores(q_tokens), dtype=float)
        m = scores.max()
        if m > 0:
            scores = scores / m
        return scores.tolist()
    except:
        return [0.5] * len(bullets)


### Hard-cap retrieval window


In [ ]:
_recent_retrieval_window = defaultdict(lambda: deque(maxlen=100))
_current_train_step      = [0]
HARD_CAP_RATIO       = 0.30
HARD_CAP_RATIO_HIGH  = 0.50
HARD_CAP_MIN_WINDOW  = 20

def _record_retrieval(bid, step):
    _recent_retrieval_window[bid].append(step)

def _bullet_retrieval_ratio(bid):
    win = _recent_retrieval_window[bid]
    if len(win) < HARD_CAP_MIN_WINDOW:
        return 0.0, 0
    ratio = len(set(win)) / 100
    if ratio > HARD_CAP_RATIO_HIGH:
        return ratio, 2
    if ratio > HARD_CAP_RATIO:
        return ratio, 1
    return ratio, 0


### Tier 1/2 Hierarchical System


In [ ]:
# Tier 1 = locked, never evicted. Always retrieved (k=TOP_K_RETRIEVAL_TIER1)
# Tier 2 = dynamic, can be evicted. Retrieved top-k=TOP_K_RETRIEVAL_TIER2

# Set of bullet IDs that are in Tier 1 (locked)
_tier1_bullets = globals().get('_tier1_bullets', set())

def _is_tier1(bullet_id):
    return bullet_id in _tier1_bullets

def _get_tier1_bullets():
    return set(_tier1_bullets)

def _promote_to_tier1(bullet_id):
    _tier1_bullets.add(bullet_id)

def _demote_from_tier1(bullet_id):
    _tier1_bullets.discard(bullet_id)

def _split_bullets_by_tier(bullets):
    """Return (tier1, tier2) lists from a flat bullet list."""
    t1, t2 = [], []
    for b in bullets:
        if _is_tier1(b['id']):
            t1.append(b)
        else:
            t2.append(b)
    return t1, t2


### Retrieval scoring


In [ ]:
USE_SECTION_AWARE_RETRIEVAL = True
RETRIEVAL_W_SEM   = 0.35
RETRIEVAL_W_BM25  = 0.25
RETRIEVAL_W_VALUE = 0.20
RETRIEVAL_W_FREQ  = 0.20
SECTION_AWARE_MIN_BULLETS = 10

def _score_bullets(question, bullets):
    if not bullets:
        return []
    if _EMBED_MODEL is None:
        q_words = set(question.lower().split())
        sem_raw = [len(q_words & set(b['content'].lower().split())) for b in bullets]
    else:
        q_emb  = _EMBED_MODEL.encode([_BGE_QUERY_PREFIX + question],
                                       normalize_embeddings=True)[0]
        b_embs = _get_bullet_embeddings(bullets)
        sem_raw = (b_embs @ q_emb).tolist()
    if sem_raw:
        s_min, s_max = min(sem_raw), max(sem_raw)
        sem_norm = ([(s - s_min) / (s_max - s_min) for s in sem_raw]
                    if s_max - s_min > 1e-8 else [0.5] * len(sem_raw))
    else:
        sem_norm = [0.5] * len(bullets)

    bm25_raw = _bm25_scores(question, bullets)
    val_scores = [b['helpful']/(b['helpful']+b['harmful']+1) if b['helpful']+b['harmful']>0
                  else 0.5 for b in bullets]
    counter = globals().get('_all_retrieved_ids', None)
    freq_scores = ([1.0/(counter.get(b['id'], 0)+1) for b in bullets]
                   if counter is not None else [0.5] * len(bullets))

    out = []
    for i, b in enumerate(bullets):
        s = (RETRIEVAL_W_SEM   * sem_norm[i]
             + RETRIEVAL_W_BM25  * bm25_raw[i]
             + RETRIEVAL_W_VALUE * val_scores[i]
             + RETRIEVAL_W_FREQ  * freq_scores[i])
        # Tier 1 bullets exempt from hard-cap penalty (always relevant)
        if not _is_tier1(b['id']):
            _, tier_pen = _bullet_retrieval_ratio(b['id'])
            if tier_pen == 2:
                s *= 0.10
            elif tier_pen == 1:
                s *= 0.50
        out.append((i, s))
    return out

def _section_aware_topk(question, bullets, k):
    scored = _score_bullets(question, bullets)
    scored.sort(key=lambda x: -x[1])
    if len(bullets) < SECTION_AWARE_MIN_BULLETS:
        return [i for i, _ in scored[:k]]

    by_sec = {}
    for i, _ in scored:
        sec = bullets[i].get('section') or 'common_errors'
        by_sec.setdefault(sec, []).append(i)
    quota = {
        'numerical_strategies': min(2, len(by_sec.get('numerical_strategies', []))),
        'table_reading_rules':  min(2, len(by_sec.get('table_reading_rules', []))),
        'program_generation':   min(1, len(by_sec.get('program_generation', []))),
        '_common_or_context':   min(1, len(by_sec.get('common_errors', []))
                                       + len(by_sec.get('context_tips', []))),
    }
    used = sum(quota.values())
    unused = k - used
    if unused > 0:
        for sec_key in ['program_generation', 'numerical_strategies', 'table_reading_rules']:
            n_avail = len(by_sec.get(sec_key, []))
            extra = min(unused, n_avail - quota.get(sec_key, 0))
            if extra > 0:
                quota[sec_key] += extra
                unused -= extra
            if unused == 0:
                break
    picked, picked_set = [], set()
    for sec_key, n in quota.items():
        if n <= 0:
            continue
        cand = (by_sec.get('common_errors', []) + by_sec.get('context_tips', [])
                if sec_key == '_common_or_context' else by_sec.get(sec_key, []))
        for idx in cand[:n]:
            if idx not in picked_set:
                picked.append(idx)
                picked_set.add(idx)
    if len(picked) < k:
        for idx, _ in scored:
            if idx not in picked_set:
                picked.append(idx)
                picked_set.add(idx)
                if len(picked) >= k:
                    break
    return picked[:k]


### Tier-aware retrieval


In [ ]:
def _retrieve_tier1_tier2(question, pb, k_tier1=None, k_tier2=None):
    """Hierarchical retrieval:
       - All Tier 1 bullets always included (up to k_tier1)
       - Top-k_tier2 from Tier 2 by score
       Returns (bullet_text, list_of_used_ids).

    If USE_TIER_SYSTEM is False or no Tier 1 bullets exist, falls back
    to flat top-k retrieval with k = k_tier1 + k_tier2.
    """
    if k_tier1 is None:
        k_tier1 = globals().get('TOP_K_RETRIEVAL_TIER1', 4)
    if k_tier2 is None:
        k_tier2 = globals().get('TOP_K_RETRIEVAL_TIER2', 3)

    bullets = _all_bullets(pb)
    if not bullets:
        return "", []

    use_tier_system = globals().get('USE_TIER_SYSTEM', True)
    tier1_set = _get_tier1_bullets()

    if not use_tier_system or not tier1_set:
        # Flat retrieval fallback
        k_total = k_tier1 + k_tier2
        if USE_SECTION_AWARE_RETRIEVAL:
            idxs = _section_aware_topk(question, bullets, k_total)
        else:
            scored = _score_bullets(question, bullets)
            scored.sort(key=lambda x: -x[1])
            idxs = [i for i, _ in scored[:k_total]]
        return ('\n'.join(f"- {bullets[i]['content']}" for i in idxs),
                [bullets[i]['id'] for i in idxs])

    t1_bullets, t2_bullets = _split_bullets_by_tier(bullets)

    # Tier 1: take all, ranked by score (so most relevant Tier 1 shown first)
    if t1_bullets:
        t1_scored = _score_bullets(question, t1_bullets)
        t1_scored.sort(key=lambda x: -x[1])
        t1_picked_idx = [i for i, _ in t1_scored[:k_tier1]]
        t1_picked = [t1_bullets[i] for i in t1_picked_idx]
    else:
        t1_picked = []

    # Tier 2: section-aware top-k
    if t2_bullets:
        if USE_SECTION_AWARE_RETRIEVAL and len(t2_bullets) >= SECTION_AWARE_MIN_BULLETS:
            t2_idxs = _section_aware_topk(question, t2_bullets, k_tier2)
        else:
            t2_scored = _score_bullets(question, t2_bullets)
            t2_scored.sort(key=lambda x: -x[1])
            t2_idxs = [i for i, _ in t2_scored[:k_tier2]]
        t2_picked = [t2_bullets[i] for i in t2_idxs]
    else:
        t2_picked = []

    combined = t1_picked + t2_picked
    return ('\n'.join(f"- {b['content']}" for b in combined),
            [b['id'] for b in combined])

def retrieve_top_k(question, pb, k=None):
    """Backward-compatible API. If k is given, behaves as flat top-k.
    Otherwise uses Tier 1/2 system with default budgets."""
    if k is None:
        text, _ = _retrieve_tier1_tier2(question, pb)
        return text
    # Flat top-k fallback
    bullets = _all_bullets(pb)
    if not bullets:
        return ""
    if USE_SECTION_AWARE_RETRIEVAL:
        idxs = _section_aware_topk(question, bullets, k)
    else:
        scored = _score_bullets(question, bullets)
        scored.sort(key=lambda x: -x[1])
        idxs = [i for i, _ in scored[:k]]
    return '\n'.join(f"- {bullets[i]['content']}" for i in idxs)

def _retrieve_with_ids(question, pb, k=None):
    """Backward-compatible API. If k is None, uses Tier 1/2."""
    if k is None:
        return _retrieve_tier1_tier2(question, pb)
    bullets = _all_bullets(pb)
    if not bullets:
        return "", []
    if USE_SECTION_AWARE_RETRIEVAL:
        idxs = _section_aware_topk(question, bullets, k)
    else:
        scored = _score_bullets(question, bullets)
        scored.sort(key=lambda x: -x[1])
        idxs = [i for i, _ in scored[:k]]
    return ('\n'.join(f"- {bullets[i]['content']}" for i in idxs),
            [bullets[i]['id'] for i in idxs])

def _update_counts(pb_str, tags):
    tag_map = {t['id']: t['tag'] for t in tags if 'id' in t and 'tag' in t}
    out = []
    for line in pb_str.splitlines():
        bid, content, h, harm = _parse_bullet_line(line)
        if bid and bid in tag_map:
            tag = tag_map[bid]
            if tag == 'helpful':
                h += 1
            elif tag == 'harmful':
                harm += 1
            indent = line[:len(line) - len(line.lstrip())]
            out.append(f"{indent}- [{bid}] {content} (h={h}, harm={harm})")
        else:
            out.append(line)
    return '\n'.join(out)


### Quality gate: text checks


In [ ]:
QG_FORBIDDEN_PHRASES = [
    'use the correct answer', 'the answer is', 'result should be',
    'exact number', 'specific value', 'remember this value',
    'identify key values', 'identify the relevant values', 'identify the correct values',
    'derive the final', 'derive the result',
    'avoid using', 'avoid the',
]

QG_DOMAIN_KEYWORDS = [
    'percent', 'change', 'revenue', 'cost', 'ratio', 'difference',
    'divide', 'subtract', 'multiply', 'add', 'operation',
    'calculate', 'compute', 'context', 'value', 'numerator', 'denominator',
    'table', 'column', 'row', 'cell', 'year', 'period',
    'program', 'step', 'step-by-step', 'formula', 'pattern',
    'unit', 'conversion', 'million', 'billion', 'thousand',
    'range', 'maximum', 'minimum', 'average', 'sum', 'total',
    'increase', 'decrease', 'growth', 'return', 'investment',
    'weighted', 'aggregate', 'compare', 'versus',
]

def _has_specific_numbers(t):
    return bool(re.search(r'\b\d{3,}\b|\d+\.\d+|\d+,\d+',
                          re.sub(r'const_\d+|#\d+', '', t)))

def _is_vague(t):
    return not any(kw in t.lower() for kw in QG_DOMAIN_KEYWORDS)

def _contains_year(t):
    return bool(re.search(r'\b(19|20)\d{2}\b', t))

_DSL_CHAIN_WITH_PARENS = re.compile(
    r'\b(add|subtract|multiply|divide|greater|exp|table_max|table_min|'
    r'table_sum|table_average)\s*\(', re.I)

def _has_dsl_chain(text):
    matches = _DSL_CHAIN_WITH_PARENS.findall(text)
    return len(matches) >= 2

_BUGGY_AVG_PADDING_RE = re.compile(
    r'add\s*\([^,)]+,\s*const_(\d+)\s*\)[^d]*?divide\s*\([^,)]+,\s*const_\1\s*\)',
    re.I)

def _has_buggy_avg_padding(text):
    return bool(_BUGGY_AVG_PADDING_RE.search(text))


### Quality gate: DSL validation


In [ ]:
_DSL_OP_ARGS_RE = re.compile(
    r'(add|subtract|multiply|divide|greater|exp'
    r'|table_max|table_min|table_sum|table_average)\s*\(([^)]+)\)', re.I)

def _check_dsl_references(text):
    ops = _DSL_OP_ARGS_RE.findall(text.lower())
    for i, (op, args) in enumerate(ops):
        for ref in re.findall(r'#(\d+)', args):
            if int(ref) >= i:
                return False, f'#{ref}_at_op{i}_invalid(only_<{i}_exist)'
        for c in re.findall(r'\bconst_\w+\b', args):
            if c not in FINQA_CONSTANTS:
                return False, f'unknown_const:{c}'
    return True, 'ok'

def _sanitize_playbook(pb_str):
    out, removed = [], []
    for line in pb_str.splitlines():
        bid, content, h, harm = _parse_bullet_line(line)
        if bid and content:
            ok, reason = _check_dsl_references(content)
            if not ok:
                removed.append({'id': bid, 'reason': reason, 'content': content[:60]})
                continue
        out.append(line)
    return '\n'.join(out), removed


### Quality gate: denominator checks


In [ ]:
_VALID_DENOMINATORS = {
    'total', 'old_value', 'old', 'earlier_value', 'previous_value',
    'count', 'n', 'denominator', 'base', 'baseline', 'sum', 'aggregate',
    'a', 'b', 'c', 'x', 'y', 'z', 'value1', 'value2', 'value3',
    'amount', 'amount1', 'amount2',
    'initial', 'initial_value', 'starting', 'starting_value',
    'first', 'first_value', 'reference', 'reference_value',
    'units', 'shares', 'count_n',
    'change_b', 'increase_b', 'decrease_b',
    'change', 'increase', 'decrease', 'difference',
    'divisor', 'deduction', 'increase_y',
    'rate', 'weight', 'factor',
    'base_value', 'prior_value', 'benchmark',
    'x_value', 'y_value', 'total_weight',
    # 4-step operands
    'total_w', 'sum_total', 'group_total',
    'w1', 'w2', 'w3',
}

def _check_divide_denominator(text):
    matches = re.findall(r'divide\(([^,]+),\s*([^)]+)\)', text.lower())
    for _, denom in matches:
        denom = denom.strip()
        if denom.startswith('#') or denom.startswith('const_'):
            continue
        if denom in _VALID_DENOMINATORS:
            continue
        root = denom.split('_')[0] if '_' in denom else denom
        if root in {'old','previous','earlier','total','sum','count',
                     'base','denominator','aggregate'}:
            continue
        return f'divide_vague_denominator:{denom[:25]}'
    return None


### Quality gate: synthetic execution


In [ ]:
_TEST_CATEGORIES = [
    {'name': 'pct_change',
     'keywords': ['percentage change', 'percent change', 'growth rate',
                   'rate of growth', 'rate of return', 'rate of increase',
                   'rate of decrease', 'percent increase', 'percent decrease',
                   'percentage increase', 'percentage decrease',
                   'expected growth', 'change from', 'change between'],
     'value_sets': [
         {'new_value': 120, 'old_value': 100, 'later_value': 120, 'earlier_value': 100,
          'final_value': 120, 'initial_value': 100, 'value1': 120, 'value2': 100,
          'a': 120, 'b': 100, 'x': 120, 'y': 100, 'expected': 0.20},
         {'new_value': 150, 'old_value': 100, 'later_value': 150, 'earlier_value': 100,
          'final_value': 150, 'initial_value': 100, 'value1': 150, 'value2': 100,
          'a': 150, 'b': 100, 'x': 150, 'y': 100, 'expected': 0.50},
         {'new_value': 80, 'old_value': 100, 'later_value': 80, 'earlier_value': 100,
          'final_value': 80, 'initial_value': 100, 'value1': 80, 'value2': 100,
          'a': 80, 'b': 100, 'x': 80, 'y': 100, 'expected': -0.20},
     ],
     'expected_range': (-0.30, 0.55), 'tolerance': 0.05},

    {'name': 'pct_share',
     'keywords': ['percent of total', 'percentage of total', 'what percent',
                   'share of', 'portion of'],
     'value_sets': [
         {'part': 30, 'total': 100, 'numerator': 30, 'denominator': 100,
          'value1': 30, 'value2': 100, 'x': 30, 'y': 100, 'expected': 0.30},
         {'part': 50, 'total': 200, 'numerator': 50, 'denominator': 200,
          'value1': 50, 'value2': 200, 'x': 50, 'y': 200, 'expected': 0.25},
         {'part': 75, 'total': 100, 'numerator': 75, 'denominator': 100,
          'value1': 75, 'value2': 100, 'x': 75, 'y': 100, 'expected': 0.75},
     ],
     'expected_range': (0.05, 1.0), 'tolerance': 0.05},

    {'name': 'average_general',
     'keywords': ['average calculation', 'average of', 'average over',
                   'mean of', 'compute the average', 'find the average',
                   'for averages', 'averages,', 'averages.', 'averages ',
                   'compute mean', 'mean of values', 'mean calculation',
                   'when averaging', 'averaging values', 'averaging the',
                   'arithmetic mean'],
     'value_sets': [
         {'variant': '3value', 'value1': 10, 'value2': 20, 'value3': 30,
          'count': 3, 'count_n': 3, 'total_count': 3, 'expected': 20},
         {'variant': '3value', 'value1': 100, 'value2': 200, 'value3': 300,
          'count': 3, 'count_n': 3, 'total_count': 3, 'expected': 200},
         {'variant': '3value', 'value1': 5, 'value2': 15, 'value3': 25,
          'count': 3, 'count_n': 3, 'total_count': 3, 'expected': 15},
         {'variant': '2value', 'value1': 10, 'value2': 20, 'expected': 15},
         {'variant': '2value', 'value1': 100, 'value2': 200, 'expected': 150},
         {'variant': '2value', 'value1': 5, 'value2': 15, 'expected': 10},
         {'variant': '2value', 'value1': 4, 'value2': 6, 'expected': 5},
     ],
     'expected_range': (4, 250), 'tolerance': 0.05},

    {'name': 'average_two',
     'keywords': ['average of two', 'average of exactly two'],
     'value_sets': [
         {'value1': 10, 'value2': 20, 'expected': 15},
         {'value1': 100, 'value2': 200, 'expected': 150},
         {'value1': 50, 'value2': 70, 'expected': 60},
     ],
     'expected_range': (10, 250), 'tolerance': 0.05},

    {'name': 'cumulative',
     'keywords': ['cumulative return', 'cumulative total'],
     'value_sets': [
         {'final_value': 150, 'initial_value': 100, 'index_value': 150, 'expected': 0.5},
         {'final_value': 120, 'initial_value': 100, 'index_value': 120, 'expected': 0.2},
         {'final_value': 200, 'initial_value': 100, 'index_value': 200, 'expected': 1.0},
     ],
     'expected_range': (0.10, 2.0), 'tolerance': 0.10},

    {'name': 'total_return',
     'keywords': ['total return on', 'total return from'],
     'value_sets': [
         {'investment': 100, 'growth_factor': 1.2, 'investment_value': 100, 'final_value': 120,
          'expected': 20},
         {'investment': 200, 'growth_factor': 1.5, 'investment_value': 200, 'final_value': 300,
          'expected': 100},
         {'investment': 50, 'growth_factor': 1.4, 'investment_value': 50, 'final_value': 70,
          'expected': 20},
     ],
     'expected_range': (5, 200), 'tolerance': 0.10},

    {'name': 'debt_ratio',
     'keywords': ['debt to asset', 'debt-to-asset', 'leverage ratio', 'debt ratio'],
     'value_sets': [
         {'debt': 50, 'liabilities': 30, 'asset_value': 200, 'total_debt': 50,
          'total_assets': 200, 'expected': 0.25},
         {'debt': 100, 'liabilities': 50, 'asset_value': 400, 'total_debt': 100,
          'total_assets': 400, 'expected': 0.25},
         {'debt': 80, 'liabilities': 40, 'asset_value': 200, 'total_debt': 80,
          'total_assets': 200, 'expected': 0.40},
     ],
     'expected_range': (0.05, 1.0), 'tolerance': 0.10},

    # 4-step weighted average (cluster C13)
    # Pattern: multiply(v1, w1), multiply(v2, w2), add(#0, #1), divide(#2, total_w)
    # Example: v1=10 w1=2, v2=20 w2=3 → (20+60)/5 = 16
    {'name': 'weighted_avg_4step',
     'keywords': ['weighted average', 'value-weighted', 'value weighted',
                   'weighted by', 'weighted mean', 'per-share basis',
                   'volume-weighted'],
     'value_sets': [
         {'v1': 10, 'v2': 20, 'w1': 2, 'w2': 3, 'total_w': 5,
          'expected': 16.0},
         {'v1': 100, 'v2': 200, 'w1': 1, 'w2': 4, 'total_w': 5,
          'expected': 180.0},
         {'v1': 50, 'v2': 30, 'w1': 3, 'w2': 7, 'total_w': 10,
          'expected': 36.0},
     ],
     'expected_range': (1, 1000), 'tolerance': 0.05},

    # 4-step difference of differences (cluster C14)
    # Pattern: subtract(a1, a2), subtract(b1, b2), subtract(#0, #1)
    {'name': 'diff_of_diff_4step',
     'keywords': ['difference of differences', 'compared to the',
                   'change versus', 'versus the change'],
     'value_sets': [
         {'a1': 100, 'a2': 80, 'b1': 50, 'b2': 30,
          'expected': 0},
         {'a1': 200, 'a2': 100, 'b1': 80, 'b2': 50,
          'expected': 70},
         {'a1': 150, 'a2': 100, 'b1': 70, 'b2': 50,
          'expected': 30},
     ],
     'expected_range': (-200, 200), 'tolerance': 0.05},

    # 4-step aggregate-then-pct (cluster C15)
    # Pattern: add(a, b), add(#0, c), divide(part, #1)
    {'name': 'aggregate_then_pct_4step',
     'keywords': ['total then percent', 'aggregate then', 'sum then divide',
                   'percent of aggregate', 'share of total'],
     'value_sets': [
         {'a': 100, 'b': 200, 'c': 200, 'part': 100, 'expected': 0.20},
         {'a': 50, 'b': 50, 'c': 100, 'part': 40, 'expected': 0.20},
         {'a': 30, 'b': 70, 'c': 100, 'part': 30, 'expected': 0.15},
     ],
     'expected_range': (0.01, 1.0), 'tolerance': 0.10},
]

_DSL_CHAIN_RE = re.compile(
    r'((?:add|subtract|multiply|divide|greater|exp|table_max|table_min|'
    r'table_sum|table_average)\s*\([^)]*\)(?:\s*,\s*'
    r'(?:add|subtract|multiply|divide|greater|exp|table_max|table_min|'
    r'table_sum|table_average)\s*\([^)]*\))*)', re.I)

def _validate_bullet_synthetically(content):
    m = _DSL_CHAIN_RE.search(content)
    if not m:
        return True, 'no_dsl'
    chain = re.sub(r'\s+', '', m.group(1))

    text = content.lower()
    cat = None
    for c in _TEST_CATEGORIES:
        if any(kw in text for kw in c['keywords']):
            cat = c
            break
    if cat is None:
        return True, 'no_test_category'
    if not cat.get('value_sets'):
        return True, f'skip_{cat["name"]}'

    valid_tok = {'add','subtract','multiply','divide','greater','exp',
                 'table_max','table_min','table_sum','table_average','none'}

    n_passed = 0
    n_in_range = 0
    n_tested = 0
    fail_details = []

    chain_lower = chain.lower()
    formula_variant = None
    if cat.get('name') == 'average_general':
        if 'value3' in chain_lower:
            formula_variant = '3value'
        elif 'count' in chain_lower and 'count_n' not in chain_lower:
            formula_variant = '3value'
        else:
            formula_variant = '2value'

    for vs in cat['value_sets']:
        if formula_variant and vs.get('variant') and vs['variant'] != formula_variant:
            continue

        sub = chain
        for name, val in sorted(vs.items(), key=lambda x: -len(x[0])):
            if name in ('expected', 'variant'):
                continue
            sub = re.sub(rf'\b{re.escape(name)}\b', str(val), sub, flags=re.I)

        leftover = [t for t in re.findall(r'[a-z][a-z_]+', sub.lower())
                    if t not in valid_tok and not t.startswith('const_')]
        if leftover:
            continue

        n_tested += 1

        try:
            result = execute_program(sub, [])
        except Exception as e:
            return False, f'exec_error({type(e).__name__})'
        if result is None:
            return False, 'exec_returned_none'
        if isinstance(result, str):
            return True, f'string_result({result})'

        try:
            r = float(result)
        except:
            return True, 'non_numeric'

        lo, hi = cat['expected_range']
        if lo <= r <= hi:
            n_in_range += 1

        exp = vs.get('expected')
        if exp is not None:
            tol = cat['tolerance']
            if exp == 0:
                if abs(r) < tol:
                    n_passed += 1
                else:
                    fail_details.append(f'{r:.3f}!={exp}')
            else:
                rel_err = abs(r - exp) / abs(exp)
                if rel_err < tol:
                    n_passed += 1
                else:
                    fail_details.append(f'{r:.3f}!={exp}(err={rel_err:.2f})')

    if n_tested == 0:
        return True, f'no_testable_set({cat["name"]})'
    if n_in_range < n_tested:
        return False, f'out_of_range({cat["name"]}_only_{n_in_range}/{n_tested}_in_range)'
    if n_passed < n_tested:
        return False, f'wrong_formula({cat["name"]}_{n_passed}/{n_tested}_pass:{fail_details[:2]})'
    return True, f'{cat["name"]}_passed_{n_passed}/{n_tested}'


### Quality gate: structural deduplication


In [ ]:
_NAMED_OPERANDS = {
    'old_value','new_value','later_value','earlier_value',
    'initial_value','final_value','index_value',
    'old','new','later','earlier','initial','final',
    'total','part','count','sum','denominator','numerator',
    'investment','growth_factor','asset_value','debt','liabilities',
    'value1','value2','value3','a','b','c','x','y',
    'amount','price','cost','revenue',
    # 4-step operands
    'v1','v2','v3','w1','w2','w3','total_w',
    'a1','a2','b1','b2',
}

def _classify_operand(arg):
    arg = arg.strip().lower()
    if arg.startswith('#'):
        return 'REF'
    if arg.startswith('const_'):
        return f'CONST({arg})'
    if arg in _NAMED_OPERANDS:
        return f'NAMED({arg})'
    try:
        float(arg.replace(',', ''))
        return 'LIT'
    except:
        return f'OTHER({arg[:15]})'

def _structural_signature(content):
    m = re.match(r'\s*(?:when|for|if|in|on)\s+([^,\.;:]{8,80})', content, re.I)
    trig = m.group(1).lower().strip() if m else ""
    trig_words = {w for w in re.findall(r'\b[a-z]+\b', trig)
                   if w not in _DEDUP_STOPWORDS and len(w) > 2}
    op_sigs = []
    for m in _DSL_OP_ARGS_RE.finditer(content):
        op = m.group(1).lower()
        kinds = tuple(_classify_operand(a) for a in m.group(2).split(','))
        op_sigs.append(f'{op}({",".join(kinds)})')
    return trig_words, '→'.join(op_sigs)

def _check_structural_duplicate(new_content, existing_bullets,
                                 trigger_overlap_thr=0.75):
    new_trig, new_ops = _structural_signature(new_content)
    if not new_ops:
        return False, None
    for b in existing_bullets:
        b_trig, b_ops = _structural_signature(b['content'])
        if not b_ops or new_ops != b_ops:
            continue
        if not new_trig or not b_trig:
            return True, f'{b["id"]}@same_ops_no_trigger'
        denom = min(len(new_trig), len(b_trig))
        sim = len(new_trig & b_trig) / denom if denom else 0.0
        if sim >= trigger_overlap_thr:
            return True, f'{b["id"]}@trig_ovr={sim:.2f}'
    return False, None


### Quality gate: lexical deduplication


In [ ]:
def _content_overlap(a, b):
    wa, wb = _content_words(a), _content_words(b)
    if not wa or not wb:
        return 0.0
    union = wa | wb
    return len(wa & wb) / len(union) if union else 0.0

def _similarity_jaccard(a, b):
    wa, wb = set(a.lower().split()), set(b.lower().split())
    return (len(wa & wb) / len(wa | wb)) if (wa | wb) else 0.0

def _check_duplicate_lexical(new_content, bullets):
    overlap_thr = globals().get('QG_OVERLAP_THRESH', 0.80)
    use_semantic = globals().get('USE_SEMANTIC_DEDUP', False) and _EMBED_MODEL is not None
    for b in bullets:
        if _content_overlap(new_content, b['content']) >= overlap_thr:
            return True, 'overlap'
    if use_semantic and bullets:
        new_emb = _EMBED_MODEL.encode([new_content], normalize_embeddings=True)[0]
        b_embs  = _get_bullet_embeddings(bullets)
        if float((b_embs @ new_emb).max()) >= QG_DEDUP_THRESH:
            return True, 'semantic'
        return False, 'semantic'
    for b in bullets:
        if _similarity_jaccard(new_content, b['content']) >= QG_DEDUP_THRESH:
            return True, 'jaccard'
    return False, 'jaccard'


### Unified quality gate


In [ ]:
def quality_gate(new_content, pb_str):
    if len(new_content) < QG_MIN_LEN:
        return 'reject', 'too_short'
    if len(new_content) > QG_MAX_LEN:
        return 'reject', 'too_long'
    if _is_vague(new_content):
        return 'reject', 'too_vague'
    for p in QG_FORBIDDEN_PHRASES:
        if p in new_content.lower():
            return 'reject', f'forbidden:{p[:20]}'
    if _has_specific_numbers(new_content):
        return 'reject', 'specific_numbers'
    if _contains_year(new_content):
        return 'reject', 'specific_year'
    if not _has_dsl_chain(new_content):
        return 'reject', 'no_dsl_chain'
    if _has_buggy_avg_padding(new_content):
        return 'reject', 'buggy_avg_const_padding'

    if re.search(r'multiply\s*\([^)]*,\s*const_100\)', new_content.lower()):
        return 'reject', 'multiply_const100'
    if re.search(r'multiply\s*\(\s*const_100\s*,', new_content.lower()):
        return 'reject', 'multiply_const100'

    ok, reason = _check_dsl_references(new_content)
    if not ok:
        return 'reject', f'dsl_invalid:{reason}'

    div_rej = _check_divide_denominator(new_content)
    if div_rej:
        return 'reject', div_rej

    ok, reason = _validate_bullet_synthetically(new_content)
    if not ok:
        return 'reject', f'synth:{reason}'

    bullets = _all_bullets(pb_str)
    is_dup, reason = _check_structural_duplicate(new_content, bullets)
    if is_dup:
        return 'reject', f'struct_dup:{reason}'

    is_dup, method = _check_duplicate_lexical(new_content, bullets)
    if is_dup:
        return 'reject', f'dup_{method}'

    if len(new_content.split()) < 5:
        return 'reject', 'too_few_words'
    if any(m in new_content.lower() for m in
           ['corp','inc','llc','ltd','plc','holdings','corporation','incorporated']):
        return 'reject', 'company_name'
    return 'add', 'ok'


### Bullet insertion and curator


In [ ]:
def _inject_bullet(pb_str, section, bid, content):
    target = _section_header(section)
    lines = pb_str.splitlines()
    out = []
    inserted = False
    in_target = False
    for i, line in enumerate(lines):
        out.append(line)
        if line.strip() == target:
            in_target = True
            continue
        if in_target and (line.startswith('## ') or i == len(lines) - 1):
            if i == len(lines) - 1 and not line.startswith('## '):
                out.append(_bullet_line(bid, content))
            else:
                out.insert(-1, _bullet_line(bid, content))
            inserted = True
            in_target = False
    if not inserted:
        out.append(target)
        out.append(_bullet_line(bid, content))
    return '\n'.join(out)

def run_curator(strategy, error_type, pb_str, next_id_map):
    action, reason = quality_gate(strategy, pb_str)
    if action == 'reject':
        return pb_str, next_id_map, 'reject', reason
    sec = ERROR_TO_SECTION.get(error_type, 'common_errors')
    if SECTION_QUOTA_PCT > 0 and _section_over_quota(pb_str, sec):
        sec = _pick_fallback_section(pb_str, sec)
    slug = SECTION_SLUGS.get(sec, 'ce')
    next_n = next_id_map.get(slug, 0) + 1
    bid = f"{slug}-{next_n:05d}"
    next_id_map[slug] = next_n
    return _inject_bullet(pb_str, sec, bid, strategy), next_id_map, 'add', 'added'


### Diagnose failure


In [ ]:
_OP_RE_DIAG = re.compile(
    r'(add|subtract|multiply|divide|greater|exp|table_\w+)\(', re.I)
_TABLE_OPS = ('table_max', 'table_min', 'table_sum', 'table_average')

def _diagnose_failure(raw_text, pred_prog, pred_ans, gold_prog, gold_ans):
    if not pred_prog:
        return ('format_ok_but_no_program' if ('```' in raw_text or 'program:' in raw_text.lower())
                else 'no_program_at_all')
    if pred_ans is None:
        return 'program_extracted_but_exec_fail'
    try:
        if gold_ans is not None:
            p, g = float(pred_ans), float(gold_ans)
            if g != 0 and abs(abs(p)-abs(g))/abs(g) < 0.01 and (p*g < 0):
                return 'sign_error'
            if g != 0 and p != 0:
                ratio = p / g
                for f in [10, 100, 1000, 10000, 100000, 1000000]:
                    if (abs(abs(ratio)-f)/f < 0.05 or abs(abs(ratio)-1/f)*f < 0.05):
                        return 'magnitude_error'
    except:
        pass
    pp, gp = (pred_prog or '').lower(), (gold_prog or '').lower()
    n_pred, n_gold = len(_OP_RE_DIAG.findall(pp)), len(_OP_RE_DIAG.findall(gp))
    if n_gold and n_pred and n_pred < n_gold:
        return 'missed_step'
    if n_gold and n_pred > n_gold:
        return 'extra_step'
    if any(o in gp for o in _TABLE_OPS) != any(o in pp for o in _TABLE_OPS):
        return 'wrong_aggregate'
    if n_pred == n_gold and n_pred >= 1:
        return 'wrong_direct_value'
    return 'wrong_value'


### Counterfactual retrieval


In [ ]:
USE_COUNTERFACTUAL          = True
COUNTERFACTUAL_N            = 3
COUNTERFACTUAL_MIN_HISTORY  = 30
COUNTERFACTUAL_MIN_CASES    = 2
COUNTERFACTUAL_OUTCOMES = {
    'wrong_reasoning', 'lucky_guess', 'missed_step', 'extra_step',
    'wrong_aggregate', 'exec_mismatch', 'magnitude_error',
    'sign_error', 'wrong_direct_value',
}

CF_W_SEM      = 0.40
CF_W_COMPLEX  = 0.30
CF_W_OPS      = 0.30

def _extract_ops_set(prog):
    return set(re.findall(
        r'(add|subtract|multiply|divide|greater|exp|table_\w+)',
        (prog or '').lower()))

def _retrieve_success_cases(question, history_list, k=COUNTERFACTUAL_N,
                             current_n_steps=None, current_prog=None,
                             exact_complexity=None, **kwargs):
    candidates = []
    for h in history_list:
        if h.get('outcome') != 'correct':
            continue
        gp = h.get('gold_prog', '') or ''
        if not gp:
            continue
        n_steps = len(_OP_RE_DIAG.findall(gp))
        candidates.append({'q': h.get('q','')[:120], 'gold_prog': gp,
                            'gold_ans': h.get('gold_ans'), 'n_steps': n_steps,
                            'ops': _extract_ops_set(gp)})
    if not candidates:
        return []

    if current_n_steps is None and exact_complexity is not None:
        current_n_steps = exact_complexity
    current_ops = _extract_ops_set(current_prog) if current_prog is not None else None

    q_emb = None
    if _EMBED_MODEL is not None:
        cache = globals().get('_train_q_embs', {})
        q_emb = cache.get(question[:80])
        if q_emb is None:
            q_emb = _EMBED_MODEL.encode([_BGE_QUERY_PREFIX + question],
                                         normalize_embeddings=True)[0]

    scored = []
    for c in candidates:
        if q_emb is not None:
            cache = globals().get('_train_q_embs', {})
            c_emb = cache.get(c['q'][:80])
            if c_emb is None:
                c_emb = _EMBED_MODEL.encode([_BGE_QUERY_PREFIX + c['q']],
                                              normalize_embeddings=True)[0]
            sem = float(c_emb @ q_emb)
            sem = (sem + 1) / 2
        else:
            q_words = set(question.lower().split())
            sem = (len(q_words & set(c['q'].lower().split()))
                    / max(1, len(q_words | set(c['q'].lower().split()))))

        if current_n_steps is not None:
            comp = (1.0 if c['n_steps'] == current_n_steps
                    else max(0, 1 - abs(c['n_steps'] - current_n_steps) / 3))
        else:
            comp = 0.5

        if current_ops is not None and (current_ops or c['ops']):
            union = current_ops | c['ops']
            ops_sim = len(current_ops & c['ops']) / len(union) if union else 0.5
        else:
            ops_sim = 0.5

        score = CF_W_SEM * sem + CF_W_COMPLEX * comp + CF_W_OPS * ops_sim
        scored.append((c, score))

    scored.sort(key=lambda x: -x[1])
    if current_n_steps is not None:
        scored = [(c, s) for c, s in scored if abs(c['n_steps'] - current_n_steps) <= 1]
    return [c for c, _ in scored[:k]]

def _format_success_cases(cases):
    if not cases:
        return ""
    lines = []
    for i, c in enumerate(cases, 1):
        ans = c.get('gold_ans')
        lines.append(f"  Case {i} ({c['n_steps']}-step): {c['q']}")
        lines.append(f"    program: {c['gold_prog']}")
        lines.append(f"    answer:  {ans if ans is not None else '(unknown)'}")
    return "\n".join(lines)

def _summarize_existing_patterns(pb_str, max_show=12):
    bullets = _all_bullets(pb_str)
    if not bullets:
        return "(empty playbook)"
    out = []
    for b in bullets[:max_show]:
        c = re.sub(r'^\s*\{[^}]*\}\s*', '', b['content'])
        ops = re.findall(r'(add|subtract|multiply|divide|greater|exp|table_\w+)\(', c.lower())
        ops_str = '→'.join(ops[:4]) if ops else 'no_ops'
        m_trig = re.match(r'^\s*(?:for|when|if|in|on)\s+([^,.;:]{1,40})', c, re.I)
        trig = m_trig.group(1).strip() if m_trig else "(no trigger)"
        out.append(f"  - [{b['id']}] '{trig}' → {ops_str}")
    return '\n'.join(out)


### Reflector prompts and backends


In [ ]:
REFLECTOR_PROMPT = """You are analyzing why a financial QA model made an error. Your goal is to create a GENERAL strategy that helps with similar problems — NOT a solution to this specific problem.

=== FINQA DSL RULES ===
- Flat format: op1(a,b), op2(#0,c). No nesting.
- Percentages are decimals: 0.41 not 41. Only use const_100 for explicit unit conversion.
- subtract(a,b) = a-b. For change: subtract(new, old), even if negative.

=== AVAILABLE CONSTANTS (use ONLY these) ===
const_1, const_2, const_3, const_4, const_5, const_6, const_7, const_8,
const_9, const_10, const_12, const_100, const_1000, const_10000,
const_100000, const_1000000, const_5_5, const_3_125, const_60.

DO NOT invent placeholder constants like const_x, const_n, const_total,
const_revenue. Use actual numeric constants OR named operands
(old_value, new_value, total, part, value1, value2, v1, v2, w1, w2,
total_w, a1, a2, b1, b2, ...).

=== COMMON FORMULA TRAPS — DO NOT FALL FOR THESE ===
- Average of N values: divide(sum, N) — NOT add(sum, const_N), divide(#0, const_N)
- Average of two: divide(add(v1, v2), const_2) — NOT add(v1, v2), divide(#0, const_2) padded
- Range: subtract(max, min) — NOT add or multiply
- Percentage: keep as decimal (0.30) — NOT multiply by const_100

=== CLUSTER ANALYSIS ===
This failure belongs to cluster: **{cluster_id}**
Cluster description : {cluster_lesson}
Canonical correct pattern: `{cluster_pattern}`
Top wrong patterns observed in this cluster:
{cluster_wrong_patterns}

Your bullet should HELP cases in this specific cluster. The retrieval
system will surface your bullet primarily for queries matching this
cluster signature, so the trigger phrase should match the cluster topic.

=== THIS CASE ===
Question: {q}
Context: {ctx}
Gold program: {gp}  →  Gold answer: {ga}
Model program: {pp}  →  Model answer: {pa}
Failure type: {diag}
Available strategies retrieved:
{bul}

=== EXISTING PLAYBOOK PATTERNS (DO NOT DUPLICATE) ===
{existing_patterns}

=== YOUR TASK ===
Output ONLY a JSON with this schema:
{{
  "error_type": "wrong_number | wrong_operation | unit_mismatch | wrong_row_col | missing_step | wrong_formula | no_program | lucky_guess | exec_mismatch | wrong_value | magnitude_error | sign_error | missed_step | extra_step | wrong_aggregate | wrong_direct_value",
  "root_cause": "One specific sentence about the type of mistake.",
  "new_strategy": "A GENERAL rule for similar problems. See CRITICAL RULES below.",
  "bullet_tags": [{{"id": "xx-00001", "tag": "helpful|harmful|neutral"}}, ...]
}}

=== CRITICAL RULES FOR new_strategy ===
1. MUST be a GENERAL pattern, NOT a solution to this specific question.
2. MUST NOT contain any specific numbers from this case. Use placeholders.
3. MUST contain AT LEAST TWO DSL operation names WITH parentheses.
4. MUST NOT mention specific company names, column names, or row labels.
5. MUST apply to MANY similar questions in cluster {cluster_id}.
6. MUST be ≤ 180 characters (one concise sentence).
7. SPECIFY EVERY OPERAND. For divide(): always specify denominator EXPLICITLY.
8. DO NOT DUPLICATE EXISTING PATTERNS shown above.
9. PERCENTAGE: CHANGE → divide(#0, OLD); SHARE → divide(part, TOTAL).
10. DSL REFERENCE RULE: at op position i, #N valid only if N < i.
    ✓ op1(a,b), op2(#0,c)         ← #0 valid at position 1
    ✗ op1(a,b), op2(#1,c)         ← #1 INVALID at position 1
11. USE ONLY constants listed above. NO const_x, const_n, const_total.
12. AVOID FORMULA TRAPS listed above (especially average pseudo-formulas).
13. START with "When [trigger phrase from cluster]" to ensure proper retrieval.

=== ENCOURAGED PATTERNS ===
1-step / 2-step:
  - 1-step pct SHARE     : divide(part, total)
  - 2-step pct CHANGE    : subtract(new_value, old_value), divide(#0, old_value)
  - 2-step unit→millions : multiply(value, factor), divide(#0, const_1000000)

3-step:
  - 3-step range         : table_min(col, none), table_max(col, none), subtract(#1, #0)
  - 3-step difference    : subtract(a, b), subtract(c, d), subtract(#0, #1)
  - 3-step ratio chain   : subtract(new, old), divide(#0, old), multiply(#1, factor)
  - 3-step sum-then-pct  : add(value1, value2), add(#0, value3), divide(part, #1)
  - 3-step cumul. return : subtract(index_value, const_100), divide(#0, const_100), multiply(#1, factor)

4-step:
  - 4-step grouping       : op_a(a, b), op_b(c, d), op_c(#0, #1), divide(#2, total)
  - 4-step weighted avg   : multiply(v1, w1), multiply(v2, w2), add(#0, #1), divide(#2, total_w)
  - 4-step diff-of-diffs  : subtract(a1, a2), subtract(b1, b2), subtract(#0, #1)
                            (or: divide(#0, #1) for ratio of changes)
  - 4-step aggregate-pct  : add(a, b), add(#0, c), divide(part, #1)

5+step:
  - 5-step cumul-diff     : subtract(idx1, const_100), divide(#0, const_100),
                            subtract(idx2, const_100), divide(#2, const_100),
                            subtract(#1, #3)
  - 5-step pct-then-comp  : <2-step block A>, <2-step block B>, greater(#1, #3)"""

LUCKY_GUESS_ADD = """

⚠ LUCKY GUESS: Model's ANSWER matches gold by coincidence — its PROGRAM differs.
Gold: {gp}  |  Model: {pp}  |  Both give: {ga}
Your new_strategy must teach the CORRECT DSL pattern. Keep it ≤ 180 characters."""

EXEC_MISMATCH_ADD = """

⚠ EXEC MISMATCH: Model's PROGRAM matches gold, but ANSWER differs.
Gold: {gp} → {ga}  Model: {pp} → {pa}
Your new_strategy must teach HOW to correctly identify numbers from context."""

COUNTERFACTUAL_ADD = """

=== COUNTERFACTUAL ANALYSIS ===
Similar questions ({n_steps} steps) where model SUCCEEDED:
{success_cases}

Identify the EDGE CONDITION distinguishing the FAILED case.
Use "When [specific condition], use [specific pattern]" — be specific."""

MAGNITUDE_ADD = """

=== MAGNITUDE ERROR ===
Predicted answer is off by factor of 10/100/1000/etc.
Your new_strategy MUST address unit-handling explicitly.
Example: multiply(value, factor), divide(#0, const_1000000) when raw is in dollars but answer in millions."""

MISSED_STEP_ADD = """

=== MISSED STEP ===
Gold n_ops = {gold_n_ops}, model n_ops = {pred_n_ops}.
Model SKIPPED {gold_minus_pred} step(s). Common patterns missed:
  - 3-step range:        table_min, table_max, subtract(#1, #0)
  - 3-step ratio:        subtract(new, old), divide(#0, old), multiply(#1, factor)
  - 3-step sum-pct:      add(v1, v2), add(#0, v3), divide(part, #1)
  - 4-step weighted avg: multiply(v1, w1), multiply(v2, w2), add(#0, #1), divide(#2, total_w)
  - 4-step diff-of-diff: subtract(a1, a2), subtract(b1, b2), subtract(#0, #1)
Your new_strategy MUST teach the FULL multi-step decomposition with named operands."""

WRONG_OP_ADD = """

=== WRONG OPERATION ===
Gold uses: {gold_first_op}.  Model used: {pred_first_op}.
Your new_strategy MUST tie a question keyword to the correct operation."""

SIGN_ERROR_ADD = """

=== SIGN ERROR ===
Gold answer: {gold_ans}  Model answer: {pred_ans}  Gold prog: {gold_prog}
Model reversed subtract() operands. Use subtract(later, earlier)."""

THREE_STEP_ADD = """

=== {n_steps}-STEP DECOMPOSITION (gold) ===
Gold program needs {n_steps} sequential ops. Specify each step with named operands.
REMINDER: #N at op position i requires N < i.
For 4-step: op1(a,b), op2(c,d), op3(#0, #1), op4(#2, x)
For 5-step: op1(a,b), op2(#0,c), op3(d,e), op4(#2,f), op5(#1, #3)"""

VERIFY_ITERATE_FEEDBACK = """

=== PREVIOUS ATTEMPT FEEDBACK (Round {round_num}) ===
Your previous bullet was rejected because: {feedback_reason}
Previous bullet content: "{previous_bullet}"

Adjust your new_strategy to AVOID this issue. Do NOT submit a paraphrase
of the previous attempt — the reviewer detects paraphrases via Jaccard
similarity. Make a substantive change in trigger phrase OR operand
specification OR pattern."""

THINKING_TRACE_BLOCK = """

=== SLM REASONING TRACE ===
Below is the model's internal reasoning style for this question type.
Note: this trace was captured in a fresh forward pass and may correspond
to a slightly different solution path than the wrong program above.
Treat it as APPROXIMATE evidence of the model's reasoning patterns.

<think>
{thinking_trace}
</think>

Use this to identify reasoning errors that produced the wrong program.
For example, if trace shows model picking the wrong year as "earlier_value",
your bullet should encode the year-disambiguation rule."""

def _count_ops(prog):
    return len(_OP_RE_DIAG.findall(prog or ''))

def _first_op(prog):
    if not prog:
        return '(none)'
    m = _OP_RE_DIAG.search(prog)
    return m.group(1).lower() if m else '(none)'

def _format_cluster_wrong_patterns(cluster_def):
    wrongs = cluster_def.get('top_wrong_patterns', [])
    if not wrongs:
        return "  (no patterns recorded — generic FinQA rules apply)"
    return '\n'.join(f"  - {w}" for w in wrongs[:4])

def build_reflector_prompt_outcome_aware(q, ctx, pp, pa, gp, ga, bul, diag, outcome,
                                          success_cases=None, pb_str=None,
                                          cluster_def=None,
                                          verify_feedback=None,
                                          thinking_trace=None):
    if cluster_def is None:
        cluster_def = _CLUSTER_BY_ID.get('C12_misc_other', {})

    base = REFLECTOR_PROMPT.format(
        q=q, ctx=ctx[:600], pp=pp or '(none)', pa=str(pa),
        gp=gp, ga=str(ga), bul=bul or '(none)', diag=diag,
        cluster_id=cluster_def.get('id', 'C12_misc_other'),
        cluster_lesson=cluster_def.get('lesson', '(no lesson)'),
        cluster_pattern=cluster_def.get('canonical_pattern', '(no canonical)'),
        cluster_wrong_patterns=_format_cluster_wrong_patterns(cluster_def),
        existing_patterns=_summarize_existing_patterns(pb_str)
                           if pb_str else "(no playbook context)")

    if outcome == 'lucky_guess':
        base += LUCKY_GUESS_ADD.format(gp=gp, pp=pp, ga=ga)
    elif outcome == 'exec_mismatch' and globals().get('USE_EXEC_MISMATCH_BRANCH', False):
        base += EXEC_MISMATCH_ADD.format(gp=gp, pp=pp, ga=ga, pa=pa)

    if diag == 'magnitude_error':
        base += MAGNITUDE_ADD
    elif diag == 'missed_step':
        n_gold = _count_ops(gp)
        n_pred = _count_ops(pp)
        base += MISSED_STEP_ADD.format(gold_n_ops=n_gold, pred_n_ops=n_pred,
                                         gold_minus_pred=n_gold - n_pred)
    elif diag in ('wrong_direct_value', 'extra_step'):
        base += WRONG_OP_ADD.format(gold_first_op=_first_op(gp), pred_first_op=_first_op(pp))
    elif diag == 'sign_error':
        base += SIGN_ERROR_ADD.format(gold_ans=str(ga), pred_ans=str(pa), gold_prog=str(gp))

    n_gold = _count_ops(gp)
    if n_gold >= 3:
        base += THREE_STEP_ADD.format(n_steps=n_gold)

    if (success_cases and len(success_cases) >= COUNTERFACTUAL_MIN_CASES
            and outcome in COUNTERFACTUAL_OUTCOMES):
        sc = _format_success_cases(success_cases)
        if sc:
            base += COUNTERFACTUAL_ADD.format(success_cases=sc, n_steps=success_cases[0]['n_steps'])

    if verify_feedback:
        base += VERIFY_ITERATE_FEEDBACK.format(
            round_num=verify_feedback.get('round', 1),
            feedback_reason=verify_feedback.get('reason', '(no reason)'),
            previous_bullet=verify_feedback.get('previous_bullet', '(none)')[:140])

    if thinking_trace and len(thinking_trace.strip()) > 30:
        base += THINKING_TRACE_BLOCK.format(thinking_trace=thinking_trace[:2000])

    return base

HYBRID_MODEL_REFLECTOR = globals().get('HYBRID_MODEL_REFLECTOR', 'gpt-4o-mini')

def _reflector_slm(q, ctx, pp, pa, gp, ga, bul, diag, outcome='wrong_reasoning',
                   success_cases=None, pb_str=None, cluster_def=None,
                   verify_feedback=None, thinking_trace=None):
    prompt = build_reflector_prompt_outcome_aware(
        q, ctx, pp, pa, gp, ga, bul, diag, outcome,
        success_cases, pb_str, cluster_def, verify_feedback, thinking_trace)
    sp = SamplingParams(temperature=REFLECTOR_TEMP, max_tokens=600,
                         repetition_penalty=1.1, skip_special_tokens=True,
                         seed=RANDOM_SEED)
    is_qwen3 = "qwen3" in MODEL_NAME.lower()
    formatted = tokenizer.apply_chat_template(
        [{"role":"user","content":prompt}], tokenize=False, add_generation_prompt=True,
        **({"enable_thinking":False} if is_qwen3 else {}))
    return model.fast_generate([formatted], sampling_params=sp)[0].outputs[0].text, None

def _reflector_hybrid(q, ctx, pp, pa, gp, ga, bul, diag, outcome='wrong_reasoning',
                      success_cases=None, pb_str=None, cluster_def=None,
                      verify_feedback=None, thinking_trace=None):
    from openai import OpenAI
    prompt = build_reflector_prompt_outcome_aware(
        q, ctx, pp, pa, gp, ga, bul, diag, outcome,
        success_cases, pb_str, cluster_def, verify_feedback, thinking_trace)
    try:
        resp = OpenAI(api_key=HYBRID_API_KEY).chat.completions.create(
            model=HYBRID_MODEL_REFLECTOR,
            messages=[{"role":"user","content":prompt}],
            temperature=REFLECTOR_TEMP,
            response_format={"type": "json_object"},
            seed=RANDOM_SEED,
            max_tokens=600)
        return resp.choices[0].message.content, resp.usage
    except Exception as e:
        print(f"[HYBRID REF] ⚠ {e}")
        return None, None

def reflector(q, ctx, pp, pa, gp, ga, bul, diag, outcome='wrong_reasoning',
              success_cases=None, pb_str=None, cluster_def=None,
              verify_feedback=None, thinking_trace=None):
    fn = _reflector_hybrid if USE_HYBRID_REFLECTOR else _reflector_slm
    return fn(q, ctx, pp, pa, gp, ga, bul, diag, outcome,
              success_cases, pb_str, cluster_def, verify_feedback, thinking_trace)


### Self-tests


In [ ]:
print("\n[ACE COMPONENTS] Running self-tests...")

# Cluster classifier tests
_test_samples = [
    ({'qa': {'question': 'what percent of total revenue was X?',
              'program': 'divide(45, 100)'}},
     'C1_pct_share_div'),
    ({'qa': {'question': 'what was the percentage change in revenue from 2018 to 2019?',
              'program': 'subtract(500, 450), divide(#0, 450)'}},
     'C2_pct_change_2step'),
    ({'qa': {'question': 'what was the change in net income?',
              'program': 'subtract(1200, 1100)'}},
     'C3_difference_simple'),
    ({'qa': {'question': 'what is the ratio of A to B?',
              'program': 'divide(50, 25)'}},
     'C5_ratio_simple'),
    ({'qa': {'question': 'what was the cumulative return on the index?',
              'program': 'subtract(150, const_100), divide(#0, const_100)'}},
     'C7_cumulative_const100'),
]
for sample, expected in _test_samples:
    got = cluster_id_for_sample(sample)
    status = '✓' if got == expected else f'✗ (got {got})'
    print(f"  {status} {expected:<28} on '{sample['qa']['question'][:50]}...'")

# 4-step cluster tests (only if v3 phase0 is loaded)
if 'C13_multistep_weighted' in _CLUSTER_BY_ID:
    _test_4step = [
        ({'qa': {'question': 'what is the weighted average return across funds?',
                  'program': 'multiply(10, 2), multiply(20, 3), add(#0, #1), divide(#2, 5)'}},
         'C13_multistep_weighted'),
    ]
    for sample, expected in _test_4step:
        got = cluster_id_for_sample(sample)
        status = '✓' if got == expected else f'✗ (got {got})'
        print(f"  {status} {expected:<28} on '{sample['qa']['question'][:50]}...'")

no_parens = "When comparing values, always ensure to use the correct category for subtraction and verify the context."
assert not _has_dsl_chain(no_parens)
qg_tr1 = quality_gate(no_parens, INITIAL_PLAYBOOK)
assert qg_tr1[0] == 'reject' and 'no_dsl_chain' in qg_tr1[1]

unworded_bug = "Use add(value1, value2), add(#0, const_2), divide(#1, const_2) for some calculation."
assert _has_buggy_avg_padding(unworded_bug)

correct_avg2 = "For averages, use add(value1, value2), divide(#0, const_2)."
ok, _ = _validate_bullet_synthetically(correct_avg2)
assert ok

# 4-step weighted avg test
weighted_bullet = "When weighted average is needed, use multiply(v1, w1), multiply(v2, w2), add(#0, #1), divide(#2, total_w)."
ok_w, reason_w = _validate_bullet_synthetically(weighted_bullet)
assert ok_w, f"4-step weighted avg should pass synth: {reason_w}"
print(f"  ✓ 4-step weighted avg synth validation passes")

test_cluster = _CLUSTER_BY_ID.get('C2_pct_change_2step',
                                    _CLUSTER_BY_ID.get('C12_misc_other'))
test_prompt = build_reflector_prompt_outcome_aware(
    q="test", ctx="test", pp="test", pa="0", gp="test", ga="0",
    bul="", diag="wrong_value", outcome="wrong_reasoning",
    cluster_def=test_cluster)
assert 'cluster' in test_prompt.lower()
assert test_cluster.get('id', 'C12') in test_prompt
assert '4-step' in test_prompt and '5+step' in test_prompt  # encouraged patterns
print(f"  ✓ Reflector prompt cluster injection + 4/5-step patterns present")

# Tier 1/2 retrieval system
_tier1_bullets.clear()
test_pb = """## Numerical Strategies
- [ns-00001] When pct change, use subtract(new, old), divide(#0, old).
- [ns-00002] When pct share, use divide(part, total) directly.
## Program Generation
- [pg-00001] When weighted avg, use multiply(v1, w1), multiply(v2, w2), add(#0, #1), divide(#2, total_w).
"""
# Without Tier 1, behavior = flat top-k
text_flat, ids_flat = _retrieve_tier1_tier2("pct change in revenue", test_pb,
                                              k_tier1=2, k_tier2=1)
assert len(ids_flat) == 3, f"Flat fallback should return 3: got {len(ids_flat)}"

# With Tier 1
_promote_to_tier1('ns-00001')
text_t1, ids_t1 = _retrieve_tier1_tier2("pct share calculation", test_pb,
                                          k_tier1=2, k_tier2=1)
assert 'ns-00001' in ids_t1, f"Tier 1 bullet must always appear: {ids_t1}"
print(f"  ✓ Tier 1/2 retrieval works (Tier 1 always in result)")
_tier1_bullets.clear()

print()
print("[ACE COMPONENTS] Self-tests passed")
print(f"  Embedding   : BAAI/bge-base-en-v1.5 (dim={_EMBED_DIM})")
print(f"  Retrieval   : sem={RETRIEVAL_W_SEM} bm25={RETRIEVAL_W_BM25} val={RETRIEVAL_W_VALUE} freq={RETRIEVAL_W_FREQ}")
print(f"  Tier system : T1_max={globals().get('TIER_1_MAX', 5)} | T1_k={globals().get('TOP_K_RETRIEVAL_TIER1', 4)} | T2_k={globals().get('TOP_K_RETRIEVAL_TIER2', 3)}")
print(f"  Cluster mode: {globals().get('CLUSTER_MATCH_MODE', 'highest_score')} | thr={globals().get('CLUSTER_MATCH_THRESHOLD', 3)}/5")
print(f"  Strict regex: {len(globals().get('STRICT_REGEX_CLUSTERS', set()))} clusters require regex match")
print(f"  Reflector   : {HYBRID_MODEL_REFLECTOR}, cluster-aware + 4/5-step patterns")
print(f"  Phase 0     : {_PHASE0_CLUSTERS_DATA['n_clusters']} clusters loaded (v={_PHASE0_CLUSTERS_DATA.get('version', 'n/a')})")
print(f"  Test cats   : {len(_TEST_CATEGORIES)} synthetic validation categories (including 4-step cases)")


## Curator and lift tracking


In [ ]:
import math
import re
import hashlib
import random
from collections import defaultdict


### Curator configuration


In [ ]:
USE_MULTI_STAGE_CURATOR         = True
VALIDATION_N_SAMPLES            = globals().get('VALIDATION_N_SAMPLES', 40)
VALIDATION_MIN_HISTORY          = 20
VALIDATION_MIN_SAMPLES_REQUIRED = 8

TRIGGER_OVERLAP_THRESH          = 0.75
ACTION_SIG_DEDUP_LEN            = 3
ACTION_SIG_TRIG_OVERLAP_THR     = 0.55

RARE_ERRORS = {'magnitude_error', 'sign_error', 'missed_step',
               'wrong_operation', 'extra_step'}
# Noise-tolerant validation thresholds.
COMMON_ERRORS_THRESHOLD  = globals().get('COMMON_ERRORS_THRESHOLD',  -0.005)
RARE_ERRORS_THRESHOLD    = globals().get('RARE_ERRORS_THRESHOLD',    -0.015)
VALIDATION_HARM_PA_TOL = globals().get('VALIDATION_HARM_PA_TOL', 3)

V95_W_EA = 0.40
V95_W_PA = 0.60

QUARANTINE_COOLDOWN = globals().get('QUARANTINE_COOLDOWN', 200)
_evicted_quarantine = globals().get('_evicted_quarantine', {})

# dev-based ablate (replaces history-based)
ABLATE_DATA_SOURCE       = globals().get('ABLATE_DATA_SOURCE', 'dev')
ABLATE_MIN_USES          = globals().get('ABLATE_MIN_USES', 10)
ABLATE_PA_LIFT_THR       = globals().get('ABLATE_PA_LIFT_THR', -0.02)
ABLATE_MIN_PLAYBOOK_SIZE = 10
ABLATE_WINDOW            = 100  # History-based fallback window

# 5-bucket stratification (1/2/3/4/5+ step)
_STRAT_TARGETS = {1: 0.20, 2: 0.25, 3: 0.25, 4: 0.20, 5: 0.10}

MAX_BULLETS_PER_CLUSTER = 2
MAX_BULLETS_C12 = 4
ABLATE_PROTECT_NON_C12 = True
_bullet_to_cluster = globals().get('_bullet_to_cluster', {})

# dev lift tracking config
USE_DEV_LIFT_TRACKING  = globals().get('USE_DEV_LIFT_TRACKING', True)
LIFT_EVAL_EVERY        = globals().get('LIFT_EVAL_EVERY', 60)
LIFT_DEV_SET_SIZE      = globals().get('LIFT_DEV_SET_SIZE', 100)
LIFT_EMA_ALPHA         = globals().get('LIFT_EMA_ALPHA', 0.4)

# Tier 1 promotion
TIER_1_MAX             = globals().get('TIER_1_MAX', 5)
TIER_1_PROMOTION_AGE   = globals().get('TIER_1_PROMOTION_AGE', 200)
TIER_1_PROMOTION_LIFT  = globals().get('TIER_1_PROMOTION_LIFT', 0.01)
TIER_1_FALLBACK_AGE    = globals().get('TIER_1_FALLBACK_AGE', 350)
TIER_1_FALLBACK_LIFT   = globals().get('TIER_1_FALLBACK_LIFT', 0.0)


### Stats tracking


In [ ]:
multistage_stats = {
    'stage1_pass': 0,
    'stage2_eval': 0, 'stage2_skip_no_history': 0, 'stage2_skip_few_samples': 0,
    'stage2_accept': 0,
    'stage2_reject_no_improvement': 0, 'stage2_reject_harmful': 0,
    'stage2_reject_lucky_regression': 0,
    'stage3_pass': 0,
    'stage3_reject_trigger_overlap': 0, 'stage3_reject_action_signature': 0,
    'final_accept': 0,
    'auto_ablate_calls': 0, 'auto_ablate_evicted': 0, 'auto_ablate_skipped_small': 0,
    'auto_ablate_skipped_tier1': 0,
    'delta_ea_log': [], 'delta_pa_log': [], 'reject_examples': [],
    'post_train_prune_evaluated': 0,
    'post_train_prune_removed':   0,
    'post_train_prune_log':       [],
    'stage0_5_cluster_quota_full': 0,
    'cluster_distribution':        {},
    'lift_evals_run':              0,
    'tier1_promotions':            0,
    'tier1_demotions':             0,
    'tier1_fallback_promotions':   0,
}


### Quarantine (cooldown extended to 200 via global)


In [ ]:
def _content_hash(content):
    return hashlib.md5(' '.join(content.lower().split()).encode()).hexdigest()[:12]

def _is_quarantined(content, current_step):
    h = _content_hash(content)
    if h not in _evicted_quarantine:
        return False
    if current_step - _evicted_quarantine[h] > QUARANTINE_COOLDOWN:
        del _evicted_quarantine[h]
        return False
    return True

def _add_to_quarantine(content, step):
    _evicted_quarantine[_content_hash(content)] = step


### Cluster quota helpers


In [ ]:
def _bullets_in_cluster(cluster_id):
    return sum(1 for cid in _bullet_to_cluster.values() if cid == cluster_id)

def _cluster_quota_full(cluster_id):
    cap = MAX_BULLETS_C12 if cluster_id == 'C12_misc_other' else MAX_BULLETS_PER_CLUSTER
    return _bullets_in_cluster(cluster_id) >= cap

def _assign_bullet_to_cluster(bullet_id, cluster_id):
    _bullet_to_cluster[bullet_id] = cluster_id

def _unassign_bullet(bullet_id):
    _bullet_to_cluster.pop(bullet_id, None)
    # also demote from Tier 1 if present
    if '_tier1_bullets' in globals():
        _demote_from_tier1(bullet_id) if '_demote_from_tier1' in globals() else None

def cluster_distribution_snapshot():
    snap = defaultdict(int)
    for cid in _bullet_to_cluster.values():
        snap[cid] += 1
    return dict(snap)


### Dev lift tracking


In [ ]:
# Maintains a fixed mini-dev set (stratified by n_ops) used to compute
# per-bullet PA-lift and EA-lift via leave-one-out evaluation.
# Cached as EMA so noise smooths over multiple eval rounds.

# Persistent state across eval rounds
_dev_lift_set        = globals().get('_dev_lift_set', None)  # list of samples
_dev_lift_ema_cache  = globals().get('_dev_lift_ema_cache', {})  # bid → {pa_lift, ea_lift, n_evals, last_step}

def build_dev_lift_set(dev_samples, size=None, seed=RANDOM_SEED):
    """Build fixed mini-dev set stratified by n_ops (1/2/3/4/5+).

    Stratify by n_ops gives balanced complexity coverage — better than
    random or by cluster (cluster dist is skewed toward C12).
    """
    global _dev_lift_set
    if size is None:
        size = LIFT_DEV_SET_SIZE
    rng = random.Random(seed)
    by_n = {1: [], 2: [], 3: [], 4: [], 5: []}
    for s in dev_samples:
        n = len(_OP_RE_DIAG.findall(s.get('qa', {}).get('program', '')))
        bucket = min(max(n, 1), 5)
        by_n[bucket].append(s)
    for k in by_n:
        rng.shuffle(by_n[k])
    # Allocate proportional to actual dev distribution, but enforce min 5 per bucket
    total_avail = sum(len(v) for v in by_n.values())
    raw_targets = {k: max(5, int(round(size * len(v) / max(1, total_avail))))
                    for k, v in by_n.items()}
    while sum(raw_targets.values()) > size:
        # Trim from largest bucket
        big = max(raw_targets, key=raw_targets.get)
        if raw_targets[big] > 5:
            raw_targets[big] -= 1
        else:
            break
    selected = []
    for k, n_target in raw_targets.items():
        selected.extend(by_n[k][:n_target])
    _dev_lift_set = selected[:size]
    print(f"[LIFT-SET] Built fixed dev mini-set: {len(_dev_lift_set)} samples, "
          f"by n_ops: " + ", ".join(f"n{k}={raw_targets.get(k,0)}" for k in sorted(raw_targets)))
    return _dev_lift_set

def get_dev_lift_set():
    """Return the fixed dev lift set; build if not yet present."""
    global _dev_lift_set
    if _dev_lift_set is None:
        if 'dev_valid' not in globals():
            return []
        _dev_lift_set = build_dev_lift_set(globals()['dev_valid'])
    return _dev_lift_set

def eval_per_bullet_dev_lift(playbook_str, current_step):
    """Compute leave-one-out PA and EA lift for each bullet on fixed dev mini-set.

    Approach:
      1. Eval full PB on dev mini-set → ea_full, pa_full
      2. For each bullet b: eval (PB minus b) on same set → ea_wo_b, pa_wo_b
      3. lift_pa(b) = pa_full - pa_wo_b   (positive = b helps)
      4. EMA update: cache[b] = α·new + (1-α)·cache[b]

    Cost: ~(N_bullets + 1) × |dev_set| forward passes. With 30 bullets × 100 samples × 2
    (with/without) we do ~3100 prompts. vLLM batch=64 on A100-40GB ≈ 10-15 minutes.

    Returns dict: bid → {'pa_lift': EMA, 'ea_lift': EMA, 'n_evals': N, 'last_step': step}
    """
    if not USE_DEV_LIFT_TRACKING:
        return {}

    dev_set = get_dev_lift_set()
    if not dev_set:
        return _dev_lift_ema_cache

    bullets = _all_bullets(playbook_str)
    if not bullets:
        return _dev_lift_ema_cache

    multistage_stats['lift_evals_run'] += 1

    # Build all prompts in one batch
    metas = [{'gold_ans': s['qa'].get('exe_ans'),
              'gold_prog': s['qa'].get('program', ''),
              'table': s.get('table', [])} for s in dev_set]

    # 1. Full PB prompts
    prompts_full = []
    for s in dev_set:
        bul = retrieve_top_k(s['qa']['question'], playbook_str)
        prompts_full.append(build_ace_prompt(s, playbook_bullets=bul))

    # 2. Leave-one-out prompts (one block per bullet)
    prompts_lo = []
    bullet_block_starts = {}
    for b in bullets:
        if is_tier1(b['id']):
            # Skip lift eval for Tier 1 — they're locked, won't be evicted
            continue
        bullet_block_starts[b['id']] = len(prompts_lo)
        pb_lo = '\n'.join(line for line in playbook_str.splitlines()
                          if f"[{b['id']}]" not in line)
        for s in dev_set:
            bul_lo = retrieve_top_k(s['qa']['question'], pb_lo)
            prompts_lo.append(build_ace_prompt(s, playbook_bullets=bul_lo))

    # Run all generates (full + LO)
    sp = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                         repetition_penalty=REPETITION_PENALTY,
                         skip_special_tokens=True, seed=RANDOM_SEED)
    n_full = len(prompts_full)
    n_lo = len(prompts_lo)
    print(f"[LIFT-EVAL] step={current_step} | dev_set={len(dev_set)} | "
          f"non-T1 bullets={len(bullet_block_starts)} | total prompts={n_full + n_lo}")

    out_full = model.fast_generate(prompts_full, sampling_params=sp)
    if n_lo > 0:
        out_lo = model.fast_generate(prompts_lo, sampling_params=sp)
    else:
        out_lo = []

    # Score full
    ea_full = pa_full = 0
    for o, m in zip(out_full, metas):
        pp = extract_program(o.outputs[0].text)
        pv = execute_program(pp, m['table']) if pp else None
        if (m['gold_ans'] is not None and pv is not None
                and check_ea(pv, m['gold_ans'], EA_DECIMAL_PLACES)):
            ea_full += 1
        if check_pa(pp, m['gold_prog']):
            pa_full += 1
    ea_full_rate = ea_full / max(1, len(dev_set))
    pa_full_rate = pa_full / max(1, len(dev_set))

    # Score each leave-one-out block, update EMA
    n = len(dev_set)
    new_lifts = {}
    for bid, start in bullet_block_starts.items():
        end = start + n
        ea_wo = pa_wo = 0
        for o, m in zip(out_lo[start:end], metas):
            pp = extract_program(o.outputs[0].text)
            pv = execute_program(pp, m['table']) if pp else None
            if (m['gold_ans'] is not None and pv is not None
                    and check_ea(pv, m['gold_ans'], EA_DECIMAL_PLACES)):
                ea_wo += 1
            if check_pa(pp, m['gold_prog']):
                pa_wo += 1
        ea_lift_now = ea_full_rate - ea_wo / n
        pa_lift_now = pa_full_rate - pa_wo / n

        prev = _dev_lift_ema_cache.get(bid, {'pa_lift': 0.0, 'ea_lift': 0.0, 'n_evals': 0})
        if prev['n_evals'] == 0:
            # First eval — no smoothing
            ema_pa = pa_lift_now
            ema_ea = ea_lift_now
        else:
            ema_pa = LIFT_EMA_ALPHA * pa_lift_now + (1 - LIFT_EMA_ALPHA) * prev['pa_lift']
            ema_ea = LIFT_EMA_ALPHA * ea_lift_now + (1 - LIFT_EMA_ALPHA) * prev['ea_lift']
        _dev_lift_ema_cache[bid] = {
            'pa_lift':   round(ema_pa, 4),
            'ea_lift':   round(ema_ea, 4),
            'n_evals':   prev['n_evals'] + 1,
            'last_step': current_step,
            'pa_lift_raw_last': round(pa_lift_now, 4),
            'ea_lift_raw_last': round(ea_lift_now, 4),
        }
        new_lifts[bid] = _dev_lift_ema_cache[bid]

    # Print top/bottom 5
    if new_lifts:
        sorted_lifts = sorted(new_lifts.items(), key=lambda x: -x[1]['pa_lift'])
        print(f"  Top 3 by pa_lift_ema:")
        for bid, info in sorted_lifts[:3]:
            print(f"    +[{bid}] pa={info['pa_lift']:+.3f} ea={info['ea_lift']:+.3f} "
                  f"(n={info['n_evals']})")
        print(f"  Bottom 3:")
        for bid, info in sorted_lifts[-3:]:
            print(f"    -[{bid}] pa={info['pa_lift']:+.3f} ea={info['ea_lift']:+.3f} "
                  f"(n={info['n_evals']})")

    return _dev_lift_ema_cache

def get_bullet_dev_lift(bid, default=None):
    """Get cached EMA dev lift for a bullet."""
    rec = _dev_lift_ema_cache.get(bid)
    if rec is None:
        return default
    return rec


### Tier 1 promotion and demotion


In [ ]:
def promote_tier1_candidates(current_step, bullet_birth_step):
    """Check existing bullets for Tier 1 promotion.

    Criteria:
      - Bullet age ≥ TIER_1_PROMOTION_AGE (200 steps)
      - dev_lift.pa_lift ≥ TIER_1_PROMOTION_LIFT (0.01)
      - n_evals ≥ 2 (at least 2 lift evals to smooth noise)
      - Not already in Tier 1
      - Tier 1 not at MAX

    Fallback (after step 350 if Tier 1 < 3):
      - Relax to lift > 0 (not 0.01)

    Returns list of newly promoted bullet IDs.
    """
    if not globals().get('USE_TIER_SYSTEM', True):
        return []

    current_t1 = get_tier1_set() if 'get_tier1_set' in globals() else _get_tier1_bullets()
    if len(current_t1) >= TIER_1_MAX:
        return []

    # Determine threshold (fallback if Tier 1 too small)
    use_fallback = (current_step >= TIER_1_FALLBACK_AGE
                    and len(current_t1) < 3)
    lift_thr = TIER_1_FALLBACK_LIFT if use_fallback else TIER_1_PROMOTION_LIFT

    # Candidates
    candidates = []
    for bid, lift_info in _dev_lift_ema_cache.items():
        if bid in current_t1:
            continue
        if lift_info.get('n_evals', 0) < 2:
            continue
        age = current_step - bullet_birth_step.get(bid, current_step)
        if age < TIER_1_PROMOTION_AGE:
            continue
        if lift_info.get('pa_lift', 0) < lift_thr:
            continue
        if lift_info.get('pa_lift_raw_last', 0) < 0:
            continue
        candidates.append((bid, lift_info['pa_lift']))

    # Promote top-K (highest lift first)
    candidates.sort(key=lambda x: -x[1])
    n_slots = TIER_1_MAX - len(current_t1)
    promoted = []
    for bid, lift in candidates[:n_slots]:
        # Use the tier1 helper
        if '_promote_to_tier1' in globals():
            globals()['_promote_to_tier1'](bid)
        elif 'add_to_tier1' in globals():
            globals()['add_to_tier1'](bid)
        promoted.append(bid)
        multistage_stats['tier1_promotions'] += 1
        if use_fallback:
            multistage_stats['tier1_fallback_promotions'] += 1
        print(f"[TIER1-PROMOTE] step={current_step} [{bid}] "
              f"pa_lift={lift:+.3f} (thr={lift_thr}{', FALLBACK' if use_fallback else ''})")
    return promoted

def _demote_from_tier1(bid):
    """Compatibility wrapper for Tier 1 demotion."""
    if 'remove_from_tier1' in globals():
        globals()['remove_from_tier1'](bid)
    elif '_demote_from_tier1' in globals() and globals()['_demote_from_tier1'] is not _demote_from_tier1:
        # Avoid infinite recursion if the existing function is named differently
        globals()['_demote_from_tier1'](bid)
    elif '_tier1_bullets' in globals():
        globals()['_tier1_bullets'].discard(bid)
    multistage_stats['tier1_demotions'] += 1

def is_tier1(bid):
    """Compatibility wrapper for Tier 1 membership."""
    if 'is_tier1' in globals() and globals()['is_tier1'] is not is_tier1:
        return globals()['is_tier1'](bid)
    if '_is_tier1' in globals():
        return globals()['_is_tier1'](bid)
    if '_tier1_bullets' in globals():
        return bid in globals()['_tier1_bullets']
    return False

def _get_tier1_bullets():
    """Compatibility wrapper for Tier 1 access."""
    if 'get_tier1_set' in globals():
        return globals()['get_tier1_set']()
    if '_tier1_bullets' in globals():
        return set(globals()['_tier1_bullets'])
    return set()


### Stage 2: stratified validation sampling (5-bucket)


In [ ]:
def _select_validation_samples(error_type, current_sample, train_subset,
                                 n=None):
    """Select validation samples stratified by n_ops bucket.

    Use five complexity buckets and prioritize the current failure type.
    """
    if n is None:
        n = VALIDATION_N_SAMPLES
    if len(history) < VALIDATION_MIN_HISTORY:
        return []

    current_q80 = current_sample['qa']['question'][:80]
    current_n = len(_OP_RE_DIAG.findall(current_sample['qa'].get('program', '').lower()))

    by_complexity = {1: [], 2: [], 3: [], 4: [], 5: []}
    for h in history:
        if h.get('outcome') == 'correct':
            continue
        if h.get('q', '')[:80] == current_q80:
            continue
        gp = h.get('gold_prog', '') or ''
        if not gp:
            continue
        bucket = min(max(len(_OP_RE_DIAG.findall(gp.lower())), 1), 5)
        by_complexity[bucket].append(h)

    def _score(h):
        s = 0
        if h.get('error_type', '') == error_type:
            s += 3
        if min(max(len(_OP_RE_DIAG.findall(h.get('gold_prog', '').lower())), 1), 5) == min(max(current_n, 1), 5):
            s += 1
        return s

    for bucket in by_complexity:
        by_complexity[bucket].sort(key=_score, reverse=True)

    targets = {bucket: max(1, int(round(n * frac))) for bucket, frac in _STRAT_TARGETS.items()}
    while sum(targets.values()) > n:
        targets[max(targets, key=targets.get)] -= 1

    selected_steps, seen = [], set()
    for bucket, target in targets.items():
        taken = 0
        for h in by_complexity[bucket]:
            if taken >= target:
                break
            if h['step'] in seen:
                continue
            seen.add(h['step'])
            selected_steps.append(h['step'])
            taken += 1

    if len(selected_steps) < n:
        all_remaining = []
        for records in by_complexity.values():
            for h in records:
                if h['step'] not in seen:
                    all_remaining.append(h)
        all_remaining.sort(key=_score, reverse=True)
        for h in all_remaining:
            if len(selected_steps) >= n:
                break
            seen.add(h['step'])
            selected_steps.append(h['step'])

    selected = []
    for step in selected_steps:
        idx = (step - 1) % len(train_subset)
        if 0 <= idx < len(train_subset):
            selected.append(train_subset[idx])
        if len(selected) >= n:
            break
    return selected


### Stage 2: evaluate bullet on samples


In [ ]:
def _evaluate_bullet_on_samples(bullet_content, samples, current_pb_str):
    """Run with-vs-without comparison on N samples. Returns delta_ea, delta_pa."""
    if not samples:
        return {'n_samples': 0, 'delta_ea': 0.0, 'delta_pa': 0.0,
                'ea_without': 0, 'ea_with': 0, 'pa_without': 0, 'pa_with': 0,
                'details': [], 'lucky_guess_regression': 0}

    prompts_w, prompts_wo, metas = [], [], []
    for s in samples:
        q = s['qa']['question']
        # Without bullet: standard retrieval
        bul_wo = retrieve_top_k(q, current_pb_str)
        # With bullet: force candidate at top, drop one slot from retrieval
        # (uses TOP_K-1 to make room for forced bullet)
        top_k_minus = max(1, globals().get('TOP_K_RETRIEVAL', 7) - 1)
        bul_baseline = retrieve_top_k(q, current_pb_str, k=top_k_minus)
        bul_w = f"- {bullet_content}\n{bul_baseline}".strip()
        prompts_wo.append(build_ace_prompt(s, playbook_bullets=bul_wo))
        prompts_w.append(build_ace_prompt(s, playbook_bullets=bul_w))
        metas.append({'gold_ans': s['qa'].get('exe_ans'),
                       'gold_prog': s['qa']['program'],
                       'table': s.get('table', [])})

    sp = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                         repetition_penalty=REPETITION_PENALTY,
                         skip_special_tokens=True, seed=RANDOM_SEED)
    out_wo = model.fast_generate(prompts_wo, sampling_params=sp)
    out_w  = model.fast_generate(prompts_w,  sampling_params=sp)

    ea_wo = pa_wo = ea_w = pa_w = 0
    details = []
    for i, (ow, wt, m) in enumerate(zip(out_wo, out_w, metas)):
        pp_o = extract_program(ow.outputs[0].text)
        pv_o = execute_program(pp_o, m['table']) if pp_o else None
        eo = (check_ea(pv_o, m['gold_ans'], EA_DECIMAL_PLACES)
              if (m['gold_ans'] is not None and pv_o is not None) else False)
        po = check_pa(pp_o, m['gold_prog'])

        pp_w = extract_program(wt.outputs[0].text)
        pv_w = execute_program(pp_w, m['table']) if pp_w else None
        ew = (check_ea(pv_w, m['gold_ans'], EA_DECIMAL_PLACES)
              if (m['gold_ans'] is not None and pv_w is not None) else False)
        pw = check_pa(pp_w, m['gold_prog'])

        if eo: ea_wo += 1
        if po: pa_wo += 1
        if ew: ea_w += 1
        if pw: pa_w += 1
        details.append({'idx': i, 'ea_without': eo, 'pa_without': po,
                         'ea_with': ew, 'pa_with': pw})

    n = len(samples)
    regression = sum(1 for d in details
                      if (d['ea_without'] and d['pa_without'])
                          and (d['ea_with'] and not d['pa_with']))
    return {'n_samples': n,
            'delta_ea': (ea_w - ea_wo) / n, 'delta_pa': (pa_w - pa_wo) / n,
            'ea_without': ea_wo, 'ea_with': ea_w,
            'pa_without': pa_wo, 'pa_with': pa_w,
            'details': details, 'lucky_guess_regression': regression}


### Stage 3: dedup


In [ ]:
_TRIGGER_RE = re.compile(r'^\s*(?:for|when|if|in|on)\s+([^,.;:]+)', re.I)
_DSL_OPS_RE = re.compile(
    r'(add|subtract|multiply|divide|greater|exp|table_max|table_min|'
    r'table_sum|table_average|const_\w+)', re.I)

def _strip_meta(content):
    m = re.match(r'^\s*\{[^}]*\}\s*(.*)$', content)
    return m.group(1) if m else content

def _check_trigger_overlap(new_content, existing, thr=TRIGGER_OVERLAP_THRESH):
    new_clean = _strip_meta(new_content)
    new_ops = set(_DSL_OPS_RE.findall(new_clean.lower()))
    if not new_ops:
        return False, None
    m_new = _TRIGGER_RE.match(new_clean)
    new_trig = {w for w in (m_new.group(1).strip().lower() if m_new else "").split()
                 if w not in _DEDUP_STOPWORDS and len(w) > 2}
    for b in existing:
        b_clean = _strip_meta(b['content'])
        b_ops = set(_DSL_OPS_RE.findall(b_clean.lower()))
        if not b_ops or new_ops != b_ops:
            continue
        m_b = _TRIGGER_RE.match(b_clean)
        b_trig = {w for w in (m_b.group(1).strip().lower() if m_b else "").split()
                   if w not in _DEDUP_STOPWORDS and len(w) > 2}
        if not new_trig or not b_trig:
            return True, f'same_ops_vague_trigger:{b["id"]}'
        sim = len(new_trig & b_trig) / min(len(new_trig), len(b_trig))
        if sim >= thr:
            return True, f'trigger_overlap:{b["id"]}@{sim:.2f}'
    return False, None

def _check_action_signature_overlap(new_content, existing, sig_len=ACTION_SIG_DEDUP_LEN,
                                     vague_max_words=1):
    new_clean = _strip_meta(new_content)
    new_seq = re.findall(r'(add|subtract|multiply|divide|greater|exp|table_\w+)\(',
                          new_clean.lower())
    if len(new_seq) < 2:
        return False, None
    new_sig = '→'.join(new_seq[:sig_len])
    m_new = _TRIGGER_RE.match(new_clean)
    new_trig = {w for w in (m_new.group(1).strip().lower() if m_new else "").split()
                 if w not in _DEDUP_STOPWORDS and len(w) > 2}
    for b in existing:
        b_clean = _strip_meta(b['content'])
        b_seq = re.findall(r'(add|subtract|multiply|divide|greater|exp|table_\w+)\(',
                            b_clean.lower())
        if len(b_seq) < 2:
            continue
        b_sig = '→'.join(b_seq[:sig_len])
        if new_sig != b_sig:
            continue
        m_b = _TRIGGER_RE.match(b_clean)
        b_trig = {w for w in (m_b.group(1).strip().lower() if m_b else "").split()
                   if w not in _DEDUP_STOPWORDS and len(w) > 2}
        if not new_trig or not b_trig:
            return True, f'action_sig_dup:{b["id"]}@{new_sig}(no_trig)'
        if len(new_trig) <= vague_max_words or len(b_trig) <= vague_max_words:
            return True, f'action_sig_dup:{b["id"]}@{new_sig}(vague_trig)'
        union = new_trig | b_trig
        sim = len(new_trig & b_trig) / len(union) if union else 0.0
        if sim >= ACTION_SIG_TRIG_OVERLAP_THR:
            return True, f'action_sig_dup:{b["id"]}@{new_sig}(trig={sim:.2f})'
    return False, None


### Multi-stage curator with cluster quota + strict thresholds


In [ ]:
def multi_stage_curator(strategy, error_type, pb_str, next_id_map,
                         current_sample=None, train_subset=None,
                         cluster_id=None):
    info = {'stages': {}, 'final_action': None, 'cluster_id': cluster_id}
    cur_step = _current_train_step[0] if '_current_train_step' in globals() else 0

    if _is_quarantined(strategy, cur_step):
        info['final_action'] = 'reject_s0_quarantined'
        return pb_str, next_id_map, 'reject', 's0:in_quarantine', info

    # Stage 0.5: cluster quota
    if cluster_id and _cluster_quota_full(cluster_id):
        multistage_stats['stage0_5_cluster_quota_full'] += 1
        cap = MAX_BULLETS_C12 if cluster_id == 'C12_misc_other' else MAX_BULLETS_PER_CLUSTER
        info['stages']['s0_5_cluster_quota'] = {
            'cluster_id': cluster_id,
            'current_count': _bullets_in_cluster(cluster_id),
            'max': cap,
        }
        info['final_action'] = 'reject_s0_5_quota'
        return pb_str, next_id_map, 'reject', \
               f's0_5:cluster_full({cluster_id}@{cap})', info

    # Empty-critical-cluster bypass.
    # If a 4+-step cluster is still empty after step 200, bypass Stage 2 and
    # accept a candidate that passes the static quality gate.
    CRITICAL_EMPTY_CLUSTERS = {'C13_multistep_weighted',
                                'C14_multistep_subtract_chain',
                                'C15_multistep_aggregate',
                                'C16_multistep_5plus'}
    EMPTY_CLUSTER_BYPASS_AFTER_STEP = 200
    bypass_stage2 = False
    if (cluster_id in CRITICAL_EMPTY_CLUSTERS
            and cur_step >= EMPTY_CLUSTER_BYPASS_AFTER_STEP
            and _bullets_in_cluster(cluster_id) == 0):
        bypass_stage2 = True
        info['stages']['s0_75_empty_critical_bypass'] = {
            'cluster_id': cluster_id,
            'reason': f'critical_cluster_empty_at_step={cur_step}',
        }

    # Stage 1: QG
    action_qg, reason_qg = quality_gate(strategy, pb_str)
    info['stages']['s1_qg'] = {'action': action_qg, 'reason': reason_qg}
    if action_qg == 'reject':
        info['final_action'] = 'reject_s1'
        return pb_str, next_id_map, 'reject', f's1:{reason_qg}', info
    multistage_stats['stage1_pass'] += 1

    # Stage 2: empirical validation
    skip_s2 = False
    if bypass_stage2:
        skip_s2 = True
        multistage_stats.setdefault('stage2_skip_critical_bypass', 0)
        multistage_stats['stage2_skip_critical_bypass'] += 1
        info['stages']['s2_validation'] = {
            'skipped': f'critical_empty_bypass({cluster_id})'
        }
    elif not USE_MULTI_STAGE_CURATOR:
        skip_s2 = True
        info['stages']['s2_validation'] = {'skipped': 'flag_off'}
    elif current_sample is None or train_subset is None:
        skip_s2 = True
        info['stages']['s2_validation'] = {'skipped': 'no_context'}
    elif len(history) < VALIDATION_MIN_HISTORY:
        skip_s2 = True
        multistage_stats['stage2_skip_no_history'] += 1
        info['stages']['s2_validation'] = {'skipped': 'history_too_short'}

    if not skip_s2:
        val_samples = _select_validation_samples(error_type, current_sample,
                                                   train_subset, n=VALIDATION_N_SAMPLES)
        if len(val_samples) < VALIDATION_MIN_SAMPLES_REQUIRED:
            multistage_stats['stage2_skip_few_samples'] += 1
            info['stages']['s2_validation'] = {'skipped': 'few_samples',
                                                 'n_found': len(val_samples)}
        else:
            multistage_stats['stage2_eval'] += 1
            vr = _evaluate_bullet_on_samples(strategy, val_samples, pb_str)
            multistage_stats['delta_ea_log'].append(vr['delta_ea'])
            multistage_stats['delta_pa_log'].append(vr['delta_pa'])
            info['stages']['s2_validation'] = {
                'n_samples': vr['n_samples'],
                'delta_ea': round(vr['delta_ea'], 4),
                'delta_pa': round(vr['delta_pa'], 4),
                'ea_without': vr['ea_without'], 'ea_with': vr['ea_with'],
                'pa_without': vr['pa_without'], 'pa_with': vr['pa_with'],
                'lucky_regression': vr['lucky_guess_regression'],
            }
            harm_pa = sum(1 for d in vr['details']
                           if d['pa_without'] and not d['pa_with'])
            if harm_pa > VALIDATION_HARM_PA_TOL:
                multistage_stats['stage2_reject_harmful'] += 1
                info['stages']['s2_validation']['harm_pa'] = harm_pa
                info['final_action'] = 'reject_s2_harmful'
                _log_reject(strategy, error_type, 's2_harmful', vr)
                return pb_str, next_id_map, 'reject', f's2:harmful_pa={harm_pa}', info
            if vr['lucky_guess_regression'] > 0:
                multistage_stats['stage2_reject_lucky_regression'] += 1
                info['final_action'] = 'reject_s2_lucky_regression'
                _log_reject(strategy, error_type, 's2_lucky_regression', vr)
                return pb_str, next_id_map, 'reject', \
                       f's2:lucky_reg={vr["lucky_guess_regression"]}', info
            weighted = V95_W_EA * vr['delta_ea'] + V95_W_PA * vr['delta_pa']
            threshold = (RARE_ERRORS_THRESHOLD if error_type in RARE_ERRORS
                         else COMMON_ERRORS_THRESHOLD)
            info['stages']['s2_validation']['weighted_score'] = round(weighted, 4)
            info['stages']['s2_validation']['threshold'] = threshold
            if weighted < threshold:
                multistage_stats['stage2_reject_no_improvement'] += 1
                info['final_action'] = 'reject_s2_no_improvement'
                _log_reject(strategy, error_type, 's2_no_improvement', vr)
                return pb_str, next_id_map, 'reject', \
                       f's2:weighted={weighted:+.3f}_thr={threshold}', info
            multistage_stats['stage2_accept'] += 1

    # Stage 3: dedup
    existing = _all_bullets(pb_str)
    is_trig, trig_r = _check_trigger_overlap(strategy, existing)
    info['stages']['s3_trigger'] = {'overlap': is_trig, 'reason': trig_r}
    if is_trig:
        multistage_stats['stage3_reject_trigger_overlap'] += 1
        info['final_action'] = 'reject_s3_trigger'
        return pb_str, next_id_map, 'reject', f's3t:{trig_r}', info

    is_act, act_r = _check_action_signature_overlap(strategy, existing)
    info['stages']['s3_action_sig'] = {'overlap': is_act, 'reason': act_r}
    if is_act:
        multistage_stats['stage3_reject_action_signature'] += 1
        info['final_action'] = 'reject_s3_action'
        return pb_str, next_id_map, 'reject', f's3a:{act_r}', info
    multistage_stats['stage3_pass'] += 1

    # Inject
    multistage_stats['final_accept'] += 1
    intended = ERROR_TO_SECTION.get(error_type, 'common_errors')
    final_section = intended
    redirected = False
    if SECTION_QUOTA_PCT > 0 and _section_over_quota(pb_str, intended):
        final_section = _pick_fallback_section(pb_str, intended)
        redirected = True
    slug = SECTION_SLUGS.get(final_section, 'ce')
    next_n = next_id_map.get(slug, 0) + 1
    bid = f"{slug}-{next_n:05d}"
    next_id_map[slug] = next_n
    updated = _inject_bullet(pb_str, final_section, bid, strategy)

    if cluster_id:
        _assign_bullet_to_cluster(bid, cluster_id)

    info['final_action'] = 'add_validated'
    info['bullet_id'] = bid
    info['cluster_id_assigned'] = cluster_id
    return updated, next_id_map, 'add', \
           ('added_redirected_validated' if redirected else 'added_validated'), info

def _log_reject(strategy, error_type, reason_tag, vr=None):
    lst = multistage_stats['reject_examples']
    lst.append({'error_type': error_type, 'reason': reason_tag,
                 'strategy_60': strategy[:60],
                 'delta_ea': vr['delta_ea'] if vr else None,
                 'delta_pa': vr['delta_pa'] if vr else None})
    if len(lst) > 20:
        lst.pop(0)


### Dev-based automatic ablation


In [ ]:
def auto_ablate_playbook(pb_str, current_step):
    """Evict low-dev-lift bullets while protecting Tier 1 entries.

    Falls back to history-based estimates before the first dev-lift evaluation.
    """
    bullets = _all_bullets(pb_str)
    if len(bullets) < ABLATE_MIN_PLAYBOOK_SIZE:
        multistage_stats['auto_ablate_skipped_small'] += 1
        return pb_str, []

    # Try dev-based first
    if ABLATE_DATA_SOURCE == 'dev' and len(_dev_lift_ema_cache) > 0:
        return _auto_ablate_dev(pb_str, current_step, bullets)

    # Fallback: history-based
    return _auto_ablate_history(pb_str, current_step, bullets)

def _auto_ablate_dev(pb_str, current_step, bullets):
    """Dev-based ablation using EMA lift cache."""
    evicted = []
    lines = pb_str.splitlines()
    for b in bullets:
        bid = b['id']
        # Tier 1 protection
        if is_tier1(bid):
            multistage_stats['auto_ablate_skipped_tier1'] += 1
            continue
        cluster = _bullet_to_cluster.get(bid, 'C12_misc_other')
        if ABLATE_PROTECT_NON_C12 and cluster != 'C12_misc_other':
            multistage_stats.setdefault('auto_ablate_skipped_specific_cluster', 0)
            multistage_stats['auto_ablate_skipped_specific_cluster'] += 1
            continue
        rec = _dev_lift_ema_cache.get(bid)
        if rec is None or rec.get('n_evals', 0) < 1:
            continue
        # Need at least min_uses retrievals (proxy: n_evals × dev_set_size approximation)
        # Effective: just require ≥ 1 eval with stable EMA
        pa_lift = rec.get('pa_lift', 0)
        if pa_lift < ABLATE_PA_LIFT_THR:
            new_lines = []
            removed_content = None
            for line in lines:
                if f"[{bid}]" in line:
                    m = re.match(r'^\s*-\s*\[' + re.escape(bid) +
                                  r'\]\s*(.+?)(?:\s*\(h=.*)?$', line)
                    if m:
                        removed_content = m.group(1).strip()
                    continue
                new_lines.append(line)
            lines = new_lines
            if removed_content:
                _add_to_quarantine(removed_content, current_step)
            evicted.append({'id': bid, 'pa_lift': pa_lift,
                             'ea_lift': rec.get('ea_lift', 0),
                             'n_evals': rec.get('n_evals', 0),
                             'source': 'dev'})
            _unassign_bullet(bid)
            # Also remove from dev cache
            _dev_lift_ema_cache.pop(bid, None)
    return '\n'.join(lines), evicted

def _auto_ablate_history(pb_str, current_step, bullets):
    """Use history-based ablation when dev-lift data is unavailable."""
    if len(history) < ABLATE_WINDOW + 50:
        return pb_str, []
    recent = history[-ABLATE_WINDOW:]
    pa_baseline = sum(1 for h in recent if h.get('pa_pass')) / len(recent)
    records = defaultdict(lambda: {'used': 0, 'pa_pass': 0})
    for h in recent:
        for bid in (h.get('used_bullets') or []):
            records[bid]['used'] += 1
            if h.get('pa_pass'):
                records[bid]['pa_pass'] += 1
    evicted = []
    lines = pb_str.splitlines()
    for bid, rec in records.items():
        if is_tier1(bid):
            multistage_stats['auto_ablate_skipped_tier1'] += 1
            continue
        if rec['used'] < ABLATE_MIN_USES:
            continue
        lift = rec['pa_pass'] / rec['used'] - pa_baseline
        if lift < ABLATE_PA_LIFT_THR:
            new_lines = []
            removed_content = None
            for line in lines:
                if f"[{bid}]" in line:
                    m = re.match(r'^\s*-\s*\[' + re.escape(bid) +
                                  r'\]\s*(.+?)(?:\s*\(h=.*)?$', line)
                    if m:
                        removed_content = m.group(1).strip()
                    continue
                new_lines.append(line)
            lines = new_lines
            if removed_content:
                _add_to_quarantine(removed_content, current_step)
            evicted.append({'id': bid, 'lift': lift, 'used': rec['used'],
                             'source': 'history'})
            _unassign_bullet(bid)
    return '\n'.join(lines), evicted


### Post-training pruning


In [ ]:
def prune_playbook_post_training(pb_str, dev_samples, lift_threshold=None,
                                   verbose=True, n_samples=None,
                                   prefer_ema_cache=True):
    """Prune by EMA dev lift, with leave-one-out fallback.

    Trust EMA values after at least two evaluations and never prune Tier 1 bullets.
    """
    if lift_threshold is None:
        lift_threshold = globals().get('PRUNE_LIFT_THRESHOLD', -0.01)
    if n_samples is None:
        n_samples = globals().get('PRUNE_DEV_SUBSET', 200)

    # Use mini-set (first N samples from dev_valid)
    eval_set = dev_samples[:n_samples]

    bullets = _all_bullets(pb_str)
    if len(bullets) <= 5:
        if verbose:
            print(f"[PRUNE] Skip — only {len(bullets)} bullets")
        return pb_str, [], {'skipped': 'too_few_bullets'}

    if verbose:
        print("\n  ─── POST-TRAINING PRUNING (EMA-prioritized) ───")
        print(f"  Eval set: {len(eval_set)} samples (mini-dev for fallback eval)")
        print(f"  Threshold: PA-lift < {lift_threshold} → mark for removal")
        print(f"  Initial playbook: {len(bullets)} bullets "
              f"(Tier 1 protected: {len([b for b in bullets if is_tier1(b['id'])])})")
        print(f"  Mode: prefer_ema_cache={prefer_ema_cache}")

    # For non-cached bullets, we still need baseline for fresh eval
    # But we can avoid the costly per-bullet eval if EMA cache is sufficient
    cached_count = sum(1 for b in bullets
                       if not is_tier1(b['id'])
                       and _dev_lift_ema_cache.get(b['id'], {}).get('n_evals', 0) >= 2)
    uncached_count = sum(1 for b in bullets
                         if not is_tier1(b['id'])
                         and _dev_lift_ema_cache.get(b['id'], {}).get('n_evals', 0) < 2)
    if verbose:
        print(f"  Cached bullets (use EMA): {cached_count}, "
              f"Uncached (fresh eval): {uncached_count}")

    # Compute baseline only if we have any uncached bullets
    if uncached_count > 0 or not prefer_ema_cache:
        ea_full, pa_full = quick_dev_eval(eval_set, pb_str, n=len(eval_set))
        if verbose:
            print(f"  Full playbook baseline: EA={ea_full:.3f} PA={pa_full:.3f}")
    else:
        ea_full = pa_full = None
        if verbose:
            print(f"  Skipping baseline — all non-T1 bullets have EMA cache")

    bullet_lifts = []
    for i, b in enumerate(bullets, 1):
        # Skip Tier 1 bullets — they are never pruned
        if is_tier1(b['id']):
            if verbose:
                print(f"  [{i:>2}/{len(bullets)}] [{b['id']}] 🔒 Tier 1 — protected, skip")
            continue

        # prefer EMA cache (more reliable than single-shot re-eval)
        ema_rec = _dev_lift_ema_cache.get(b['id'], {})
        use_ema = (prefer_ema_cache and ema_rec.get('n_evals', 0) >= 2)

        if use_ema:
            lift_pa = ema_rec['pa_lift']
            lift_ea = ema_rec['ea_lift']
            bullet_lifts.append({
                'id': b['id'], 'content': b['content'][:80],
                'pa_lift': round(lift_pa, 4), 'ea_lift': round(lift_ea, 4),
                'source': f"EMA(n={ema_rec['n_evals']})",
            })
            if verbose:
                flag = ("⚠ harmful" if lift_pa < lift_threshold else
                        "✓ helpful" if lift_pa > 0.005 else "  neutral")
                print(f"  [{i:>2}/{len(bullets)}] [{b['id']}] "
                      f"lift_pa={lift_pa:+.3f} lift_ea={lift_ea:+.3f}  {flag} "
                      f"(EMA n={ema_rec['n_evals']})")
        else:
            # Fallback: fresh leave-one-out eval
            pb_without = '\n'.join(line for line in pb_str.splitlines()
                                    if f"[{b['id']}]" not in line)
            ea_w, pa_w = quick_dev_eval(eval_set, pb_without, n=len(eval_set))
            lift_pa = pa_full - pa_w
            lift_ea = ea_full - ea_w
            bullet_lifts.append({
                'id': b['id'], 'content': b['content'][:80],
                'pa_lift': round(lift_pa, 4), 'ea_lift': round(lift_ea, 4),
                'pa_with': round(pa_full, 3), 'pa_without': round(pa_w, 3),
                'ea_with': round(ea_full, 3), 'ea_without': round(ea_w, 3),
                'source': 'fresh_eval',
            })
            if verbose:
                flag = ("⚠ harmful" if lift_pa < lift_threshold else
                        "✓ helpful" if lift_pa > 0.005 else "  neutral")
                print(f"  [{i:>2}/{len(bullets)}] [{b['id']}] "
                      f"lift_pa={lift_pa:+.3f} lift_ea={lift_ea:+.3f}  {flag} "
                      f"(fresh eval)")
        multistage_stats['post_train_prune_evaluated'] += 1

    candidates = [bl for bl in bullet_lifts if bl['pa_lift'] < lift_threshold]
    candidates.sort(key=lambda x: x['pa_lift'])

    if not candidates:
        if verbose:
            print(f"  ✅ No bullets below threshold — nothing to prune")
        return pb_str, [], {
            'baseline_ea': ea_full, 'baseline_pa': pa_full,
            'all_lifts': bullet_lifts, 'removed': [],
        }

    cand_ids = {c['id'] for c in candidates}
    pruned_pb = '\n'.join(line for line in pb_str.splitlines()
                           if not any(f"[{cid}]" in line for cid in cand_ids))

    if verbose:
        print(f"\n  Marking {len(candidates)} bullets for removal:")
        for c in candidates:
            print(f"    [{c['id']}] pa_lift={c['pa_lift']:+.3f} — {c['content'][:60]}")

    # If baseline was skipped (all cached), compute it now for verification
    if ea_full is None or pa_full is None:
        ea_full, pa_full = quick_dev_eval(eval_set, pb_str, n=len(eval_set))
        if verbose:
            print(f"  (Computing baseline for verification step) "
                  f"EA={ea_full:.3f} PA={pa_full:.3f}")

    ea_after, pa_after = quick_dev_eval(eval_set, pruned_pb, n=len(eval_set))
    # composite-aware verification using EA priority
    ea_w = globals().get('COMPOSITE_EA_WEIGHT', 0.6)
    pa_w = globals().get('COMPOSITE_PA_WEIGHT', 0.4)
    score_before = ea_w * ea_full + pa_w * pa_full
    score_after  = ea_w * ea_after + pa_w * pa_after

    if verbose:
        print(f"\n  Verification:")
        print(f"    Before: EA={ea_full:.3f} PA={pa_full:.3f}  score={score_before:.3f}")
        print(f"    After : EA={ea_after:.3f} PA={pa_after:.3f}  score={score_after:.3f}")

    if score_after < score_before - 0.005:
        if verbose:
            print(f"  ⚠ REVERT: combined removal hurt score by {score_before - score_after:+.3f}")
        return pb_str, [], {
            'baseline_ea': ea_full, 'baseline_pa': pa_full,
            'reverted': True, 'all_lifts': bullet_lifts, 'removed': [],
            'verification': {'ea_after': ea_after, 'pa_after': pa_after,
                              'score_before': score_before, 'score_after': score_after},
        }

    for cid in cand_ids:
        _unassign_bullet(cid)

    multistage_stats['post_train_prune_removed'] = len(candidates)
    if verbose:
        print(f"  ✅ Adopt: removed {len(candidates)} bullets, "
              f"score change {score_after - score_before:+.3f}")
        print(f"  Final playbook: {len(_all_bullets(pruned_pb))} bullets")

    multistage_stats['post_train_prune_log'] = candidates
    return pruned_pb, candidates, {
        'baseline_ea': ea_full, 'baseline_pa': pa_full,
        'verification': {'ea_after': ea_after, 'pa_after': pa_after,
                          'score_before': score_before, 'score_after': score_after},
        'all_lifts': bullet_lifts, 'removed': candidates,
    }


### Self-tests


In [ ]:
print("\n[CURATOR] Running self-tests...")

# Stratified targets sum to 1.0 (5-bucket)
assert abs(sum(_STRAT_TARGETS.values()) - 1.0) < 1e-6
assert 5 in _STRAT_TARGETS, "5-bucket stratification must include n=5"
print(f"  ✓ Stratified 5-bucket targets: {_STRAT_TARGETS}")

# Cluster quota
_bullet_to_cluster.clear()
assert _bullets_in_cluster('C1_pct_share_div') == 0
assert not _cluster_quota_full('C1_pct_share_div')
_assign_bullet_to_cluster('test1', 'C1_pct_share_div')
_assign_bullet_to_cluster('test2', 'C1_pct_share_div')
assert _bullets_in_cluster('C1_pct_share_div') == 2
assert _cluster_quota_full('C1_pct_share_div')
_unassign_bullet('test1')
assert _bullets_in_cluster('C1_pct_share_div') == 1
_bullet_to_cluster.clear()
print(f"  ✓ Cluster quota MAX={MAX_BULLETS_PER_CLUSTER}")

# Trigger overlap dedup
existing = [{'id': 'ns-1', 'content': 'For percentage change, use subtract(new, old) then divide(#0, old).'}]
dup, _ = _check_trigger_overlap(
    'For percentage change in revenue, use subtract(new, old) then divide(#0, old).', existing)
assert dup
print(f"  ✓ Trigger overlap dup detection (thr={TRIGGER_OVERLAP_THRESH})")

# Action signature dup
existing_pg = [{'id': 'pg-1', 'content': 'For ratios, use subtract(new, old) followed by divide(#0, total).'}]
dup, _ = _check_action_signature_overlap(
    'For financial calculations, apply subtract(new, old) followed by divide(#0, total).', existing_pg)
assert dup
print(f"  ✓ Action signature dup (thr={ACTION_SIG_TRIG_OVERLAP_THR})")

# Quarantine cooldown extended to 200
_evicted_quarantine.clear()
_add_to_quarantine("test bullet content", 100)
assert _is_quarantined("test bullet content", 110)  # within cooldown
assert _is_quarantined("test bullet content", 250)  # still within (200 cooldown)
assert not _is_quarantined("test bullet content", 350)  # past cooldown
print(f"  ✓ Quarantine cooldown extended to {QUARANTINE_COOLDOWN}")
_evicted_quarantine.clear()

# Tier 1 protection in lift cache (Tier 1 bullets are skipped)
_dev_lift_ema_cache.clear()
# Simulate: bullet ns-99999 is Tier 1, should NOT enter lift eval prompts
# (We can't run real eval here without model, but we can test the skip logic)
print(f"  ✓ Tier 1 protection in lift eval (skip via is_tier1)")

# Stage 2 thresholds
assert COMMON_ERRORS_THRESHOLD >= -0.01, f"common threshold should be ≥ -0.01, got {COMMON_ERRORS_THRESHOLD}"
assert RARE_ERRORS_THRESHOLD >= -0.02, f"rare threshold should be ≥ -0.02, got {RARE_ERRORS_THRESHOLD}"
print(f"  ✓ Stage 2 thresholds: common={COMMON_ERRORS_THRESHOLD}, rare={RARE_ERRORS_THRESHOLD}, harm_tol={VALIDATION_HARM_PA_TOL}")

# Validation N samples
assert VALIDATION_N_SAMPLES >= 30, f"should validate on ≥30 samples, got {VALIDATION_N_SAMPLES}"
print(f"  ✓ Stage 2 N samples: {VALIDATION_N_SAMPLES}")

# Ablate config
assert ABLATE_DATA_SOURCE == 'dev', "ablation should default to dev source"
assert ABLATE_PA_LIFT_THR == -0.02, f"unexpected ablation threshold {ABLATE_PA_LIFT_THR}"
print(f"  ✓ Ablate: source={ABLATE_DATA_SOURCE}, thr={ABLATE_PA_LIFT_THR}")

# Lift EMA initialization
assert isinstance(_dev_lift_ema_cache, dict)
assert LIFT_EMA_ALPHA == 0.4
print(f"  ✓ Dev lift cache initialized; EMA α={LIFT_EMA_ALPHA}")

# Tier 1 promotion config
assert TIER_1_MAX == 5, "TIER_1_MAX should be 5"
assert TIER_1_PROMOTION_AGE >= 100
assert TIER_1_PROMOTION_LIFT > 0
print(f"  ✓ Tier 1 promote: age≥{TIER_1_PROMOTION_AGE}, lift>{TIER_1_PROMOTION_LIFT} "
      f"(fallback after step {TIER_1_FALLBACK_AGE})")

print()
print("[CURATOR] Self-tests passed")
print(f"  Stage 2 thresholds: (common={COMMON_ERRORS_THRESHOLD}, rare={RARE_ERRORS_THRESHOLD})")
print(f"  Empty-critical-cluster bypass: (C13/C14/C15/C16 after step 200)")
print("  Post-train pruning uses EMA cache when n_evals≥2")
print(f"  Stage 0.5 cluster quota   : max {MAX_BULLETS_PER_CLUSTER}/cluster")
print(f"  Stage 1 quality gate      : enabled")
print(f"  Stage 2 validation        : N={VALIDATION_N_SAMPLES} | thr_common={COMMON_ERRORS_THRESHOLD} | thr_rare={RARE_ERRORS_THRESHOLD} | harm_tol={VALIDATION_HARM_PA_TOL}")
print(f"  Stage 3 dedup             : trig_thr={TRIGGER_OVERLAP_THRESH}, action_thr={ACTION_SIG_TRIG_OVERLAP_THR}")
print(f"  Quarantine cooldown       : {QUARANTINE_COOLDOWN} steps")
print(f"  Auto-ablate source        : {ABLATE_DATA_SOURCE} (lift_thr={ABLATE_PA_LIFT_THR}, T1 protected)")
print(f"  Dev lift tracking         : every {LIFT_EVAL_EVERY} steps × {LIFT_DEV_SET_SIZE} samples (EMA α={LIFT_EMA_ALPHA})")
print(f"  Tier 1 promotion          : age≥{TIER_1_PROMOTION_AGE}, lift≥{TIER_1_PROMOTION_LIFT}, max={TIER_1_MAX}")
print(f"  Post-train prune          : dev mini-{globals().get('PRUNE_DEV_SUBSET', 200)}, thr={globals().get('PRUNE_LIFT_THRESHOLD', -0.01)}, EMA-prioritized")
print(f"  Stratified val 5-bucket   : {_STRAT_TARGETS}")


## Training pipeline


In [ ]:
LOCAL_SCRATCH_DIR = "/local-scratch"
if os.path.exists(LOCAL_SCRATCH_DIR) and os.access(LOCAL_SCRATCH_DIR, os.W_OK):
    HISTORY_PATH = f"{LOCAL_SCRATCH_DIR}/history.jsonl"
    print(f"[I/O] history → local-scratch: {HISTORY_PATH}")
else:
    HISTORY_PATH = f"{OUTPUT_DIR}/history.jsonl"
    print(f"[I/O] history → Drive: {HISTORY_PATH}")

print("\n[PIPELINE] Initializing...")


### Verify-iterate configuration


In [ ]:
USE_VERIFY_ITERATE        = globals().get('USE_VERIFY_ITERATE', True)
MAX_VERIFY_ROUNDS         = globals().get('MAX_VERIFY_ROUNDS', 3)
VERIFY_DEDUP_JACCARD      = globals().get('VERIFY_DEDUP_JACCARD', 0.90)
VERIFY_FORCE_INCLUDE      = True
VERIFY_REQUIRE_PA         = globals().get('VERIFY_REQUIRE_PA', True)
VERIFY_SKIP_LUCKY_GUESS   = globals().get('VERIFY_SKIP_LUCKY_GUESS', True)


### State


In [ ]:
playbook = INITIAL_PLAYBOOK
best_playbook = playbook
best_dev_ea = 0.0
best_dev_pa = 0.0
next_bullet_id = _next_id(playbook)
history = []
error_dist = {}; diag_dist = {}; outcome_dist = {}
qg_stats = {'total': 0, 'passed': 0, 'rejected_curator': 0, 'rejected_qg': 0,
             'evicted': 0, 'reasons': {}}
api_cost = {'calls': 0, 'tokens': 0}
bullet_birth_step = {}
counterfactual_stats = {'triggered': 0, 'cases_found': 0, 'cases_avg': []}

# verify-iterate stats with proper round 1/2/3 tracking
verify_stats = {
    'total_attempts':       0,
    'round_1_pass':         0,
    'round_2_pass':         0,
    'round_3_pass':         0,
    'exhausted_all_rounds': 0,  # renamed from exhausted_3_rounds
    'paraphrase_break':     0,
    'reflector_unfixable':  0,
    'verify_pass_count':    0,
    'verify_fail_count':    0,
    'verify_pa_fail_count': 0,  # PA failed but EA passed (lucky-guess avoided)
    'fewshot_used':         0,
    'fewshot_bootstrap':    0,  # used train_sub.gold when no correct history
    'cluster_assigned':     {},
    # Thinking trace
    'thinking_captured':    0,
    'thinking_empty':       0,
    'thinking_skipped':     0,
    # Reflector skip
    'reflector_skipped_lucky': 0,  # outcome=lucky_guess, full Reflector skip
    'reflector_skipped_correct': 0,  # outcome=correct
}

# Resume
if START_STEP > 0:
    _pb_path = f"{PLAYBOOK_DIR}/step_{START_STEP:05d}.txt"
    if not os.path.exists(_pb_path):
        raise FileNotFoundError(f'[RESUME] Missing playbook checkpoint: {_pb_path}')
    with open(_pb_path) as f:
        playbook = f.read()
    next_bullet_id = _next_id(playbook)
    if os.path.exists(f"{OUTPUT_DIR}/best_playbook.txt"):
        with open(f"{OUTPUT_DIR}/best_playbook.txt") as f: best_playbook = f.read()
    _drive_hist = f"{OUTPUT_DIR}/history.jsonl"
    if HISTORY_PATH != _drive_hist and os.path.exists(_drive_hist) and not os.path.exists(HISTORY_PATH):
        shutil.copy(_drive_hist, HISTORY_PATH)
    if os.path.exists(HISTORY_PATH):
        with open(HISTORY_PATH) as f:
            history = [json.loads(l) for l in f if l.strip()]
    if os.path.exists(PROGRESS_PATH):
        with open(PROGRESS_PATH) as f:
            _p = json.load(f)
            error_dist = _p.get('error_dist', {})
            diag_dist = _p.get('diag_dist', {})
            outcome_dist = _p.get('outcome_dist', {})
            qg_stats = _p.get('qg_stats', qg_stats)
            best_dev_ea = _p.get('best_dev_ea', 0.0)
            best_dev_pa = _p.get('best_dev_pa', 0.0)
            api_cost = _p.get('api_cost', api_cost)
            bullet_birth_step = _p.get('bullet_birth_step', {})
            next_bullet_id = _p.get('next_id', next_bullet_id)
            counterfactual_stats.update(_p.get('counterfactual_stats', {}))
            verify_stats.update(_p.get('verify_stats', {}))
            _bullet_to_cluster.clear()
            _bullet_to_cluster.update(_p.get('bullet_to_cluster', {}))
            _tier1_bullets.clear()
            _tier1_bullets.update(_p.get('tier1_bullets', []))
            _dev_lift_ema_cache.clear()
            _dev_lift_ema_cache.update(_p.get('dev_lift_ema_cache', {}))
    print(f"[RESUME] ✅ step={START_STEP}, bullets={_pb_stats(playbook)['total']}")

def save_progress(step, epoch=0):
    with open(f"{PLAYBOOK_DIR}/step_{step:05d}.txt", 'w') as f: f.write(playbook)
    with open(f"{OUTPUT_DIR}/best_playbook.txt", 'w') as f: f.write(best_playbook)
    with open(HISTORY_PATH, 'w') as f:
        for h in history: f.write(json.dumps(h, default=str) + '\n')
    if HISTORY_PATH != f"{OUTPUT_DIR}/history.jsonl":
        try: shutil.copy(HISTORY_PATH, f"{OUTPUT_DIR}/history.jsonl")
        except Exception as e: print(f"[I/O] ⚠ {e}")
    # Snapshot Tier 1 set for resume
    tier1_snap = sorted(list(globals().get('_tier1_bullets', set())))
    with open(PROGRESS_PATH, 'w') as f:
        json.dump({'completed_step': step, 'epoch': epoch, 'next_id': next_bullet_id,
                    'error_dist': error_dist, 'diag_dist': diag_dist,
                    'outcome_dist': outcome_dist, 'qg_stats': qg_stats,
                    'best_dev_ea': best_dev_ea, 'best_dev_pa': best_dev_pa,
                    'bullet_birth_step': bullet_birth_step,
                    'counterfactual_stats': counterfactual_stats,
                    'verify_stats': verify_stats,
                    'bullet_to_cluster': dict(_bullet_to_cluster),
                    'tier1_bullets': tier1_snap,
                    'dev_lift_ema_cache': dict(_dev_lift_ema_cache),
                    'api_cost': api_cost, 'model': MODEL_TAG,
                    'timestamp': datetime.now().isoformat()}, f, indent=2)


### Outcome classification


In [ ]:
def _program_similarity(p1, p2):
    o1 = set(re.findall(r'(add|subtract|multiply|divide|greater|exp|table_\w+)', (p1 or '').lower()))
    o2 = set(re.findall(r'(add|subtract|multiply|divide|greater|exp|table_\w+)', (p2 or '').lower()))
    return (len(o1 & o2) / len(o1 | o2)) if (o1 and o2) else 0.0

def classify_outcome(ea, pa, pp, pv, gp, ga):
    if ea and pa: return 'correct'
    if ea and not pa: return 'lucky_guess'
    if not ea and pa: return 'exec_mismatch'
    if USE_CLOSE_BUT_WRONG and pp and gp and _program_similarity(pp, gp) >= 0.7:
        return 'close_but_wrong'
    return 'wrong_reasoning'


### BATCHED quick_dev_eval


In [ ]:
def quick_dev_eval(dev_samples, pb, n=None):
    if n is None:
        n = globals().get('EVAL_DEV_SIZE', len(dev_samples))
    subset = dev_samples[:n]
    if not subset: return 0.0, 0.0
    prompts, metas = [], []
    for s in subset:
        bul = retrieve_top_k(s['qa']['question'], pb)
        prompts.append(build_ace_prompt(s, playbook_bullets=bul))
        metas.append({'gold_ans': s['qa'].get('exe_ans'),
                       'gold_prog': s['qa'].get('program', ''),
                       'table': s.get('table', [])})
    outputs = model.fast_generate(prompts, sampling_params=SAMPLING_PARAMS)
    ea_ok = pa_ok = 0
    for out, meta in zip(outputs, metas):
        pp = extract_program(out.outputs[0].text)
        pv = execute_program(pp, meta['table']) if pp else None
        if (meta['gold_ans'] is not None and pv is not None
                and check_ea(pv, meta['gold_ans'], EA_DECIMAL_PLACES)):
            ea_ok += 1
        if check_pa(pp, meta['gold_prog']): pa_ok += 1
    return ea_ok / len(subset), pa_ok / len(subset)


### Bullet budget enforcement with Tier 1 PROTECTION


In [ ]:
def _enforce_bullet_budget(pb_str, current_step):
    """Enforce the bullet budget without evicting Tier 1 entries."""
    if not ENFORCE_BULLET_BUDGET: return pb_str, 0
    bullets = _all_bullets(pb_str)
    if len(bullets) <= MAX_PLAYBOOK_BULLETS: return pb_str, 0

    # Filter: only consider non-Tier-1 bullets for eviction
    candidates = []
    for b in bullets:
        if is_tier1(b['id']):
            continue  # Tier 1 protection
        score = b['helpful'] - 2 * b['harmful']
        age = current_step - bullet_birth_step.get(b['id'], 0)
        if age >= BULLET_MIN_AGE_STEPS and score <= BULLET_EVICT_SCORE:
            candidates.append((b['id'], score))
    if not candidates: return pb_str, 0
    candidates.sort(key=lambda x: x[1])
    n_excess = len(bullets) - MAX_PLAYBOOK_BULLETS
    to_evict = set(c[0] for c in candidates[:n_excess])
    out = []
    for line in pb_str.splitlines():
        bid, _, _, _ = _parse_bullet_line(line)
        if bid and bid in to_evict: continue
        out.append(line)
    for bid in to_evict:
        bullet_birth_step.pop(bid, None)
        _unassign_bullet(bid)  # also free cluster slot (and Tier 1 if somehow there)
    return '\n'.join(out), len(to_evict)


### Verify helper with PA-aware passing + last-round relaxation


In [ ]:
def _verify_bullet_on_sample(candidate_bullet, sample, current_pb_str,
                              is_last_round=False):
    """Re-run a sample with the candidate bullet force-included.

    When ``VERIFY_REQUIRE_PA`` is enabled, both EA and PA are mandatory.
    When disabled, an EA match may pass without an exact program match.

    Returns a mapping with pass, EA, PA, and prediction details.
    """
    q = sample['qa']['question']
    gold_prog = sample['qa']['program']
    gold_ans = sample['qa'].get('exe_ans')
    table = sample.get('table', [])

    # Force-include candidate at TOP, drop one slot from base retrieval
    top_k_minus = max(1, globals().get('TOP_K_RETRIEVAL', 7) - 1)
    base_bul = retrieve_top_k(q, current_pb_str, k=top_k_minus)
    forced_bul = f"- {candidate_bullet}\n{base_bul}".strip()

    # Optional fewshot for hard queries
    n_query = _n_ops_program(gold_prog)
    fewshot = None
    if n_query >= 3:
        fewshot = _select_fewshot_for_query_safe(sample, n_examples=2,
                                                   exclude_question=q)

    prompt = build_ace_prompt(sample, playbook_bullets=forced_bul,
                                fewshot_examples=fewshot)
    sp = SamplingParams(temperature=0.0, max_tokens=MAX_TOKENS,
                         repetition_penalty=REPETITION_PENALTY,
                         skip_special_tokens=True, seed=RANDOM_SEED)
    out = model.fast_generate([prompt], sampling_params=sp)[0]
    raw = out.outputs[0].text

    pp = extract_program(raw)
    pv = execute_program(pp, table) if pp else None
    ea = check_ea(pv, gold_ans, EA_DECIMAL_PLACES) if (gold_ans is not None and pv is not None) else False
    pa = check_pa(pp, gold_prog)

    if VERIFY_REQUIRE_PA:
        passed = ea and pa
    else:
        passed = ea

    return {'pass': passed, 'ea': ea, 'pa': pa,
            'pred_prog': pp, 'pred_ans': pv}

def _jaccard_text(s1, s2):
    """Word-level Jaccard for paraphrase detection."""
    w1 = set(re.findall(r'\b[a-z_]+\b', s1.lower()))
    w2 = set(re.findall(r'\b[a-z_]+\b', s2.lower()))
    if not w1 or not w2: return 0.0
    return len(w1 & w2) / len(w1 | w2)


### Fewshot bootstrap


In [ ]:
# When history.correct is empty (early training), bootstrap fewshot from train_sub gold
# programs. This addresses the issue that 3+step queries get NO fewshot in steps 1-50.
def _select_fewshot_for_query_safe(sample, n_examples=2, exclude_question=None):
    """Wrapper around _select_fewshot_for_query with bootstrap fallback.

    Tries history-based selection first.
    If empty, falls back to train_sub gold programs matched by complexity.
    """
    try:
        result = _select_fewshot_for_query(sample, n_examples=n_examples,
                                             exclude_question=exclude_question)
        if result and len(result) >= 1:
            return result
    except Exception:
        pass

    # Bootstrap from train_sub
    train_sub = globals().get('train_sub', None)
    if not train_sub:
        return None

    # Match complexity (n_ops same as query, exclude same question)
    try:
        n_query = _n_ops_program(sample['qa'].get('program', ''))
    except Exception:
        return None
    if exclude_question is None:
        exclude_question = sample['qa']['question']

    # Pick first N samples from train_sub matching n_ops, excluding self
    candidates = []
    for s in train_sub:
        if s['qa']['question'] == exclude_question:
            continue
        n_s = _n_ops_program(s['qa'].get('program', ''))
        if n_s == n_query:
            candidates.append(s)
        if len(candidates) >= n_examples:
            break
    if candidates:
        verify_stats['fewshot_bootstrap'] += 1
        return candidates[:n_examples]
    return None


### PROCESS_ONE — with PA-aware verify + lucky_guess full skip


In [ ]:
def process_one(sample, step, total):
    global playbook, next_bullet_id, best_playbook, best_dev_ea, best_dev_pa, api_cost

    q = sample['qa']['question']
    ctx = build_oracle_context(sample)
    gold_prog = sample['qa']['program']
    gold_ans = sample['qa'].get('exe_ans')
    table = sample.get('table', [])

    # Cluster classification
    cluster_id = cluster_id_for_sample(sample)
    cluster_def = _CLUSTER_BY_ID.get(cluster_id)

    # 1. Retrieval (Tier 1 + Tier 2 if USE_TIER_SYSTEM=True)
    bullets_str, used_ids = _retrieve_with_ids(q, playbook)

    # Optional fewshot for hard queries
    n_gold = _n_ops_program(gold_prog)
    fewshot_examples = None
    if n_gold >= 3:
        fewshot_n = _fewshot_count_for_query(sample) if '_fewshot_count_for_query' in globals() else 2
        fewshot_examples = _select_fewshot_for_query_safe(sample, n_examples=fewshot_n,
                                                            exclude_question=q)
        if fewshot_examples:
            verify_stats['fewshot_used'] += 1

    # 2. Generator
    out = model.fast_generate(
        [build_ace_prompt(sample, playbook_bullets=bullets_str,
                            fewshot_examples=fewshot_examples)],
        sampling_params=SAMPLING_PARAMS)
    raw_text = out[0].outputs[0].text

    # 3. Executor + metrics
    pred_prog = extract_program(raw_text)
    pred_ans = execute_program(pred_prog, table) if pred_prog else None
    ea = check_ea(pred_ans, gold_ans, EA_DECIMAL_PLACES) if (gold_ans is not None and pred_ans is not None) else False
    pa = check_pa(pred_prog, gold_prog)

    # 4. Outcome
    outcome = classify_outcome(ea, pa, pred_prog, pred_ans, gold_prog, gold_ans)
    outcome_dist[outcome] = outcome_dist.get(outcome, 0) + 1

    diag = ''
    if not ea:
        diag = _diagnose_failure(raw_text, pred_prog, pred_ans, gold_prog, gold_ans)
        diag_dist[diag] = diag_dist.get(diag, 0) + 1

    result = {'step': step, 'q': q[:80], 'gold_ans': gold_ans, 'pred_ans': pred_ans,
               'gold_prog': gold_prog, 'pred_prog': pred_prog,
               'ea_pass': ea, 'pa_pass': pa, 'outcome': outcome,
               'diag': diag, 'used_bullets': used_ids,
               'pb_size': len(_all_bullets(playbook)),
               'cluster_id': cluster_id,
               'fewshot_n': len(fewshot_examples) if fewshot_examples else 0}

    # Outcome-based Reflector control
    # Skip Reflector ENTIRELY for:
    # - correct outcome (no error to reflect on)
    # - lucky_guess outcome (avoid encoding lucky patterns as bullets)
    skip_reflector = (
        outcome == 'correct' or
        (outcome == 'lucky_guess' and not USE_LUCKY_GUESS_REFLECT)
    )
    skip_curator = skip_reflector or (
        outcome == 'lucky_guess' and not USE_LUCKY_GUESS_ADD_BULLET
    )

    if outcome == 'correct':
        verify_stats['reflector_skipped_correct'] += 1
    if outcome == 'lucky_guess' and not USE_LUCKY_GUESS_REFLECT:
        verify_stats['reflector_skipped_lucky'] += 1

    if skip_reflector:
        ref = {'error_type': 'none', 'root_cause': 'skipped',
                'new_strategy': '', 'key_insight': '', 'bullet_tags': []}
        result['error_type'] = outcome
        error_dist[outcome] = error_dist.get(outcome, 0) + 1
    else:
        # SLM thinking trace
        thinking_trace = ""
        if globals().get('USE_THINKING_TRACE', False):
            thinking_trace = capture_slm_thinking(sample) if 'capture_slm_thinking' in globals() else ""
            if thinking_trace:
                verify_stats['thinking_captured'] += 1
            else:
                verify_stats['thinking_empty'] += 1
        else:
            verify_stats['thinking_skipped'] += 1

        # Counterfactual retrieval
        success_cases = None
        if (USE_COUNTERFACTUAL and outcome in COUNTERFACTUAL_OUTCOMES
                and len(history) >= COUNTERFACTUAL_MIN_HISTORY):
            counterfactual_stats['triggered'] += 1
            success_cases = _retrieve_success_cases(
                q, history, k=COUNTERFACTUAL_N,
                current_n_steps=n_gold,
                current_prog=gold_prog,
                exact_complexity=n_gold)
            if success_cases and len(success_cases) >= COUNTERFACTUAL_MIN_CASES:
                counterfactual_stats['cases_found'] += 1
                counterfactual_stats['cases_avg'].append(len(success_cases))
            else:
                success_cases = None

        # Verify-iterate loop
        skip_verify = (not USE_VERIFY_ITERATE
                       or skip_curator
                       or step % CURATOR_EVERY != 0)

        ref = None
        bullet_history_in_round = []
        verify_round_used = 0
        verify_outcome = None

        if skip_verify:
            # Single Reflector call (no verify-iterate)
            ref_fn = _reflector_hybrid if USE_HYBRID_REFLECTOR else _reflector_slm
            raw_ref, usage = ref_fn(q, ctx, pred_prog or '', str(pred_ans),
                                      gold_prog, str(gold_ans), bullets_str, diag, outcome,
                                      success_cases=success_cases, pb_str=playbook,
                                      cluster_def=cluster_def,
                                      thinking_trace=thinking_trace)
            if usage and USE_HYBRID_REFLECTOR:
                api_cost['calls'] += 1
                api_cost['tokens'] += usage.total_tokens
            ref = _parse_reflector_json(raw_ref, fallback_outcome=outcome)
            verify_round_used = 0
        else:
            # Run the configured number of verification rounds.
            verify_stats['total_attempts'] += 1
            verify_feedback = None

            for round_idx in range(MAX_VERIFY_ROUNDS):
                ref_fn = _reflector_hybrid if USE_HYBRID_REFLECTOR else _reflector_slm
                raw_ref, usage = ref_fn(q, ctx, pred_prog or '', str(pred_ans),
                                          gold_prog, str(gold_ans),
                                          bullets_str, diag, outcome,
                                          success_cases=success_cases, pb_str=playbook,
                                          cluster_def=cluster_def,
                                          verify_feedback=verify_feedback,
                                          thinking_trace=thinking_trace)
                if usage and USE_HYBRID_REFLECTOR:
                    api_cost['calls'] += 1
                    api_cost['tokens'] += usage.total_tokens

                round_ref = _parse_reflector_json(raw_ref, fallback_outcome=outcome)

                # Reflector confessed unfixable → break
                if round_ref.get('error_type') == 'unfixable':
                    verify_stats['reflector_unfixable'] += 1
                    ref = round_ref
                    verify_outcome = 'unfixable'
                    break

                strat = round_ref.get('new_strategy', '').strip()
                if not strat:
                    ref = round_ref
                    verify_outcome = 'empty_strategy'
                    break

                # Paraphrase check (round ≥ 2 only) — skip on LAST round (give it a chance)
                is_last_round = (round_idx == MAX_VERIFY_ROUNDS - 1)
                if round_idx >= 1 and bullet_history_in_round and not is_last_round:
                    max_jacc = max(_jaccard_text(strat, prev)
                                    for prev in bullet_history_in_round)
                    if max_jacc >= VERIFY_DEDUP_JACCARD:
                        verify_stats['paraphrase_break'] += 1
                        ref = round_ref
                        verify_outcome = 'paraphrase'
                        break

                bullet_history_in_round.append(strat)

                # Quick QG check before SLM verify (saves SLM call)
                qg_action, qg_reason = quality_gate(strat, playbook)
                if qg_action == 'reject':
                    verify_feedback = {
                        'round': round_idx + 1,
                        'reason': f'QG rejected: {qg_reason}',
                        'previous_bullet': strat,
                    }
                    if is_last_round:
                        ref = round_ref
                        verify_outcome = 'qg_exhausted'
                    continue

                # SLM verify on original sample
                vres = _verify_bullet_on_sample(strat, sample, playbook,
                                                  is_last_round=is_last_round)

                if vres['pass']:
                    verify_stats['verify_pass_count'] += 1
                    if round_idx == 0:
                        verify_stats['round_1_pass'] += 1
                    elif round_idx == 1:
                        verify_stats['round_2_pass'] += 1
                    else:
                        verify_stats['round_3_pass'] += 1
                    verify_round_used = round_idx + 1
                    verify_outcome = 'verified'
                    ref = round_ref
                    break
                else:
                    verify_stats['verify_fail_count'] += 1
                    # Track if EA passed but PA failed (lucky-guess avoided)
                    if vres.get('ea') and not vres.get('pa'):
                        verify_stats['verify_pa_fail_count'] += 1
                    verify_feedback = {
                        'round': round_idx + 1,
                        'reason': (f'SLM verify failed: pred="{vres["pred_prog"]}" → '
                                    f'{vres["pred_ans"]} vs gold={gold_ans} '
                                    f'(EA={vres["ea"]}, PA={vres["pa"]})'),
                        'previous_bullet': strat,
                    }
                    if is_last_round:
                        verify_stats['exhausted_all_rounds'] += 1
                        ref = round_ref
                        verify_outcome = 'exhausted'

            if ref is None:
                ref = {'error_type': outcome, 'root_cause': 'verify_loop_no_result',
                        'new_strategy': '', 'bullet_tags': []}

            result['verify'] = {
                'rounds_used': verify_round_used,
                'outcome': verify_outcome,
                'n_attempts': len(bullet_history_in_round),
            }

        result['error_type'] = ref.get('error_type', outcome)
        error_dist[result['error_type']] = error_dist.get(result['error_type'], 0) + 1
        if success_cases:
            result['counterfactual_used'] = True
            result['counterfactual_n'] = len(success_cases)
        if ref.get('bullet_tags'):
            playbook = _update_counts(playbook, ref['bullet_tags'])

    # 6. Curator (with cluster_id)
    if step % CURATOR_EVERY == 0 and not skip_curator:
        strat = ref.get('new_strategy', '').strip()
        etype = ref.get('error_type', outcome)
        verify_pass = (skip_verify  # Single-shot fallback
                       or (result.get('verify', {}).get('outcome') == 'verified'))

        if strat and etype not in ('none', '', 'unfixable') and verify_pass:
            qg_stats['total'] += 1
            if USE_MULTI_STAGE_CURATOR:
                _train_sub = globals().get('train_sub', None)
                playbook, next_bullet_id, action, reason, val_info = \
                    multi_stage_curator(strat, etype, playbook, next_bullet_id,
                                          current_sample=sample,
                                          train_subset=_train_sub,
                                          cluster_id=cluster_id)
                result['validation'] = {
                    's0_5':  val_info['stages'].get('s0_5_cluster_quota', {}),
                    's1':    val_info['stages'].get('s1_qg', {}),
                    's2':    val_info['stages'].get('s2_validation', {}),
                    's3':    val_info['stages'].get('s3_trigger', {}),
                    'final_action': val_info.get('final_action'),
                    'cluster_id_assigned': val_info.get('cluster_id_assigned'),
                }
            else:
                playbook, next_bullet_id, action, reason = run_curator(
                    strat, etype, playbook, next_bullet_id)

            if action == 'add':
                qg_stats['passed'] += 1
                result['added_bullet'] = True
                cs = verify_stats['cluster_assigned']
                cs[cluster_id] = cs.get(cluster_id, 0) + 1
                for b in _all_bullets(playbook):
                    if b['id'] not in bullet_birth_step:
                        bullet_birth_step[b['id']] = step
            elif action == 'reject':
                qg_stats['rejected_qg'] += 1
                qg_stats['reasons'][reason] = qg_stats['reasons'].get(reason, 0) + 1

    # 7. Bullet budget enforcement
    if ENFORCE_BULLET_BUDGET and result.get('added_bullet'):
        playbook, n_evicted = _enforce_bullet_budget(playbook, step)
        if n_evicted > 0:
            qg_stats['evicted'] = qg_stats.get('evicted', 0) + n_evicted
            result['evicted'] = n_evicted

    history.append(result)
    with open(HISTORY_PATH, 'a') as f:
        f.write(json.dumps(result, default=str) + '\n')
    return result

def _parse_reflector_json(raw_ref, fallback_outcome='wrong_reasoning'):
    """Robust JSON parser for Reflector output."""
    if not raw_ref:
        return {'error_type': fallback_outcome, 'root_cause': 'empty_response',
                'new_strategy': '', 'bullet_tags': [], 'key_insight': ''}
    ref = None
    try: ref = json.loads(raw_ref.strip())
    except: pass
    if ref is None:
        m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw_ref, re.S)
        if m:
            try: ref = json.loads(m.group(1))
            except: pass
    if ref is None:
        m = re.search(r'\{.*\}', raw_ref, re.S)
        if m:
            try: ref = json.loads(m.group(0))
            except: pass
    if ref is None:
        ref = {'error_type': fallback_outcome, 'root_cause': 'parse_failed',
                'new_strategy': '', 'bullet_tags': [], 'key_insight': ''}
    return ref

print("[PIPELINE] Verification and reflection helpers loaded")
print(f"   USE_VERIFY_ITERATE      : {USE_VERIFY_ITERATE} (max {MAX_VERIFY_ROUNDS} rounds)")
print(f"   VERIFY_REQUIRE_PA       : {VERIFY_REQUIRE_PA}")


### PROCESS_ONE — with PA-aware verify + lucky_guess full skip — continued


In [ ]:
print(f"   VERIFY_DEDUP_JACCARD    : {VERIFY_DEDUP_JACCARD} "      "(last round skips the check)")
print(f"   USE_LUCKY_GUESS_REFLECT : {USE_LUCKY_GUESS_REFLECT} (skip Reflector entirely if False)")
print(f"   FEWSHOT bootstrap       : enabled (train_sub fallback if no correct history)")
print(f"   Tier 1 budget protected : enabled")
print(f"   Cluster-aware Reflector : enabled")
print(f"   HISTORY_PATH            : {HISTORY_PATH}")

# Explicit run-state policy
_progress = f"{OUTPUT_DIR}/progress.json"
_drive_history = f"{OUTPUT_DIR}/history.jsonl"
_state_paths = {_progress, HISTORY_PATH, _drive_history}
if RESET_RUN_STATE:
    for _state_path in sorted(_state_paths):
        if os.path.exists(_state_path):
            os.remove(_state_path)
            print(f"[RESET] Removed {_state_path}")
elif START_STEP == 0 and any(os.path.exists(path) for path in _state_paths):
    raise RuntimeError(
        "Existing run state detected. Set ACE_FINQA_START_STEP to resume, "
        "ACE_FINQA_RESET_RUN_STATE=1 to reset, or choose a new output directory."
    )
elif START_STEP > 0:
    print(f"[RESUME] Preserving restored state from step {START_STEP}")
else:
    print("[STATE] Starting a new run in an empty output directory")


## Training loop and artifacts


In [ ]:
import time, random, json, shutil
from datetime import datetime
from collections import defaultdict


### Resolve prompt and playbook mode


In [ ]:
RUN_NAME           = globals().get('RUN_NAME', 'FULL_thesis')
USE_BARE_PLAYBOOK  = globals().get('USE_BARE_PLAYBOOK', True)
USE_THINKING_TRACE = globals().get('USE_THINKING_TRACE', False)
MODE               = globals().get('MODE', 'FULL_thesis')

print(f"[TRAIN] MODE={MODE} | RUN_NAME={RUN_NAME}")
print(f"[TRAIN] USE_BARE_PLAYBOOK={USE_BARE_PLAYBOOK} | "
      f"USE_THINKING_TRACE={USE_THINKING_TRACE}")


### Training configuration


In [ ]:
NUM_EPOCHS            = globals().get('NUM_EPOCHS', 1)
TRAIN_SUBSET          = globals().get('TRAIN_SUBSET', 600)
EVAL_EVERY            = globals().get('EVAL_EVERY', 60)
AUTO_ABLATE_EVERY     = 100
EARLY_STOP_PATIENCE   = globals().get('EARLY_STOP_PATIENCE', 5)
LIFT_EVAL_EVERY       = globals().get('LIFT_EVAL_EVERY', 60)

# EA-priority composite scoring
COMPOSITE_SELECTION   = True
COMPOSITE_EA_WEIGHT   = globals().get('COMPOSITE_EA_WEIGHT', 0.60)
COMPOSITE_PA_WEIGHT   = globals().get('COMPOSITE_PA_WEIGHT', 0.40)
USE_PA_GUARD          = globals().get('USE_PA_GUARD', True)
PA_GUARD_TOL          = globals().get('PA_GUARD_TOLERANCE', 0.02)
EA_STRICT_PA_FLOOR_DELTA = globals().get('EA_STRICT_PA_FLOOR_DELTA', 0.02)

# playbook capacity
MAX_PLAYBOOK_BULLETS  = globals().get('MAX_PLAYBOOK_BULLETS', 30)
ENFORCE_BULLET_BUDGET = globals().get('ENFORCE_BULLET_BUDGET', True)
BULLET_MIN_AGE_STEPS  = globals().get('BULLET_MIN_AGE_STEPS', 80)
BULLET_EVICT_SCORE    = globals().get('BULLET_EVICT_SCORE', 0)

USE_HYBRID_REFLECTOR    = globals().get('USE_HYBRID_REFLECTOR', True)
HYBRID_MODEL_REFLECTOR  = globals().get('HYBRID_MODEL_REFLECTOR', 'gpt-4o-mini')

USE_VERIFY_ITERATE        = globals().get('USE_VERIFY_ITERATE', True)
MAX_VERIFY_ROUNDS         = globals().get('MAX_VERIFY_ROUNDS', 3)

# post-train prune (dev mini, skip-EA)
USE_POST_TRAINING_PRUNE = globals().get('USE_POST_TRAINING_PRUNE', True)
PRUNE_LIFT_THRESHOLD    = globals().get('PRUNE_LIFT_THRESHOLD', -0.01)
PRUNE_DEV_SUBSET        = globals().get('PRUNE_DEV_SUBSET', 200)
SKIP_PRUNE_EA           = globals().get('SKIP_PRUNE_EA', True)

# stratified ratios (5-bucket)
STRATIFIED_RATIOS = globals().get('STRATIFIED_RATIOS', (140, 170, 170, 100, 20))


### Stratified training sample


In [ ]:
def _build_stratified_train(train_valid, total=600,
                              ratios=None, seed=RANDOM_SEED):
    """Sample five operation-count buckets; default to (140, 170, 170, 100, 20).

    Pattern interleaves bucket complexities to avoid all-1-step then all-4-step blocks.
    """
    if ratios is None:
        ratios = STRATIFIED_RATIOS
    rng = random.Random(seed)
    by_n = {1: [], 2: [], 3: [], 4: [], 5: []}
    for s in train_valid:
        n = len(_OP_RE_DIAG.findall(s.get('qa', {}).get('program', '')))
        bucket = min(max(n, 1), 5)
        by_n[bucket].append(s)
    for k in by_n:
        rng.shuffle(by_n[k])

    targets = dict(zip([1, 2, 3, 4, 5], ratios))
    selected = {k: by_n[k][:targets[k]] for k in [1, 2, 3, 4, 5]}
    actual = {k: len(v) for k, v in selected.items()}
    print(f"[STRATIFY] target={targets}, actual={actual}")

    # Interleave pattern: 5 buckets weighted approximately by ratio
    # Pattern length 25, weights derived from ratio percentages
    pattern = ([1] * 5 + [2] * 6 + [3] * 6 + [4] * 4 + [5] * 1)
    rng.shuffle(pattern)
    out = []
    pos = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
    cycle_idx = 0
    while len(out) < total:
        b = pattern[cycle_idx % len(pattern)]
        cycle_idx += 1
        if pos[b] < len(selected[b]):
            out.append(selected[b][pos[b]])
            pos[b] += 1
        else:
            # Fallback: pick from any available bucket
            for fb in [1, 2, 3, 4, 5]:
                if pos[fb] < len(selected[fb]):
                    out.append(selected[fb][pos[fb]])
                    pos[fb] += 1
                    break
            else:
                break  # all quota buckets exhausted

    # Fill a short quota (for example, a scarce 5+ bucket) from the remaining pool.
    if len(out) < total:
        selected_ids = {id(sample) for sample in out}
        remaining = [sample for bucket in by_n.values() for sample in bucket
                     if id(sample) not in selected_ids]
        rng.shuffle(remaining)
        out.extend(remaining[:total - len(out)])

    if len(out) != min(total, len(train_valid)):
        raise RuntimeError(f'Unable to build requested training subset: {len(out)}/{total}')
    return out[:total]

train_sub = _build_stratified_train(train_valid, total=TRAIN_SUBSET)
print(f"[STRATIFY] train_sub size: {len(train_sub)} "
      f"(epochs: {NUM_EPOCHS} → total: {len(train_sub) * NUM_EPOCHS})")


### Init: cluster distribution + fewshot cache + dev lift set


In [ ]:
print(f"\n{'='*70}")
print(f"  INITIALIZATION")
print(f"{'='*70}")

cluster_dist_train = log_cluster_distribution(train_sub, label=f'train_sub ({len(train_sub)})')
cluster_dist_dev   = log_cluster_distribution(dev_valid, label=f'dev_valid ({len(dev_valid)})')

_n_distinct_clusters_train = len(cluster_dist_train)
if _n_distinct_clusters_train < 5:
    print(f"\n{'='*70}")
    print(f"  ❌ Cluster classifier produced only {_n_distinct_clusters_train} cluster(s)")
    print(f"{'='*70}")
    print(f"  Distribution: {cluster_dist_train}")
    print(f"  Likely: phase0_clusters.json not loaded or classifier broken")
    print("  Initialize ACE components before training.")
    print(f"{'='*70}\n")
    raise RuntimeError(
        f"Cluster classifier degraded ({_n_distinct_clusters_train} clusters).")
print(f"\n[CLUSTERS] ✅ {_n_distinct_clusters_train} distinct clusters on train_sub")

# Build fewshot cache from train_sub
print()
_build_fewshot_cache_from_train(train_sub)

# Build fixed dev lift mini-set (stratified by n_ops)
print()
_dev_lift_set_built = build_dev_lift_set(dev_valid, size=globals().get('LIFT_DEV_SET_SIZE', 100))


### Checkpoint candidates


In [ ]:
# Track 3 candidate "best" playbooks:
# 1. best_pb_composite — max(EA·0.6 + PA·0.4) with PA guard
# 2. best_pb_ea_strict — max EA with PA ≥ baseline_PA - 0.02 (avoid lucky-EA)
# 3. best_pb_pa        — max PA (rarely chosen; safety net for PA goal)
# Snapshots saved IMMEDIATELY at detection. No race condition.
eval_log = []
no_improve_count = 0

best_composite_score = 0.0
best_pb_composite     = INITIAL_PLAYBOOK
best_pb_composite_ea  = 0.0
best_pb_composite_pa  = 0.0
best_pb_composite_step = 0

best_ea_strict_score = 0.0
best_pb_ea_strict     = INITIAL_PLAYBOOK
best_pb_ea_strict_ea  = 0.0
best_pb_ea_strict_pa  = 0.0
best_pb_ea_strict_step = 0

best_pa_score = 0.0
best_pb_pa     = INITIAL_PLAYBOOK
best_pb_pa_ea  = 0.0
best_pb_pa_pa  = 0.0
best_pb_pa_step = 0

print(f"\n{'='*70}")
print(f"  STEP 0 (INITIAL — baseline)")
print(f"{'='*70}")
ea_init, pa_init = quick_dev_eval(dev_valid, INITIAL_PLAYBOOK, n=len(dev_valid))
print(f"[EVAL] step=0 EA={ea_init:.4f} PA={pa_init:.4f}")
score_init = COMPOSITE_EA_WEIGHT * ea_init + COMPOSITE_PA_WEIGHT * pa_init

best_composite_score   = score_init
best_pb_composite_ea   = ea_init
best_pb_composite_pa   = pa_init
best_ea_strict_score   = ea_init
best_pb_ea_strict_ea   = ea_init
best_pb_ea_strict_pa   = pa_init
best_pa_score          = pa_init
best_pb_pa_ea          = ea_init
best_pb_pa_pa          = pa_init
baseline_pa            = pa_init  # for ea_strict floor

eval_log.append({
    'step': 0, 'epoch': 0, 'ea': ea_init, 'pa': pa_init, 'score': score_init,
    'pb_size': _pb_stats(INITIAL_PLAYBOOK)['total'],
    'is_full_dev': True, 'is_initial': True,
    'cluster_distribution': {},
})


### Main training loop


In [ ]:
overall_start = time.time()
total_steps = len(train_sub) * NUM_EPOCHS
global_step = 0

print(f"\n{'='*70}")
print(f"  TRAINING — {RUN_NAME} | {NUM_EPOCHS} epochs × {len(train_sub)} samples")
print(f"  Reflector: {HYBRID_MODEL_REFLECTOR} | Verify-iterate: {USE_VERIFY_ITERATE} ({MAX_VERIFY_ROUNDS} rounds)")
print(f"  Composite: EA·{COMPOSITE_EA_WEIGHT} + PA·{COMPOSITE_PA_WEIGHT} | PA guard tol={PA_GUARD_TOL}")
print(f"  Max bullets: {MAX_PLAYBOOK_BULLETS} (T1 max=5) | Cluster quota: 2/cluster")
print(f"  Lift tracking: every {LIFT_EVAL_EVERY} steps × {globals().get('LIFT_DEV_SET_SIZE', 100)} samples")
print(f"  Tier 1 promote: age≥{globals().get('TIER_1_PROMOTION_AGE', 200)}, lift≥{globals().get('TIER_1_PROMOTION_LIFT', 0.01)}")
print(f"  SYSTEM_PROMPT: FULL | "
      f"INITIAL_PLAYBOOK: {'EMPTY' if USE_BARE_PLAYBOOK else 'FULL'} | "
      f"Thinking: {'ON' if USE_THINKING_TRACE else 'OFF'}")
print(f"{'='*70}\n")

for epoch in range(NUM_EPOCHS):
    if epoch > 0:
        train_sub = _build_stratified_train(train_valid, total=TRAIN_SUBSET,
                                              seed=RANDOM_SEED)
        _build_fewshot_cache_from_train(train_sub)
        print(f"\n[EPOCH {epoch+1}] Re-shuffled (seed={RANDOM_SEED})\n")

    for step_in_epoch, sample in enumerate(train_sub):
        global_step += 1
        _current_train_step[0] = global_step

        result = process_one(sample, global_step, total_steps)

        for bid in result.get('used_bullets', []):
            _record_retrieval(bid, global_step)

        # Status print every 20 steps
        if global_step % 20 == 0:
            elapsed = time.time() - overall_start
            rate = global_step / elapsed if elapsed > 0 else 0
            eta_sec = (total_steps - global_step) / rate if rate > 0 else 0
            outcome_str = ' '.join(f"{k}={v}" for k, v in
                                     sorted(outcome_dist.items(), key=lambda x: -x[1])[:4])
            v_str = (f"v_pass={verify_stats['verify_pass_count']} "
                      f"r1={verify_stats['round_1_pass']} "
                      f"r2={verify_stats['round_2_pass']} "
                      f"r3={verify_stats['round_3_pass']}")
            t1_size = len(_get_tier1_bullets()) if '_get_tier1_bullets' in globals() else 0
            print(f"[STEP {global_step:>4}/{total_steps}] "
                  f"e{epoch+1}-s{step_in_epoch+1:<3} "
                  f"pb={_pb_stats(playbook)['total']:>3} (T1={t1_size}) | {outcome_str} | "
                  f"qg_pass={qg_stats['passed']}/{qg_stats['total']} | {v_str} | "
                  f"eta={eta_sec/60:.0f}m")

        # Auto-ablate every 100 steps (now dev-based)
        if global_step % AUTO_ABLATE_EVERY == 0 and global_step > 0:
            multistage_stats['auto_ablate_calls'] += 1
            playbook, evicted = auto_ablate_playbook(playbook, global_step)
            if evicted:
                multistage_stats['auto_ablate_evicted'] += len(evicted)
                src = evicted[0].get('source', 'unknown')
                print(f"[ABLATE-{src}] step={global_step} evicted {len(evicted)}: " +
                      ", ".join(f"{e['id']}(pa_lift={e.get('pa_lift', e.get('lift', 0)):+.3f})"
                                  for e in evicted[:5]))

        # Dev lift tracking + Tier 1 promotion every LIFT_EVAL_EVERY steps
        # (Same cadence as eval, but BEFORE eval so promoted bullets show benefit)
        if global_step % LIFT_EVAL_EVERY == 0 and global_step > 0:
            print(f"\n[LIFT-EVAL] step={global_step}")
            eval_per_bullet_dev_lift(playbook, global_step)
            promoted = promote_tier1_candidates(global_step, bullet_birth_step)
            if promoted:
                print(f"[TIER1] step={global_step} promoted {len(promoted)} bullets to Tier 1")

        # Full eval every EVAL_EVERY steps
        if global_step % EVAL_EVERY == 0 and global_step > 0:
            ea, pa = quick_dev_eval(dev_valid, playbook, n=len(dev_valid))
            score = COMPOSITE_EA_WEIGHT * ea + COMPOSITE_PA_WEIGHT * pa
            cluster_snap = cluster_distribution_snapshot()
            t1_size = len(_get_tier1_bullets()) if '_get_tier1_bullets' in globals() else 0

            print(f"\n[EVAL] step={global_step:>4} "
                  f"EA={ea:.4f}  PA={pa:.4f}  score={score:.4f}  "
                  f"pb={_pb_stats(playbook)['total']} T1={t1_size} "
                  f"clusters_covered={len(cluster_snap)}")

            eval_log.append({
                'step': global_step, 'epoch': epoch + 1,
                'ea': ea, 'pa': pa, 'score': score,
                'pb_size': _pb_stats(playbook)['total'],
                'tier1_size': t1_size,
                'is_full_dev': True,
                'cluster_distribution': cluster_snap,
                'verify_stats_snapshot': dict(verify_stats),
            })

            # 3-snapshot best tracking
            improved_any = False

            # 1. Best composite (with PA guard)
            improved_composite = False
            if score > best_composite_score:
                if (USE_PA_GUARD and pa < best_pb_composite_pa - PA_GUARD_TOL):
                    print(f"  ⚠ PA guard: pa {pa:.3f} < best_pa {best_pb_composite_pa:.3f}"
                          f" - {PA_GUARD_TOL}, skipping composite update")
                else:
                    best_composite_score   = score
                    best_pb_composite       = playbook
                    best_pb_composite_ea    = ea
                    best_pb_composite_pa    = pa
                    best_pb_composite_step  = global_step
                    improved_composite = True
                    improved_any = True
                    print(f"  ✓ NEW best COMPOSITE (score={score:.4f}) at step {global_step}")
                    # IMMEDIATELY save snapshot to avoid race
                    with open(f"{OUTPUT_DIR}/best_playbook.txt", 'w') as f:
                        f.write(best_pb_composite)

            # 2. Best EA-strict (max EA with PA floor)
            improved_ea_strict = False
            pa_floor = baseline_pa - EA_STRICT_PA_FLOOR_DELTA
            if ea > best_ea_strict_score and pa >= pa_floor:
                best_ea_strict_score   = ea
                best_pb_ea_strict       = playbook
                best_pb_ea_strict_ea    = ea
                best_pb_ea_strict_pa    = pa
                best_pb_ea_strict_step  = global_step
                improved_ea_strict = True
                improved_any = True
                print(f"  ✓ NEW best EA-STRICT (ea={ea:.4f}, pa={pa:.4f} ≥ floor={pa_floor:.3f}) "
                      f"at step {global_step}")
                with open(f"{OUTPUT_DIR}/best_playbook_ea.txt", 'w') as f:
                    f.write(best_pb_ea_strict)

            # 3. Best PA (rarely useful but safety net)
            improved_pa = False
            if pa > best_pa_score:
                best_pa_score    = pa
                best_pb_pa        = playbook
                best_pb_pa_ea     = ea
                best_pb_pa_pa     = pa
                best_pb_pa_step   = global_step
                improved_pa = True
                improved_any = True
                print(f"  ✓ NEW best PA-only (pa={pa:.4f}) at step {global_step}")
                with open(f"{OUTPUT_DIR}/best_playbook_pa.txt", 'w') as f:
                    f.write(best_pb_pa)

            # Update compatibility state used by the retrieval cache
            best_playbook = best_pb_composite
            best_dev_ea   = best_pb_composite_ea
            best_dev_pa   = best_pb_composite_pa

            if not improved_any:
                no_improve_count += 1
                print(f"  no-improve count = {no_improve_count}/{EARLY_STOP_PATIENCE}")
                if no_improve_count >= EARLY_STOP_PATIENCE:
                    print(f"  ⚠ EARLY STOP at step {global_step}")
                    save_progress(global_step, epoch=epoch + 1)
                    break
            else:
                no_improve_count = 0

            save_progress(global_step, epoch=epoch + 1)

    else:
        save_progress(global_step, epoch=epoch + 1)
        continue
    break

elapsed_total = time.time() - overall_start
print(f"\n{'='*70}")
print(f"  TRAINING DONE — {global_step}/{total_steps} steps in {elapsed_total/60:.1f}min")
print(f"{'='*70}")


### Post-training pruning


In [ ]:
if USE_POST_TRAINING_PRUNE and len(_all_bullets(best_pb_composite)) >= 5:
    print(f"\n[PRUNE] Running post-training prune on COMPOSITE-best only "
          f"(dev mini-{PRUNE_DEV_SUBSET})...")
    pruned_pb, removed, prune_info = prune_playbook_post_training(
        best_pb_composite, dev_valid,
        lift_threshold=PRUNE_LIFT_THRESHOLD,
        n_samples=PRUNE_DEV_SUBSET,
        verbose=True)

    if removed and not prune_info.get('reverted'):
        # Verify on FULL dev
        ea_pr, pa_pr = quick_dev_eval(dev_valid, pruned_pb, n=len(dev_valid))
        score_pr = COMPOSITE_EA_WEIGHT * ea_pr + COMPOSITE_PA_WEIGHT * pa_pr
        print(f"\n[PRUNE] Pruned full-dev eval: EA={ea_pr:.4f} PA={pa_pr:.4f} score={score_pr:.4f}")
        if score_pr > best_composite_score:
            best_pb_composite = pruned_pb
            best_pb_composite_ea = ea_pr
            best_pb_composite_pa = pa_pr
            best_composite_score = score_pr
            with open(f"{OUTPUT_DIR}/best_playbook.txt", 'w') as f:
                f.write(best_pb_composite)
            print(f"[PRUNE] ✅ Pruned playbook adopted on full dev")
        else:
            print(f"[PRUNE] ⚠ Pruned score worse on full dev — keeping unpruned best")

if SKIP_PRUNE_EA:
    print("\n[PRUNE-EA] Skipped by configuration")
else:
    if best_pb_ea_strict != best_pb_composite and len(_all_bullets(best_pb_ea_strict)) >= 5:
        print(f"\n[PRUNE-EA] Running prune on EA-strict (dev mini-{PRUNE_DEV_SUBSET})...")
        pruned_ea, _, _ = prune_playbook_post_training(
            best_pb_ea_strict, dev_valid,
            lift_threshold=PRUNE_LIFT_THRESHOLD,
            n_samples=PRUNE_DEV_SUBSET,
            verbose=False)
        ea_eap, pa_eap = quick_dev_eval(dev_valid, pruned_ea, n=len(dev_valid))
        if ea_eap > best_pb_ea_strict_ea:
            best_pb_ea_strict = pruned_ea
            best_pb_ea_strict_ea = ea_eap
            best_pb_ea_strict_pa = pa_eap
            with open(f"{OUTPUT_DIR}/best_playbook_ea.txt", 'w') as f:
                f.write(best_pb_ea_strict)
            print(f"[PRUNE-EA] ✅ EA-strict pruned: EA={ea_eap:.4f}")


### Final artifacts


In [ ]:
print(f"\n{'='*70}")
print(f"  FINAL — saving artifacts")
print(f"{'='*70}")

with open(f"{OUTPUT_DIR}/eval_log_{RUN_NAME}.jsonl", 'w') as f:
    for ev in eval_log:
        f.write(json.dumps(ev, default=str) + '\n')

# Save all 3 candidates with run-specific suffixes
with open(f"{OUTPUT_DIR}/playbook_composite_pre_prune_{RUN_NAME}.txt", 'w') as f:
    f.write(best_pb_composite)
with open(f"{OUTPUT_DIR}/playbook_ea_strict_pre_prune_{RUN_NAME}.txt", 'w') as f:
    f.write(best_pb_ea_strict)
with open(f"{OUTPUT_DIR}/playbook_pa_pre_prune_{RUN_NAME}.txt", 'w') as f:
    f.write(best_pb_pa)

with open(f"{OUTPUT_DIR}/best_playbook.txt", 'w') as f:
    f.write(best_pb_composite)
with open(f"{OUTPUT_DIR}/best_playbook_ea.txt", 'w') as f:
    f.write(best_pb_ea_strict)
with open(f"{OUTPUT_DIR}/best_playbook_pa.txt", 'w') as f:
    f.write(best_pb_pa)
with open(f"{OUTPUT_DIR}/best_playbook_{RUN_NAME}.txt", 'w') as f:
    f.write(best_pb_composite)
with open(f"{OUTPUT_DIR}/best_playbook_ea_{RUN_NAME}.txt", 'w') as f:
    f.write(best_pb_ea_strict)
with open(f"{OUTPUT_DIR}/best_playbook_pa_{RUN_NAME}.txt", 'w') as f:
    f.write(best_pb_pa)

def _per_bullet_pa_lift_on_history(playbook_str, history_list, min_uses=10):
    """For diagnostic only — history-based lift (potentially noisy)."""
    bullets = _all_bullets(playbook_str)
    if not history_list:
        return {}
    pa_baseline = sum(1 for h in history_list if h.get('pa_pass')) / len(history_list)
    out = {}
    for b in bullets:
        rec = {'used': 0, 'pa_pass': 0}
        for h in history_list:
            if b['id'] in (h.get('used_bullets') or []):
                rec['used'] += 1
                if h.get('pa_pass'):
                    rec['pa_pass'] += 1
        if rec['used'] >= min_uses:
            out[b['id']] = {
                'used': rec['used'],
                'pa_lift': round(rec['pa_pass'] / rec['used'] - pa_baseline, 4),
                'content_60': b['content'][:60],
                'tier1': is_tier1(b['id']),
            }
    return out

bullet_lifts_history = _per_bullet_pa_lift_on_history(best_pb_composite, history)


### Run manifest


In [ ]:
run_meta = {
    'metric_profile':     'notebook-diagnostic',
    'context_mode':       'oracle_gold_inds',
    'run_name':           RUN_NAME,
    'model_tag':          MODEL_TAG,
    'timestamp':          datetime.now().isoformat(),
    'duration_min':       round(elapsed_total / 60, 1),
    'total_steps':        global_step,
    'completed_epochs':   epoch + 1 if 'epoch' in dir() else 0,
    'num_epochs_target':  NUM_EPOCHS,
    'train_subset_size':  len(train_sub),

    'experiment_config': {
        'mode':                            MODE,
        'use_bare_playbook':               USE_BARE_PLAYBOOK,
        'use_thinking_trace':              USE_THINKING_TRACE,
        'cluster_aware_reflector':         True,
        'cluster_quota_curator':           True,
        'cluster_match_mode':              globals().get('CLUSTER_MATCH_MODE', 'highest_score'),
        'verify_iterate':                  USE_VERIFY_ITERATE,
        'max_verify_rounds':               MAX_VERIFY_ROUNDS,
        'verify_require_pa':               globals().get('VERIFY_REQUIRE_PA', True),
        'verify_dedup_jaccard':            globals().get('VERIFY_DEDUP_JACCARD', 0.90),
        'fewshot_for_hard_queries':        True,
        'fewshot_bootstrap':               True,
        'max_bullets_per_cluster':         MAX_BULLETS_PER_CLUSTER,
        'reflector_model':                 HYBRID_MODEL_REFLECTOR,
        'qg_dedup_threshold':              QG_DEDUP_THRESH,
        'trigger_overlap_threshold':       TRIGGER_OVERLAP_THRESH,
        'action_sig_trig_overlap_thresh':  ACTION_SIG_TRIG_OVERLAP_THR,
        'composite_ea_weight':             COMPOSITE_EA_WEIGHT,
        'composite_pa_weight':             COMPOSITE_PA_WEIGHT,
        'use_pa_guard':                    USE_PA_GUARD,
        'pa_guard_tol':                    PA_GUARD_TOL,
        'max_playbook_bullets':            MAX_PLAYBOOK_BULLETS,
        'tier_1_max':                      globals().get('TIER_1_MAX', 5),
        'tier_1_promotion_age':            globals().get('TIER_1_PROMOTION_AGE', 200),
        'tier_1_promotion_lift':           globals().get('TIER_1_PROMOTION_LIFT', 0.01),
        'eval_every':                      EVAL_EVERY,
        'lift_eval_every':                 LIFT_EVAL_EVERY,
        'lift_dev_set_size':               globals().get('LIFT_DEV_SET_SIZE', 100),
        'auto_ablate_every':               AUTO_ABLATE_EVERY,
        'ablate_data_source':              globals().get('ABLATE_DATA_SOURCE', 'dev'),
        'ablate_pa_lift_thr':              globals().get('ABLATE_PA_LIFT_THR', -0.02),
        'quarantine_cooldown':             globals().get('QUARANTINE_COOLDOWN', 200),
        'early_stop_patience':             EARLY_STOP_PATIENCE,
        'post_training_prune_thr':         PRUNE_LIFT_THRESHOLD,
        'prune_dev_subset':                PRUNE_DEV_SUBSET,
        'skip_prune_ea':                   SKIP_PRUNE_EA,
        'stratified_ratios':               STRATIFIED_RATIOS,
        'validation_n_samples':            globals().get('VALIDATION_N_SAMPLES', 40),
        'common_errors_threshold':         globals().get('COMMON_ERRORS_THRESHOLD', 0.005),
        'rare_errors_threshold':           globals().get('RARE_ERRORS_THRESHOLD', 0.0),
    },

    'best_composite': {
        'step':       best_pb_composite_step,
        'ea':         round(best_pb_composite_ea, 4),
        'pa':         round(best_pb_composite_pa, 4),
        'score':      round(best_composite_score, 4),
        'n_bullets':  _pb_stats(best_pb_composite)['total'],
        'by_section': _pb_stats(best_pb_composite)['by_section'],
    },
    'best_ea_strict': {
        'step':       best_pb_ea_strict_step,
        'ea':         round(best_pb_ea_strict_ea, 4),
        'pa':         round(best_pb_ea_strict_pa, 4),
        'n_bullets':  _pb_stats(best_pb_ea_strict)['total'],
    },
    'best_pa': {
        'step':       best_pb_pa_step,
        'ea':         round(best_pb_pa_ea, 4),
        'pa':         round(best_pb_pa_pa, 4),
        'n_bullets':  _pb_stats(best_pb_pa)['total'],
    },

    'baseline_step0': {
        'ea': round(ea_init, 4),
        'pa': round(pa_init, 4),
    },

    'outcome_dist':   outcome_dist,
    'error_dist':     error_dist,
    'diag_dist':      diag_dist,

    'qg_stats':                qg_stats,
    'multistage_stats':        {k: v for k, v in multistage_stats.items()
                                 if not isinstance(v, list) or len(v) <= 20},

    'cluster_distribution_train':  cluster_dist_train,
    'cluster_distribution_dev':    cluster_dist_dev,
    'cluster_distribution_final':  cluster_distribution_snapshot(),
    'bullet_to_cluster':           dict(_bullet_to_cluster),
    'tier1_bullets_final':         sorted(list(_get_tier1_bullets() if '_get_tier1_bullets' in globals() else set())),
    'verify_stats':                dict(verify_stats),

    'per_bullet_dev_lift_ema':     dict(_dev_lift_ema_cache),
    'per_bullet_pa_lift_history':  bullet_lifts_history,

    'counterfactual_stats':        counterfactual_stats,
    'api_cost':                    api_cost,

    'hyperparams': {
        'top_k_retrieval':       globals().get('TOP_K_RETRIEVAL', 7),
        'top_k_retrieval_tier1': globals().get('TOP_K_RETRIEVAL_TIER1', 4),
        'top_k_retrieval_tier2': globals().get('TOP_K_RETRIEVAL_TIER2', 3),
        'temperature':           TEMPERATURE,
        'max_tokens':            MAX_TOKENS,
        'max_seq_length':        MAX_SEQ_LENGTH,
        'ea_decimal_places':          EA_DECIMAL_PLACES,
    },
}

with open(f"{OUTPUT_DIR}/run_meta_{RUN_NAME}.json", 'w') as f:
    json.dump(run_meta, f, indent=2, default=str)


### Summary


In [ ]:
print(f"\n{'='*70}")
print(f"  TRAINING SUMMARY — {RUN_NAME}")
print(f"{'='*70}")
print(f"  Duration:              {elapsed_total/60:.1f} minutes")
print(f"  Steps completed:       {global_step}/{total_steps}")
print(f"  Reflector model:       {HYBRID_MODEL_REFLECTOR}")
print()
print(f"  ─── 3 BEST CANDIDATES ───")
print(f"  COMPOSITE-BEST:")
print(f"    Step:               {best_pb_composite_step}")
print(f"    Dev EA:             {best_pb_composite_ea:.4f}  ({best_pb_composite_ea*100:.2f}%)")
print(f"    Dev PA:             {best_pb_composite_pa:.4f}  ({best_pb_composite_pa*100:.2f}%)")
print(f"    Score:              {best_composite_score:.4f}")
print(f"    Bullets:            {_pb_stats(best_pb_composite)['total']}")
print()
print(f"  EA-STRICT-BEST (PA ≥ baseline_PA - {EA_STRICT_PA_FLOOR_DELTA}):")
print(f"    Step:               {best_pb_ea_strict_step}")
print(f"    Dev EA:             {best_pb_ea_strict_ea:.4f}")
print(f"    Dev PA:             {best_pb_ea_strict_pa:.4f}")
print(f"    Bullets:            {_pb_stats(best_pb_ea_strict)['total']}")
print()
print(f"  PA-BEST (rarely chosen, safety net):")
print(f"    Step:               {best_pb_pa_step}")
print(f"    Dev EA:             {best_pb_pa_ea:.4f}")
print(f"    Dev PA:             {best_pb_pa_pa:.4f}")
print(f"    Bullets:            {_pb_stats(best_pb_pa)['total']}")
print()
print(f"  MODE                  : {MODE}")
print(f"    SYSTEM_PROMPT       : FULL")
print(f"    INITIAL_PLAYBOOK    : {'EMPTY (pure ACE)' if USE_BARE_PLAYBOOK else 'FULL'}")
print(f"    Step 0 baseline     : EA={ea_init:.4f} PA={pa_init:.4f}")
print(f"    Composite gain      : EA {ea_init:.4f}→{best_pb_composite_ea:.4f} "
      f"({(best_pb_composite_ea-ea_init)*100:+.2f}pp), "
      f"PA {pa_init:.4f}→{best_pb_composite_pa:.4f} "
      f"({(best_pb_composite_pa-pa_init)*100:+.2f}pp)")
print()
print("  ─── FEATURE ACTIVATION ───")
print(f"    Cluster highest-score      : ✅ ({len(_PHASE0_CLUSTERS_DATA['clusters'])} clusters, "
      f"mode={globals().get('CLUSTER_MATCH_MODE', 'highest_score')})")
print(f"    Cluster quota curator      : ✅ "
      f"(rejected {multistage_stats['stage0_5_cluster_quota_full']} due to quota)")
print(f"    Verify-iterate (PA-strict) : ✅ "
      f"(triggered {verify_stats['total_attempts']}, "
      f"r1={verify_stats['round_1_pass']}, "
      f"r2={verify_stats['round_2_pass']}, "
      f"r3={verify_stats['round_3_pass']}, "
      f"PA-fail-blocked={verify_stats.get('verify_pa_fail_count', 0)})")
print(f"    Fewshot for 3+step         : ✅ "
      f"(used {verify_stats['fewshot_used']}, bootstrap {verify_stats['fewshot_bootstrap']})")
print(f"    Tier 1 system              : ✅ "
      f"({multistage_stats['tier1_promotions']} promoted, "
      f"{multistage_stats.get('tier1_fallback_promotions', 0)} fallback)")
print(f"    Dev lift tracking          : ✅ "
      f"({multistage_stats['lift_evals_run']} eval rounds)")
print(f"    Auto-ablate (dev-based)    : ✅ "
      f"(evicted {multistage_stats['auto_ablate_evicted']}, "
      f"T1-protected {multistage_stats.get('auto_ablate_skipped_tier1', 0)})")
print(f"    Lucky-guess Reflector skip : ✅ "
      f"(skipped {verify_stats.get('reflector_skipped_lucky', 0)})")
print()
print(f"  Cluster coverage (final playbook):")
final_cluster_snap = cluster_distribution_snapshot()
for cid in _CLUSTER_PRIORITY:
    n = final_cluster_snap.get(cid, 0)
    bar = '█' * n
    print(f"    {cid:<28} {n}/{MAX_BULLETS_PER_CLUSTER}  {bar}")
print()
print(f"  Outcome distribution (train history):")
for k, v in sorted(outcome_dist.items(), key=lambda x: -x[1]):
    pct = v / global_step * 100 if global_step else 0
    print(f"    {k:<20} {v:>4} ({pct:>5.1f}%)")
print()
print(f"  Top 10 bullets by DEV PA-lift EMA:")
sorted_dev = sorted(_dev_lift_ema_cache.items(),
                     key=lambda x: -x[1].get('pa_lift', 0))[:10]
for bid, info in sorted_dev:
    sign = '✓' if info['pa_lift'] > 0 else ('•' if info['pa_lift'] == 0 else '⚠')
    t1_marker = '🔒' if is_tier1(bid) else '  '
    print(f"    {sign}{t1_marker}[{bid}] pa_lift={info['pa_lift']:+.3f} "
          f"ea_lift={info.get('ea_lift', 0):+.3f} n_evals={info.get('n_evals', 0)}")
print()
print(f"  Files saved to {OUTPUT_DIR}:")
print(f"    - best_playbook.txt              (composite-best; used by evaluation)")
print(f"    - best_playbook_ea.txt           (EA-strict candidate)")
print(f"    - best_playbook_pa.txt           (PA-best candidate)")
print(f"    - best_playbook_{RUN_NAME}.txt   (composite, run-tagged)")
print(f"    - playbook_composite_pre_prune_{RUN_NAME}.txt")
print(f"    - run_meta_{RUN_NAME}.json")
print(f"    - eval_log_{RUN_NAME}.jsonl")
print(f"    - history.jsonl")
print(f"\n{'='*70}")
print(f"  ✅ {RUN_NAME} TRAINING COMPLETE — proceed to evaluation")
print(f"{'='*70}\n")


## Diagnostics and coverage


In [ ]:
print(f"\n{'='*60}")
print(f"  DIAGNOSTIC + COVERAGE AUDIT — {MODEL_TAG}")
print(f"{'='*60}")

from collections import Counter
import math
import re


### A. FAILURE DIAGNOSIS


In [ ]:
print(f"\n[A] FAILURE DIAGNOSIS:")
total_fail = sum(diag_dist.values())
for diag_type, cnt in sorted(diag_dist.items(), key=lambda x:-x[1]):
    pct = cnt/total_fail*100 if total_fail else 0
    bar = "█" * int(pct/2)
    print(f"  {diag_type:<35} {cnt:>4} ({pct:>5.1f}%) {bar}")

# B. ERROR TYPES
print(f"\n[B] ERROR TYPE DISTRIBUTION:")
total_err = sum(error_dist.values())
for etype, cnt in sorted(error_dist.items(), key=lambda x:-x[1]):
    pct = cnt/total_err*100 if total_err else 0
    print(f"  {etype:<25} {cnt:>4} ({pct:>5.1f}%)")

# C. QG STATS
print(f"\n[C] QUALITY GATE:")
print(f"  Total candidates: {qg_stats['total']}")
print(f"  ✅ Passed (added): {qg_stats['passed']} ({qg_stats['passed']/max(1,qg_stats['total'])*100:.1f}%)")
print(f"  ✗ Rejected by QG: {qg_stats.get('rejected_qg',0)}")
print(f"  Top rejection reasons:")
for reason, cnt in sorted(qg_stats.get('reasons',{}).items(), key=lambda x:-x[1])[:5]:
    print(f"    {reason:<40} {cnt:>4}")


### D. PLAYBOOK


In [ ]:
print(f"\n[D] PLAYBOOK ({_pb_stats(playbook)['total']} bullets):")
for sec, cnt in _pb_stats(playbook)['by_section'].items():
    print(f"  {sec}: {cnt}")
bullets = _all_bullets(playbook)
sorted_b = sorted(bullets, key=lambda b: b['helpful']-b['harmful'], reverse=True)
print(f"\n  Top 5 helpful:")
for b in sorted_b[:5]:
    print(f"    [{b['id']}] h={b['helpful']} harm={b['harmful']} :: {b['content'][:80]}")

# E. OUTCOME ANALYSIS
oc = Counter(h.get('outcome', 'unknown') for h in history)
total = sum(oc.values())
n_lucky   = oc.get('lucky_guess', 0)
n_exec    = oc.get('exec_mismatch', 0)
n_correct = oc.get('correct', 0)
n_wrong   = oc.get('wrong_reasoning', 0)
n_ea_ok   = n_correct + n_lucky
n_pa_ok   = n_correct + n_exec

print(f"\n[E] OUTCOME ANALYSIS ({total} samples):")
for outcome in ['correct','lucky_guess','exec_mismatch','close_but_wrong','wrong_reasoning']:
    count = oc.get(outcome, 0)
    pct = count/total*100 if total else 0
    bar = '█' * int(pct/2)
    print(f"  {outcome:<18} {count:>4} ({pct:>5.1f}%) {bar}")
print(f"\n  EA pass rate         : {n_ea_ok/max(1,total)*100:.1f}% ({n_ea_ok}/{total})")
print(f"  PA pass rate         : {n_pa_ok/max(1,total)*100:.1f}% ({n_pa_ok}/{total})")
print(f"  Lucky/EA-correct     : {n_lucky/max(1,n_ea_ok)*100:.1f}%")


### F. Coverage audit


In [ ]:
print(f"\n[F] COVERAGE AUDIT:")

# L1 — Outcome coverage
_with_outcome = sum(1 for h in history if h.get('outcome'))
_l1_rate = _with_outcome / max(1, len(history))
print(f"  L1 Outcome coverage  : {_l1_rate*100:.1f}% ({_with_outcome}/{len(history)})")

# L2 — Error-type specificity
_vague_errors = {'', 'other', 'parse_failed', 'wrong_reasoning', None}
_vague = sum(1 for h in history if h.get('error_type') in _vague_errors)
_l2_rate = 1.0 - (_vague / max(1, len(history)))
print(f"  L2 Error-type specific: {_l2_rate*100:.1f}%")

# L3 — Diagnosis specificity
_dumping_diags = {'wrong_value'}
_diag_total = sum(1 for h in history if not h.get('ea_pass', False))
_diag_vague = sum(1 for h in history
                  if not h.get('ea_pass', False) and h.get('diag','') in _dumping_diags)
_l3_rate = (1.0 - (_diag_vague / _diag_total)) if _diag_total > 0 else 1.0
print(f"  L3 Diag specific     : {_l3_rate*100:.1f}% (wrong_value dumping: {_diag_vague}/{_diag_total})")

# L4 — Bullet retrieval distribution
_section_prefix_map = {'ns':'NUMERICAL','tr':'TABLE_READING','pg':'PROGRAM_GEN',
                       'ce':'COMMON_ERRORS','ct':'CONTEXT_TIPS'}
_retrieved_sections = Counter()
for h in history:
    for bid in (h.get('used_bullets') or []):
        if '-' in bid:
            sec = _section_prefix_map.get(bid.split('-')[0].lower(), 'UNKNOWN')
            _retrieved_sections[sec] += 1
total_retrieved = sum(_retrieved_sections.values())
print(f"  L4 Bullet retrieval  : {len(_retrieved_sections)} sections active")
for sec, cnt in _retrieved_sections.most_common():
    pct = cnt / max(1, total_retrieved) * 100
    print(f"    {sec:<17} {cnt:>5} ({pct:>5.1f}%)")


### SAVE


In [ ]:
report = {
    'model': MODEL_TAG,
    'total_history': len(history),
    'diag_dist': diag_dist,
    'error_dist': error_dist,
    'qg_stats': qg_stats,
    'playbook_stats': _pb_stats(playbook),
    'best_dev_ea': best_dev_ea,
    'best_dev_pa': best_dev_pa,
    'api_cost': api_cost,
    'outcome_distribution': dict(oc),
    'ea_rate': round(n_ea_ok/max(1,total), 4),
    'pa_rate': round(n_pa_ok/max(1,total), 4),
    'coverage': {
        'L1_outcome': round(_l1_rate, 4),
        'L2_error_specific': round(_l2_rate, 4),
        'L3_diag_specific': round(_l3_rate, 4),
        'L4_section_retrieval': dict(_retrieved_sections),
    },
}
with open(f"{OUTPUT_DIR}/diagnostic_report.json", 'w') as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)
print(f"\n[SAVE] → {OUTPUT_DIR}/diagnostic_report.json")
print(f"{'='*60}")


## Candidate evaluation


In [ ]:
import os, json, time, re
from collections import Counter
from vllm import SamplingParams


### Consistency check — verify mode is active before eval


In [ ]:
CURRENT_RUN     = globals().get('RUN_NAME', 'FULL_thesis')
EVAL_BATCH      = _BATCH_HINT if '_BATCH_HINT' in globals() else 256
INFER_TEMP      = 0.0

TEST_PATH         = globals().get('TEST_PATH', os.path.join(DATA_DIR, 'test.json'))
TEST_RESULTS_PATH = f"{OUTPUT_DIR}/test_results.json"
TEST_DETAIL_PATH  = f"{OUTPUT_DIR}/test_detail.jsonl"

dev_full = dev_valid

print("="*70)
print("  EVALUATION CONSISTENCY CHECK")
print("="*70)
sys_prompt_len  = len(tokenizer.encode(SYSTEM_PROMPT))
sys_prompt_kind = "FULL"
print(f"  Run name            : {CURRENT_RUN}")
print(f"  Active SYSTEM_PROMPT: {sys_prompt_kind} ({sys_prompt_len} tokens)")
print(f"  USE_BARE_PLAYBOOK   : {globals().get('USE_BARE_PLAYBOOK', '?')}")
print("="*70)


### Evaluation summary


In [ ]:
print(f"\n{'='*70}")
print(f"  UNIFIED EVAL — All Candidates × (Dev + Test)")
print(f"  Run         : {CURRENT_RUN}")
print(f"  Model       : {MODEL_TAG}")
print(f"  API usage   : ✗ NONE (SLM-only)")
print(f"  Self-Consist: ✗ DISABLED (pure ACE)")
print(f"  Decoding    : single-shot greedy (temp={INFER_TEMP})")
print(f"  Dev samples : {len(dev_full)} (FULL)")
print("  Thesis ref. : EA=0.6806, PA=0.6190")
print(f"{'='*70}\n")


### Candidate playbooks


In [ ]:
playbooks_to_eval = []

primary_path = f"{OUTPUT_DIR}/best_playbook_{CURRENT_RUN}.txt"
if not os.path.exists(primary_path):
    raise FileNotFoundError(
        f"Primary playbook not found: {primary_path}\n"
        f"Run training for {CURRENT_RUN} first.")
with open(primary_path) as f:
    primary_pb = f.read().strip()
playbooks_to_eval.append(
    ('CAND_A (composite-best)', f'{CURRENT_RUN}__cand_A', primary_path, 'best_step'))

final_path = f"{OUTPUT_DIR}/final_playbook_{CURRENT_RUN}.txt"
if os.path.exists(final_path):
    with open(final_path) as f:
        final_pb = f.read().strip()
    if final_pb != primary_pb:
        playbooks_to_eval.append(
            ('CAND_B (final-step, more bullets)', f'{CURRENT_RUN}__cand_B', final_path, 'final_step'))
    else:
        print(f"[INFO] {os.path.basename(final_path)} identical to primary → skip")

ea_path = f"{OUTPUT_DIR}/best_playbook_ea_{CURRENT_RUN}.txt"
if os.path.exists(ea_path):
    with open(ea_path) as f:
        ea_pb = f.read().strip()
    if ea_pb != primary_pb:
        playbooks_to_eval.append(
            ('CAND_C (ea-only ablation)', f'{CURRENT_RUN}__cand_C', ea_path, 'ea_only_step'))
    else:
        print(f"[INFO] {os.path.basename(ea_path)} identical to primary → skip")

print(f"[CANDIDATES] {len(playbooks_to_eval)} playbook(s) to evaluate:")
for label, name, path, src in playbooks_to_eval:
    pb_size = open(path).read().count('- [')
    print(f"  {label:<40}  bullets={pb_size:>3}  ({os.path.basename(path)})")


### Load test data


In [ ]:
if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(f"test.json not found at {TEST_PATH}")

with open(TEST_PATH) as f:
    test_raw = json.load(f)
test_valid = [s for s in test_raw if s.get('qa', {}).get('program', '').strip()]
print(f"\n[DATA] Test: {len(test_valid)} valid / {len(test_raw)}")


### Evaluation function


In [ ]:
def eval_playbook(playbook_text, samples, eval_label, batch_size=EVAL_BATCH,
                    keep_details=False):
    pb_size = _pb_stats(playbook_text)['total']

    t0 = time.time()
    print(f"  [1/3] Building prompts...", end=" ", flush=True)
    all_prompts, all_meta = [], []
    for s in samples:
        bul = retrieve_top_k(s['qa']['question'], playbook_text, TOP_K_RETRIEVAL)
        prompt = build_ace_prompt(s, playbook_bullets=bul)
        all_prompts.append(prompt)
        all_meta.append({
            'gold_ans':  s['qa'].get('exe_ans'),
            'gold_prog': s['qa']['program'],
            'table':     s.get('table', []),
            'question':  s['qa'].get('question', '')[:80],
        })
    print(f"done ({time.time()-t0:.1f}s)")

    print(f"  [2/3] Generating (batch={batch_size})...")
    t1 = time.time()
    sp = SamplingParams(
        temperature=INFER_TEMP, max_tokens=MAX_TOKENS,
        repetition_penalty=REPETITION_PENALTY, skip_special_tokens=True,
        seed=RANDOM_SEED)
    all_outputs = []
    n_batches = (len(all_prompts) + batch_size - 1) // batch_size
    for b_idx in range(n_batches):
        start = b_idx * batch_size
        end   = min(start + batch_size, len(all_prompts))
        outs  = model.fast_generate(all_prompts[start:end], sampling_params=sp)
        all_outputs.extend(outs)
        elapsed = time.time() - t1
        eta = elapsed / (b_idx + 1) * (n_batches - b_idx - 1)
        print(f"    batch {b_idx+1}/{n_batches}  elapsed {elapsed:.0f}s  ETA {eta:.0f}s",
              end="\r")
    t_gen = time.time() - t1
    print(f"\n  ✅ Gen done in {t_gen:.1f}s ({t_gen/60:.1f} min)")

    print(f"  [3/3] Scoring...")
    final_ea = final_pa = final_none = 0
    ea_by_steps, pa_by_steps = {}, {}
    outcome_dist = Counter()
    details = []

    _OP_RE = re.compile(
        r'(add|subtract|multiply|divide|greater|exp|table_\w+)\(', re.I)

    for i, (out, meta) in enumerate(zip(all_outputs, all_meta)):
        raw_text = out.outputs[0].text
        pp = extract_program(raw_text)
        pv = execute_program(pp, meta['table']) if pp else None
        gold_ans  = meta['gold_ans']
        gold_prog = meta['gold_prog']

        ea_pass = (check_ea(pv, gold_ans, EA_DECIMAL_PLACES)
                   if (gold_ans is not None and pv is not None) else False)
        pa_pass = check_pa(pp, gold_prog)

        if ea_pass:    final_ea   += 1
        if pa_pass:    final_pa   += 1
        if pv is None: final_none += 1

        n_ops = len(_OP_RE.findall((gold_prog or '').lower()))
        if n_ops > 0:
            ea_by_steps.setdefault(n_ops, [0, 0])
            pa_by_steps.setdefault(n_ops, [0, 0])
            ea_by_steps[n_ops][0] += 1
            pa_by_steps[n_ops][0] += 1
            if ea_pass: ea_by_steps[n_ops][1] += 1
            if pa_pass: pa_by_steps[n_ops][1] += 1

        try:
            oc = classify_outcome(ea_pass, pa_pass, pp, pv, gold_prog, gold_ans)
        except Exception:
            oc = ('correct' if ea_pass and pa_pass else
                  'lucky_guess' if ea_pass else
                  'exec_mismatch' if pa_pass else 'wrong_reasoning')
        outcome_dist[oc] += 1

        if keep_details:
            details.append({
                'idx': i,
                'question':  meta['question'],
                'gold_prog': gold_prog,
                'gold_ans':  gold_ans,
                'pred_prog': pp,
                'pred_ans':  pv,
                'ea_pass':   ea_pass,
                'pa_pass':   pa_pass,
                'outcome':   oc,
                'n_ops':     n_ops,
            })

    total = len(samples)
    t_total = time.time() - t0

    return {
        'label':         eval_label,
        'total':         total,
        'EA':            round(final_ea / total, 4),
        'PA':            round(final_pa / total, 4),
        'EA_PA_gap':     round((final_ea - final_pa) / total, 4),
        'None_rate':     round(final_none / total, 4),
        'EA_correct':    final_ea,
        'PA_correct':    final_pa,
        'None_count':    final_none,
        'bullets':       pb_size,
        'ea_by_steps':   {str(k): v for k, v in ea_by_steps.items()},
        'pa_by_steps':   {str(k): v for k, v in pa_by_steps.items()},
        'outcome_dist':  dict(outcome_dist),
        'eval_time':     round(t_total, 1),
        'throughput':    round(total / max(t_gen, 1), 2),
        'details':       details if keep_details else None,
    }

def print_results(label, r, total):
    print(f"\n  ─── RESULTS [{label}] ───")
    print(f"  EA   : {r['EA']:.4f}  ({r['EA_correct']}/{total})")
    print(f"  PA   : {r['PA']:.4f}  ({r['PA_correct']}/{total})")
    print(f"  Gap  : {r['EA_PA_gap']:.4f}")
    print(f"  None : {r['None_rate']:.4f}  ({r['None_count']}/{total})")
    print(f"  Time : {r['eval_time']:.1f}s | throughput={r['throughput']} q/s")

    print(f"\n  ─── PER-STEP BREAKDOWN ───")
    print(f"  {'#ops':<6} {'count':>7} {'EA%':>8} {'PA%':>8}")
    for n_ops in sorted(r['ea_by_steps'].keys(), key=int):
        et, ec = r['ea_by_steps'][n_ops]
        pt, pc = r['pa_by_steps'][n_ops]
        print(f"  {n_ops:<6} {et:>7} {ec/et:>7.1%} {pc/pt:>7.1%}")

    print(f"\n  ─── OUTCOME DIST ───")
    for oc, cnt in sorted(r['outcome_dist'].items(), key=lambda x: -x[1]):
        print(f"  {oc:<20}: {cnt:>4} ({cnt/total:.1%})")


### Development evaluation


In [ ]:
print(f"\n\n{'═'*70}")
print(f"  STAGE 1 — DEV EVAL ({len(dev_full)} samples × {len(playbooks_to_eval)} candidates)")
print(f"{'═'*70}")

dev_results = {}
for label, run_name, pb_path, source_step in playbooks_to_eval:
    print(f"\n[{run_name}] {label}")
    print(f"  File: {os.path.basename(pb_path)}")
    with open(pb_path) as f:
        pb = f.read()

    r = eval_playbook(pb, dev_full, label, keep_details=False)
    r['run_name']      = run_name
    r['playbook_file'] = os.path.basename(pb_path)
    r['source_step']   = source_step
    r['playbook_path'] = pb_path
    dev_results[run_name] = r
    print_results(run_name, r, len(dev_full))


### Test evaluation


In [ ]:
print(f"\n\n{'═'*70}")
print(f"  STAGE 2 — TEST EVAL ({len(test_valid)} samples × {len(playbooks_to_eval)} candidates)")
print(f"{'═'*70}")

test_results = {}
for label, run_name, pb_path, source_step in playbooks_to_eval:
    print(f"\n[{run_name}] {label}")
    print(f"  File: {os.path.basename(pb_path)}")
    with open(pb_path) as f:
        pb = f.read()

    is_last = (run_name == playbooks_to_eval[-1][1])
    r = eval_playbook(pb, test_valid, label, keep_details=is_last)
    r['run_name']      = run_name
    r['playbook_file'] = os.path.basename(pb_path)
    r['source_step']   = source_step
    test_results[run_name] = r
    print_results(run_name, r, len(test_valid))


### Dev/test comparison


In [ ]:
print(f"\n\n{'═'*78}")
print(f"  STAGE 3 — DEV vs TEST COMPARISON ({CURRENT_RUN})")
print(f"{'═'*78}")

sorted_by_dev = sorted(dev_results.items(), key=lambda x: -x[1]['EA'])

print(f"\n{'Candidate':<42} {'Set':<6} {'EA':>8} {'PA':>8} {'Gap':>8} {'None':>8} {'Bull':>5}")
print(f"{'-'*42} {'-'*6} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*5}")

for rank, (run_name, dev_r) in enumerate(sorted_by_dev, 1):
    test_r = test_results[run_name]
    label = dev_r['label'][:40]
    marker = "🏆 " if rank == 1 else "   "

    print(f"{marker}{label:<40} {'DEV':<6} "
          f"{dev_r['EA']:>8.4f} {dev_r['PA']:>8.4f} {dev_r['EA_PA_gap']:>8.4f} "
          f"{dev_r['None_rate']:>8.4f} {dev_r['bullets']:>5}")
    print(f"   {'':<40} {'TEST':<6} "
          f"{test_r['EA']:>8.4f} {test_r['PA']:>8.4f} {test_r['EA_PA_gap']:>8.4f} "
          f"{test_r['None_rate']:>8.4f} {'':>5}")
    delta_ea = test_r['EA'] - dev_r['EA']
    delta_pa = test_r['PA'] - dev_r['PA']
    print(f"   {'':<40} {'Δ':<6} "
          f"{delta_ea:>+8.4f} {delta_pa:>+8.4f}")
    print()

print(f"{'═'*78}")


### Winner


In [ ]:
winner_name, winner_dev = sorted_by_dev[0]
winner_test = test_results[winner_name]

print(f"\n  🏆 DEV WINNER (selected by dev{len(dev_full)} EA):")
print(f"     {winner_dev['label']}")
print(f"     File: {winner_dev['playbook_file']}")
print(f"     Bullets: {winner_dev['bullets']}")
print(f"")
print(f"     Dev   : EA={winner_dev['EA']:.4f}  PA={winner_dev['PA']:.4f}")
print(f"     Test  : EA={winner_test['EA']:.4f}  PA={winner_test['PA']:.4f}")
print(f"     Gen.Δ : EA={winner_test['EA']-winner_dev['EA']:+.4f}  "
      f"PA={winner_test['PA']-winner_dev['PA']:+.4f}")

winner_pb_path = winner_dev['playbook_path']
canonical_path = f"{OUTPUT_DIR}/best_playbook.txt"
with open(winner_pb_path) as fr, open(canonical_path, 'w') as fw:
    fw.write(fr.read())
print(f"\n  💾 Canonical best_playbook.txt → updated to winner")


### Thesis reference comparison


In [ ]:
ea_pass = winner_test['EA'] >= 0.6806
pa_pass = winner_test['PA'] >= 0.6190

print(f"\n  ─── THESIS REFERENCE COMPARISON (diagnostic test run) ───")
print(f"  EA ≥ 0.6806 : {winner_test['EA']:.4f}  {'✅ PASS' if ea_pass else '❌ FAIL'}")
print(f"  PA ≥ 0.6190 : {winner_test['PA']:.4f}  {'✅ PASS' if pa_pass else '❌ FAIL'}")
print(f"  API-free    : ✅ PASS (no API calls)")
print(f"  Pure ACE    : ✅ PASS (no Self-Consistency)")


### Save artifacts


In [ ]:
for run_name, r in dev_results.items():
    save_path = f"{OUTPUT_DIR}/final_results_{run_name}_dev.json"
    save_data = {k: v for k, v in r.items() if k != 'details'}
    with open(save_path, 'w') as f:
        json.dump(save_data, f, indent=2, ensure_ascii=False)

for run_name, r in test_results.items():
    save_path = f"{OUTPUT_DIR}/final_results_{run_name}_test.json"
    save_data = {k: v for k, v in r.items() if k != 'details'}
    with open(save_path, 'w') as f:
        json.dump(save_data, f, indent=2, ensure_ascii=False)

test_save = {
    'metric_profile': 'notebook-diagnostic',
    'context_mode': 'oracle_gold_inds',
    'model': MODEL_TAG,
    'method': 'pure ACE (playbook only) + single-shot greedy',
    'inference_paradigm': 'SLM-only, pure ACE (no SC)',
    'config': {
        'use_self_consistency': False,
        'inference_temperature': INFER_TEMP,
        'top_k_retrieval': TOP_K_RETRIEVAL,
        'playbook_bullets': winner_dev['bullets'],
        'winner_run_name': winner_name,
        'winner_label': winner_dev['label'],
        'winner_file': winner_dev['playbook_file'],
    },
    'total_samples': len(test_valid),
    'test': {
        'EA':         winner_test['EA'],
        'PA':         winner_test['PA'],
        'EA_PA_gap':  winner_test['EA_PA_gap'],
        'ea_correct': winner_test['EA_correct'],
        'pa_correct': winner_test['PA_correct'],
        'none_count': winner_test['None_count'],
        'none_rate':  winner_test['None_rate'],
    },
    'ea_by_steps':  winner_test['ea_by_steps'],
    'pa_by_steps':  winner_test['pa_by_steps'],
    'outcome_dist': winner_test['outcome_dist'],
    'dev_baseline': {'EA': winner_dev['EA'], 'PA': winner_dev['PA']},
    'goals': {
        'thesis_EA_reference': 0.6806, 'thesis_PA_reference': 0.6190,
        'EA_pass':   ea_pass, 'PA_pass': pa_pass,
        'no_api_pass': True, 'no_sc_pass': True,
    },
    'eval_time_seconds':         winner_test['eval_time'],
    'eval_throughput_samples_s': winner_test['throughput'],
}
with open(TEST_RESULTS_PATH, 'w') as f:
    json.dump(test_save, f, indent=2, ensure_ascii=False)

last_test = test_results[playbooks_to_eval[-1][1]]
if last_test.get('details'):
    with open(TEST_DETAIL_PATH, 'w') as f:
        for d in last_test['details']:
            f.write(json.dumps(d, ensure_ascii=False, default=str) + '\n')
    print(f"\n  📄 Test details (last candidate): {TEST_DETAIL_PATH}")

summary_path = f"{OUTPUT_DIR}/comparison_summary_{CURRENT_RUN}.json"
with open(summary_path, 'w') as f:
    json.dump({
        'current_run':    CURRENT_RUN,
        'eval_dev_size':  len(dev_full),
        'eval_test_size': len(test_valid),
        'mode': 'unified_dev_test_eval_all_candidates',
        'candidates_evaluated': sorted(dev_results.keys()),
        'winner': {
            'run_name': winner_name,
            'label':    winner_dev['label'],
            'file':     winner_dev['playbook_file'],
            'bullets':  winner_dev['bullets'],
            'dev':  {'EA': winner_dev['EA'], 'PA': winner_dev['PA']},
            'test': {'EA': winner_test['EA'], 'PA': winner_test['PA']},
            'gen_gap': {
                'EA': round(winner_test['EA'] - winner_dev['EA'], 4),
                'PA': round(winner_test['PA'] - winner_dev['PA'], 4),
            },
        },
        'all_candidates': {
            name: {
                'label':    dev_results[name]['label'],
                'file':     dev_results[name]['playbook_file'],
                'bullets':  dev_results[name]['bullets'],
                'source_step': dev_results[name]['source_step'],
                'dev': {
                    'EA': dev_results[name]['EA'],
                    'PA': dev_results[name]['PA'],
                    'gap': dev_results[name]['EA_PA_gap'],
                    'None_rate': dev_results[name]['None_rate'],
                    'ea_by_steps': dev_results[name]['ea_by_steps'],
                    'pa_by_steps': dev_results[name]['pa_by_steps'],
                    'outcome_dist': dev_results[name]['outcome_dist'],
                },
                'test': {
                    'EA': test_results[name]['EA'],
                    'PA': test_results[name]['PA'],
                    'gap': test_results[name]['EA_PA_gap'],
                    'None_rate': test_results[name]['None_rate'],
                    'ea_by_steps': test_results[name]['ea_by_steps'],
                    'pa_by_steps': test_results[name]['pa_by_steps'],
                    'outcome_dist': test_results[name]['outcome_dist'],
                },
                'gen_gap': {
                    'EA': round(test_results[name]['EA'] - dev_results[name]['EA'], 4),
                    'PA': round(test_results[name]['PA'] - dev_results[name]['PA'], 4),
                },
            }
            for name in dev_results.keys()
        },
    }, f, indent=2, ensure_ascii=False)

print(f"\n  ─── ARTIFACTS SAVED ───")
print(f"  📄 Per-cand dev    : {OUTPUT_DIR}/final_results_{CURRENT_RUN}__cand_*_dev.json")
print(f"  📄 Per-cand test   : {OUTPUT_DIR}/final_results_{CURRENT_RUN}__cand_*_test.json")
print(f"  📄 Diagnostic test  : {TEST_RESULTS_PATH}")
print(f"  📄 Unified summary : {summary_path}")
print(f"  📄 Canonical PB    : {canonical_path}")

print(f"\n{'═'*70}")
print(f"  ✅ UNIFIED EVAL DONE — {CURRENT_RUN}")
print(f"     {len(playbooks_to_eval)} candidates × 2 sets = {len(playbooks_to_eval)*2} eval runs")
print(f"     Winner: {winner_dev['label']}")
print(f"     Dev EA={winner_dev['EA']:.4f}  Test EA={winner_test['EA']:.4f}")
print(f"{'═'*70}")


## Visualization and diagnostics


In [ ]:
# Fixes from v2:
# - Read final_results_{RUN_NAME}__cand_*_test.json
# - Read test_results.json (winner candidate)
# - Use eval_log instead of _dev_eval_history
# - Use best_pb_composite_ea/pa instead of best_dev_ea
# - Use multistage_stats / verify_stats
# - Bullets parsed via _all_bullets, helpful/harmful from history retrieval

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re, os, json, glob
from datetime import datetime
from collections import Counter

VIZ_DIR = f"{OUTPUT_DIR}/viz"
os.makedirs(VIZ_DIR, exist_ok=True)

plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10,
    'figure.dpi': 100, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

COLORS = {
    'ea':'#2E7D32','pa':'#1565C0','none':'#D32F2F','playbook':'#F57C00',
    'helpful':'#4CAF50','harmful':'#E53935',
    'wrong_value':'#FF9800','magnitude':'#F44336','format':'#9C27B0',
    'sign':'#009688','exec':'#3F51B5',
    'correct':'#2E7D32','lucky':'#FB8C00','exec_mis':'#1976D2',
    'close':'#7B1FA2','wrong':'#C62828',
    'bullets':'#FF9800',
}

def _save_fig(fig, name):
    path = f"{VIZ_DIR}/{name}.png"
    fig.savefig(path, dpi=300, bbox_inches='tight')
    print(f"  💾 {path}")
    plt.close(fig)

def _count_ops(prog):
    if not prog: return 0
    return len(re.findall(
        r'(add|subtract|multiply|divide|greater|exp|table_\w+)\(',
        prog.lower()))

# Load run_meta, test_results, eval_log
_run_name = globals().get('RUN_NAME', 'FULL_thesis')

# Load run_meta_{RUN_NAME}.json
run_meta_path = f"{OUTPUT_DIR}/run_meta_{_run_name}.json"
run_meta = {}
if os.path.exists(run_meta_path):
    with open(run_meta_path) as f:
        run_meta = json.load(f)
    print(f"[VIZ] ✅ Loaded run_meta: best EA={run_meta.get('best_composite',{}).get('ea',0):.4f}")
else:
    print(f"[VIZ] ⚠ run_meta not found: {run_meta_path}")

# Load the selected evaluation result
test_results_path = f"{OUTPUT_DIR}/test_results.json"
test_results = {}
if os.path.exists(test_results_path):
    with open(test_results_path) as f:
        test_results = json.load(f)
    print(f"[VIZ] ✅ Loaded test_results: test EA={test_results.get('test',{}).get('EA',0):.4f}")
else:
    print(f"[VIZ] ⚠ test_results not found — run evaluation first")

# Load comparison_summary.json
comparison_path = f"{OUTPUT_DIR}/comparison_summary_{_run_name}.json"
comparison = {}
if os.path.exists(comparison_path):
    with open(comparison_path) as f:
        comparison = json.load(f)
    print(f"[VIZ] ✅ Loaded comparison_summary")
else:
    # Try without RUN_NAME suffix
    fallback = glob.glob(f"{OUTPUT_DIR}/comparison_summary_*.json")
    if fallback:
        with open(fallback[0]) as f:
            comparison = json.load(f)
        print(f"[VIZ] ✅ Loaded comparison_summary (fallback): {fallback[0]}")

# Load eval_log_{RUN_NAME}.jsonl
eval_log_path = f"{OUTPUT_DIR}/eval_log_{_run_name}.jsonl"
eval_log_data = []
if os.path.exists(eval_log_path):
    with open(eval_log_path) as f:
        for line in f:
            eval_log_data.append(json.loads(line.strip()))
    print(f"[VIZ] ✅ Loaded eval_log: {len(eval_log_data)} eval points")
else:
    print(f"[VIZ] ⚠ eval_log not found: {eval_log_path}")

# Resolve final metrics
final_test_ea = test_results.get('test', {}).get('EA',
                  comparison.get('winner', {}).get('test', {}).get('EA', 0.0))
final_test_pa = test_results.get('test', {}).get('PA',
                  comparison.get('winner', {}).get('test', {}).get('PA', 0.0))
final_dev_ea  = test_results.get('dev_baseline', {}).get('EA',
                  run_meta.get('best_composite', {}).get('ea', 0.0))
final_dev_pa  = test_results.get('dev_baseline', {}).get('PA',
                  run_meta.get('best_composite', {}).get('pa', 0.0))
final_none_rate = test_results.get('test', {}).get('none_rate', 0.0)

# Best from training
best_pb_composite_ea = globals().get('best_pb_composite_ea',
                                       run_meta.get('best_composite', {}).get('ea', 0.0))
best_pb_composite_pa = globals().get('best_pb_composite_pa',
                                       run_meta.get('best_composite', {}).get('pa', 0.0))
best_pb_composite_step = run_meta.get('best_composite', {}).get('step', 0)

# Baseline (step 0 from run_meta)
baseline_ea = run_meta.get('baseline_step0', {}).get('ea', None)
HAS_BASELINE = baseline_ea is not None and baseline_ea > 0
if not HAS_BASELINE:
    baseline_ea = 0.0
    print(f"[VIZ] ℹ No baseline available")

# EA by steps (from test_results)
ea_by_steps_test = test_results.get('ea_by_steps', {})
ea_by_steps_test_int = {int(k): v for k, v in ea_by_steps_test.items()}

# Stats from run_meta
diag_dist = run_meta.get('diag_dist', {})
error_dist = run_meta.get('error_dist', {})
outcome_dist = run_meta.get('outcome_dist', {})
qg_stats = run_meta.get('qg_stats', {})
multistage_stats = run_meta.get('multistage_stats', {})
verify_stats = run_meta.get('verify_stats', {})
api_cost = run_meta.get('api_cost', {})

# Test outcome
test_outcome_dist = test_results.get('outcome_dist', {})

# Cluster final
cluster_dist_final = run_meta.get('cluster_distribution_final', {})

# Bullets from progress.json or current globals
progress_path = f"{OUTPUT_DIR}/progress.json"
bullet_birth = {}
if os.path.exists(progress_path):
    with open(progress_path) as f:
        prog = json.load(f)
    bullet_birth = prog.get('bullet_birth_step', {})

# Dev_lift_ema (per-bullet stats)
dev_lift_ema = run_meta.get('per_bullet_dev_lift_ema', {})

# If history available, use it for trajectory plots
if 'history' not in globals() or not history:
    print("[VIZ] ⚠ Training history is empty; using eval_log only.")
    history = []

print(f"\n{'='*70}")
print(f"  VISUALIZATION — {globals().get('MODEL_TAG', '?')} ({_run_name})")
print(f"  Eval points  : {len(eval_log_data)}")
print(f"  Best Dev EA  : {best_pb_composite_ea:.4f} at step {best_pb_composite_step}")
print(f"  Diagnostic test EA: {final_test_ea:.4f}")
print(f"  Diagnostic test PA: {final_test_pa:.4f}")
print(f"  Bullets      : {run_meta.get('best_composite',{}).get('n_bullets', 0)}")
print(f"{'='*70}\n")


### Figure 1: Dev EA trajectory from eval_log


In [ ]:
if eval_log_data:
    steps_eval = [e['step'] for e in eval_log_data]
    ea_eval = [e['ea'] for e in eval_log_data]
    pa_eval = [e['pa'] for e in eval_log_data]
    pb_sizes_eval = [e.get('pb_size', 0) for e in eval_log_data]

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(steps_eval, ea_eval, color=COLORS['ea'], linewidth=2.5,
            marker='D', markersize=7, label='Dev EA')
    ax.plot(steps_eval, pa_eval, color=COLORS['pa'], linewidth=2.5,
            marker='s', markersize=7, label='Dev PA')
    ax.fill_between(steps_eval, pa_eval, ea_eval,
                     where=[e > p for e, p in zip(ea_eval, pa_eval)],
                     alpha=0.15, color='red', label='EA-PA gap')

    # Mark best point
    ax.scatter([best_pb_composite_step], [best_pb_composite_ea],
                marker='*', s=400, color='#1B5E20', zorder=10,
                edgecolors='black', linewidths=1.5,
                label=f'Best Dev EA={best_pb_composite_ea:.3f} @ step {best_pb_composite_step}')

    # Thesis publication-reference lines
    ax.axhline(y=0.6806, color='green', linestyle=':', alpha=0.5, label='Thesis EA=0.6806')
    ax.axhline(y=0.6190, color='blue', linestyle=':', alpha=0.5, label='Thesis PA=0.6190')

    ax.set_xlabel('Adaptation Step')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Figure 1: Dev Trajectory — {globals().get("MODEL_TAG", "?")} ({_run_name})')
    ax.legend(loc='lower right', fontsize=9)
    ax.set_ylim([0, 1.0])
    ax.grid(True, alpha=0.3)
    _save_fig(fig, '01_dev_trajectory')


### Figure 2: Dev vs Test final comparison


In [ ]:
if HAS_BASELINE or final_test_ea > 0:
    fig, ax = plt.subplots(figsize=(11, 5))
    cats = ['Baseline\n(step 0)' if HAS_BASELINE else None,
            'Best Dev\n(training)',
            f'Final Dev\n({run_meta.get("hyperparams", {}).get("ea_decimal_places", "")})',
            'Test\n(unseen)']
    cats = [c for c in cats if c is not None]

    eas = []
    pas = []
    if HAS_BASELINE:
        eas.append(baseline_ea)
        pas.append(run_meta.get('baseline_step0', {}).get('pa', 0))
    eas.append(best_pb_composite_ea)
    pas.append(best_pb_composite_pa)
    eas.append(final_dev_ea)
    pas.append(final_dev_pa)
    eas.append(final_test_ea)
    pas.append(final_test_pa)

    x = np.arange(len(cats))
    width = 0.35
    bars1 = ax.bar(x - width/2, eas, width, label='EA', color=COLORS['ea'])
    bars2 = ax.bar(x + width/2, pas, width, label='PA', color=COLORS['pa'])
    for bars in (bars1, bars2):
        for b in bars:
            h = b.get_height()
            ax.text(b.get_x() + b.get_width()/2, h + 0.01,
                    f'{h:.3f}', ha='center', fontsize=10, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(cats)
    ax.axhline(y=0.6806, color='green', linestyle=':', alpha=0.5)
    ax.axhline(y=0.6190, color='blue', linestyle=':', alpha=0.5)
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Figure 2: Pipeline Comparison — {_run_name}')
    ax.legend()
    ax.set_ylim([0, 1.05])
    ax.grid(True, axis='y', alpha=0.3)
    _save_fig(fig, '02_pipeline_comparison')


### Figure 3: Test outcome distribution


In [ ]:
if test_outcome_dist:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    labels = list(test_outcome_dist.keys())
    sizes = list(test_outcome_dist.values())
    colors_map = {'correct': COLORS['correct'], 'lucky_guess': COLORS['lucky'],
                   'exec_mismatch': COLORS['exec_mis'],
                   'wrong_reasoning': COLORS['wrong']}
    pie_colors = [colors_map.get(l, '#999') for l in labels]

    ax1.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90,
             colors=pie_colors)
    ax1.set_title('Test Outcome Distribution')

    sorted_oc = sorted(test_outcome_dist.items(), key=lambda x: -x[1])
    oc_labels = [o[0] for o in sorted_oc]
    oc_vals = [o[1] for o in sorted_oc]
    ax2.barh(oc_labels, oc_vals,
              color=[colors_map.get(l, '#999') for l in oc_labels])
    total_test = sum(oc_vals)
    for i, v in enumerate(oc_vals):
        ax2.text(v + 5, i, f'{v} ({v/total_test*100:.1f}%)',
                  va='center', fontweight='bold')
    ax2.set_xlabel('Count')
    ax2.set_title('Test Outcome Count')
    ax2.invert_yaxis()
    fig.suptitle(f'Figure 3: Test Outcome — {_run_name}',
                  fontsize=14, fontweight='bold')
    _save_fig(fig, '03_test_outcome')


### Figure 4: Failure diagnosis (training history)


In [ ]:
if diag_dist:
    total_fail = sum(diag_dist.values())
    fig, ax = plt.subplots(figsize=(11, 6))
    sorted_diag = sorted(diag_dist.items(), key=lambda x: -x[1])
    diag_labels = [d[0] for d in sorted_diag]
    diag_vals = [d[1] for d in sorted_diag]
    ax.barh(diag_labels, diag_vals, color=COLORS['wrong_value'])
    for i, v in enumerate(diag_vals):
        ax.text(v + 0.5, i, f'{v} ({v/total_fail*100:.1f}%)',
                  va='center', fontweight='bold')
    ax.set_xlabel('Count')
    ax.set_title(f'Figure 4: Training Failure Diagnosis — {_run_name}')
    ax.invert_yaxis()
    _save_fig(fig, '04_failure_diagnosis')


### Figure 5: Test EA by complexity


In [ ]:
if ea_by_steps_test_int:
    fig, ax = plt.subplots(figsize=(10, 5))
    xs = sorted(ea_by_steps_test_int.keys())
    vals = [ea_by_steps_test_int[x][1]/ea_by_steps_test_int[x][0]
            if ea_by_steps_test_int[x][0] else 0 for x in xs]
    totals = [ea_by_steps_test_int[x][0] for x in xs]
    bars = ax.bar([f'{n}-step' for n in xs], vals,
                   color=COLORS['exec'], edgecolor='white', linewidth=2)
    for bar, v, t in zip(bars, vals, totals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.3f}\n(n={t})', ha='center', fontweight='bold')
    ax.axhline(y=0.6806, color='green', linestyle=':', alpha=0.5)
    ax.set_ylabel('Diagnostic EA')
    ax.set_title(f'Figure 5: Diagnostic EA by complexity — {_run_name}')
    ax.set_ylim([0, 1.05])
    ax.grid(True, axis='y', alpha=0.3)
    _save_fig(fig, '05_test_ea_by_complexity')


### Figure 6: Playbook trajectory + cluster final


In [ ]:
if eval_log_data:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(steps_eval, pb_sizes_eval, color=COLORS['playbook'],
              linewidth=2.5, marker='o')
    ax1.fill_between(steps_eval, pb_sizes_eval, alpha=0.3, color=COLORS['playbook'])
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Bullets')
    ax1.set_title('Playbook Growth')
    ax1.grid(True, alpha=0.3)

    if cluster_dist_final:
        cluster_labels = list(cluster_dist_final.keys())
        cluster_vals = list(cluster_dist_final.values())
        ax2.pie(cluster_vals,
                 labels=[l.split('_')[0] for l in cluster_labels],
                 autopct='%1.0f%%', startangle=90)
        ax2.set_title('Final Cluster Distribution')
    fig.suptitle(f'Figure 6: Playbook — {_run_name}', fontsize=14, fontweight='bold')
    _save_fig(fig, '06_playbook')


### Figure 7: Per-bullet dev lift EMA


In [ ]:
if dev_lift_ema:
    fig, ax = plt.subplots(figsize=(11, 6))
    sorted_b = sorted(dev_lift_ema.items(),
                       key=lambda x: -x[1].get('pa_lift', 0))[:15]
    b_ids = [b[0] for b in sorted_b]
    pa_lifts = [b[1].get('pa_lift', 0) for b in sorted_b]
    ea_lifts = [b[1].get('ea_lift', 0) for b in sorted_b]

    y_pos = np.arange(len(b_ids))
    width = 0.4
    ax.barh(y_pos - width/2, pa_lifts, width,
              color=COLORS['pa'], label='PA-lift')
    ax.barh(y_pos + width/2, ea_lifts, width,
              color=COLORS['ea'], label='EA-lift')
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(b_ids)
    ax.set_xlabel('Dev Lift (EMA)')
    ax.set_title(f'Figure 7: Top Bullets by Dev Lift — {_run_name}')
    ax.legend()
    ax.grid(True, axis='x', alpha=0.3)
    ax.invert_yaxis()
    _save_fig(fig, '07_bullet_lifts')


### Figure 8: Verify-iterate stats


In [ ]:
if verify_stats:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    rounds_data = {
        'Round 1 pass': verify_stats.get('round_1_pass', 0),
        'Round 2 pass': verify_stats.get('round_2_pass', 0),
        'Round 3+ pass': verify_stats.get('round_3_pass', 0),
        'Exhausted all': verify_stats.get('exhausted_all_rounds', 0),
    }
    ax1.bar(rounds_data.keys(), rounds_data.values(),
              color=[COLORS['helpful'], '#8BC34A', '#FFC107', COLORS['harmful']])
    for i, (k, v) in enumerate(rounds_data.items()):
        ax1.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
    ax1.set_title('Verify-Iterate Rounds')
    ax1.set_ylabel('Count')
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=15, ha='right')

    cluster_assigned = verify_stats.get('cluster_assigned', {})
    if cluster_assigned:
        ax2.barh(list(cluster_assigned.keys()),
                  list(cluster_assigned.values()),
                  color=COLORS['exec'])
        for i, (k, v) in enumerate(cluster_assigned.items()):
            ax2.text(v + 0.1, i, str(v), va='center', fontweight='bold')
        ax2.set_xlabel('Count')
        ax2.set_title('Cluster Distribution (Verified)')

    fig.suptitle(f'Figure 8: Verify-Iterate — {_run_name}',
                  fontsize=14, fontweight='bold')
    _save_fig(fig, '08_verify_iterate')


### CSV exports


In [ ]:
main_metrics = [
    {'metric': 'Best Dev EA', 'value': f'{best_pb_composite_ea:.4f}'},
    {'metric': 'Best Dev PA', 'value': f'{best_pb_composite_pa:.4f}'},
    {'metric': f'Best step', 'value': f'{best_pb_composite_step}'},
    {'metric': 'Final Dev EA', 'value': f'{final_dev_ea:.4f}'},
    {'metric': 'Final Dev PA', 'value': f'{final_dev_pa:.4f}'},
    {'metric': 'Diagnostic EA', 'value': f'{final_test_ea:.4f}'},
    {'metric': 'Diagnostic PA', 'value': f'{final_test_pa:.4f}'},
    {'metric': 'Test None rate', 'value': f'{final_none_rate:.4f}'},
    {'metric': 'Bullets at best', 'value': f'{run_meta.get("best_composite",{}).get("n_bullets", 0)}'},
    {'metric': 'Tier 1 bullets', 'value': f'{len(run_meta.get("tier1_bullets_final", []))}'},
    {'metric': 'Cluster diversity', 'value': f'{len(cluster_dist_final)}'},
    {'metric': 'Counterfactual triggered',
     'value': f'{run_meta.get("counterfactual_stats", {}).get("triggered", 0)}'},
    {'metric': 'Verify pass rate',
     'value': f'{verify_stats.get("verify_pass_count", 0)}/{verify_stats.get("total_attempts", 1)}'},
    {'metric': 'API calls', 'value': f'{api_cost.get("calls", 0)}'},
    {'metric': 'Duration (min)', 'value': f'{run_meta.get("duration_min", 0)}'},
]
if HAS_BASELINE:
    main_metrics.insert(0, {'metric': 'Baseline step 0 EA',
                              'value': f'{baseline_ea:.4f}'})
pd.DataFrame(main_metrics).to_csv(f"{VIZ_DIR}/main_metrics.csv", index=False)

if eval_log_data:
    pd.DataFrame([{
        'step': e['step'], 'ea': round(e['ea'], 4), 'pa': round(e['pa'], 4),
        'score': round(e.get('score', 0), 4),
        'pb_size': e.get('pb_size', 0),
        'tier1_size': e.get('tier1_size', 0),
        'clusters': len(e.get('cluster_distribution', {})),
    } for e in eval_log_data
    ]).to_csv(f"{VIZ_DIR}/eval_trajectory.csv", index=False)

if diag_dist:
    total_fail = sum(diag_dist.values())
    pd.DataFrame([
        {'diagnosis': k, 'count': v, 'percentage': f'{v/total_fail*100:.2f}'}
        for k, v in sorted(diag_dist.items(), key=lambda x: -x[1])
    ]).to_csv(f"{VIZ_DIR}/failure_diagnosis.csv", index=False)

if dev_lift_ema:
    pd.DataFrame([{
        'bullet_id': bid,
        'pa_lift': round(info.get('pa_lift', 0), 4),
        'ea_lift': round(info.get('ea_lift', 0), 4),
        'n_evals': info.get('n_evals', 0),
        'last_step': info.get('last_step', 0),
        'tier1': bid in run_meta.get('tier1_bullets_final', []),
    } for bid, info in sorted(dev_lift_ema.items(),
                                 key=lambda x: -x[1].get('pa_lift', 0))
    ]).to_csv(f"{VIZ_DIR}/bullet_lifts.csv", index=False)

if ea_by_steps_test_int:
    pd.DataFrame([
        {'n_steps': n, 'total': t, 'correct': c,
         'ea': f'{c/t:.4f}' if t else '0', 'source': 'test'}
        for n, (t, c) in sorted(ea_by_steps_test_int.items())
    ]).to_csv(f"{VIZ_DIR}/test_ea_by_complexity.csv", index=False)


### Diagnostic report export


In [ ]:
report = [
    f"# ACE-FinQA run diagnostics — {globals().get('MODEL_TAG', '?')} ({_run_name})",
    f"",
    f"> **Metric provenance:** `notebook-diagnostic`, oracle `gold_inds` context. "
    f"Publish results only after evaluation with `ace-finqa evaluate`.",
    f"",
    f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"**Method:** Cluster-aware ACE + Tier 1 promotion + Dev lift tracking",
    f"",
    f"## Configuration",
    f"",
    f"| Parameter | Value |",
    f"|-----------|-------|",
    f"| Model | `{globals().get('MODEL_NAME', '?')}` |",
    f"| Mode | {run_meta.get('experiment_config', {}).get('mode', '?')} |",
    f"| Train subset | {run_meta.get('train_subset_size', 0)} |",
    f"| Total steps | {run_meta.get('total_steps', 0)} |",
    f"| Epochs entered / configured | {run_meta.get('completed_epochs', 0)}/{run_meta.get('num_epochs_target', 0)} |",
    f"| Reflector | {run_meta.get('experiment_config', {}).get('reflector_model', '?')} |",
    f"| TOP_K Retrieval | {run_meta.get('hyperparams', {}).get('top_k_retrieval', '?')} |",
    f"| Duration | {run_meta.get('duration_min', 0)} min |",
    f"",
    f"## Notebook diagnostic metrics",
    f"",
    f"| Metric | Value |",
    f"|--------|-------|",
]
if HAS_BASELINE:
    report.append(f"| Baseline step 0 EA | {baseline_ea:.4f} |")
report.extend([
    f"| Best Dev EA (step {best_pb_composite_step}) | **{best_pb_composite_ea:.4f}** |",
    f"| Best Dev PA (step {best_pb_composite_step}) | **{best_pb_composite_pa:.4f}** |",
    f"| **Diagnostic test EA** | **{final_test_ea:.4f}** |",
    f"| **Diagnostic test PA** | **{final_test_pa:.4f}** |",
    f"| Test None rate | {final_none_rate:.4f} |",
    f"| Bullets at best | {run_meta.get('best_composite',{}).get('n_bullets', 0)} |",
    f"| Tier 1 promoted | {len(run_meta.get('tier1_bullets_final', []))} |",
    f"| Cluster diversity | {len(cluster_dist_final)} clusters |",
    f"",
])

# Comparison with the thesis publication record
report.extend([
    f"## Thesis reference comparison",
    f"",
    f"| Metric | Thesis reference | Diagnostic run | Status |",
    f"|--------|-----------|----------|--------|",
    f"| Diagnostic EA | 0.6806 | {final_test_ea:.4f} | "
    f"{'✅ PASSED' if final_test_ea >= 0.6806 else f'❌ MISSED by {(0.6806-final_test_ea)*100:.2f}pp'} |",
    f"| Diagnostic PA | 0.6190 | {final_test_pa:.4f} | "
    f"{'✅ PASSED' if final_test_pa >= 0.6190 else f'❌ MISSED by {(0.6190-final_test_pa)*100:.2f}pp'} |",
    f"",
])

# Test EA by complexity
if ea_by_steps_test_int:
    report.extend([
        f"## Diagnostic EA by complexity",
        f"",
        f"| Steps | Total | Correct | EA |",
        f"|-------|-------|---------|-----|",
    ])
    for n, (t, c) in sorted(ea_by_steps_test_int.items()):
        report.append(f"| {n}-step | {t} | {c} | {c/t:.3f} |")
    report.append("")

# Failure diagnosis
if diag_dist:
    total_fail = sum(diag_dist.values())
    report.extend([
        f"## Training Failure Diagnosis",
        f"",
        f"| Diagnosis | Count | % |",
        f"|-----------|-------|---|",
    ])
    for k, v in sorted(diag_dist.items(), key=lambda x: -x[1]):
        report.append(f"| `{k}` | {v} | {v/total_fail*100:.1f}% |")
    report.append("")

# Multistage stats
if multistage_stats:
    report.extend([
        f"## Multi-Stage Curator Stats",
        f"",
        f"| Stage | Metric | Count |",
        f"|-------|--------|-------|",
        f"| Stage 0.5 | Cluster quota full | {multistage_stats.get('stage0_5_cluster_quota_full', 0)} |",
        f"| Stage 1 | Quality gate passed | {multistage_stats.get('stage1_pass', 0)} |",
        f"| Stage 2 | Eval count | {multistage_stats.get('stage2_eval', 0)} |",
        f"| Stage 2 | Accept | {multistage_stats.get('stage2_accept', 0)} |",
        f"| Stage 2 | Reject (no improve) | {multistage_stats.get('stage2_reject_no_improvement', 0)} |",
        f"| Stage 2 | Reject (harmful) | {multistage_stats.get('stage2_reject_harmful', 0)} |",
        f"| Stage 3 | Pass dedup | {multistage_stats.get('stage3_pass', 0)} |",
        f"| Final | Accept | {multistage_stats.get('final_accept', 0)} |",
        f"| Auto-ablate | Calls | {multistage_stats.get('auto_ablate_calls', 0)} |",
        f"| Auto-ablate | Evicted | {multistage_stats.get('auto_ablate_evicted', 0)} |",
        f"| Tier 1 | Promotions | {multistage_stats.get('tier1_promotions', 0)} |",
        f"| Tier 1 | Fallback promotions | {multistage_stats.get('tier1_fallback_promotions', 0)} |",
        f"| Lift evals | Rounds run | {multistage_stats.get('lift_evals_run', 0)} |",
        f"",
    ])

# Verify stats
if verify_stats:
    report.extend([
        f"## Verify-Iterate Stats",
        f"",
        f"| Metric | Count |",
        f"|--------|-------|",
        f"| Total attempts | {verify_stats.get('total_attempts', 0)} |",
        f"| Round 1 pass | {verify_stats.get('round_1_pass', 0)} |",
        f"| Round 2 pass | {verify_stats.get('round_2_pass', 0)} |",
        f"| Round 3+ pass | {verify_stats.get('round_3_pass', 0)} |",
        f"| Total pass | {verify_stats.get('verify_pass_count', 0)} |",
        f"| Exhausted all rounds | {verify_stats.get('exhausted_all_rounds', 0)} |",
        f"| Fewshot used | {verify_stats.get('fewshot_used', 0)} |",
        f"| Reflector skipped (lucky) | {verify_stats.get('reflector_skipped_lucky', 0)} |",
        f"| Reflector skipped (correct) | {verify_stats.get('reflector_skipped_correct', 0)} |",
        f"",
    ])

# Counterfactual
cf_stats = run_meta.get('counterfactual_stats', {})
if cf_stats:
    report.extend([
        f"## Counterfactual Reflection",
        f"",
        f"- Triggered: **{cf_stats.get('triggered', 0)}**",
        f"- Cases found: {cf_stats.get('cases_found', 0)}",
        f"",
    ])

# Final bullets
if dev_lift_ema:
    report.extend([
        f"## Top Bullets by Dev Lift (EMA)",
        f"",
        f"| Bullet ID | PA-lift | EA-lift | n_evals | Tier 1 |",
        f"|-----------|---------|---------|---------|--------|",
    ])
    tier1_set = set(run_meta.get('tier1_bullets_final', []))
    for bid, info in sorted(dev_lift_ema.items(),
                               key=lambda x: -x[1].get('pa_lift', 0))[:10]:
        report.append(
            f"| `{bid}` | {info.get('pa_lift', 0):+.4f} | "
            f"{info.get('ea_lift', 0):+.4f} | {info.get('n_evals', 0)} | "
            f"{'🔒' if bid in tier1_set else ''} |")
    report.append("")

# Cluster final
if cluster_dist_final:
    report.extend([
        f"## End-of-training playbook cluster distribution",
        f"",
        f"This terminal state may differ from the selected winning checkpoint.",
        f"",
        f"| Cluster | Bullets |",
        f"|---------|---------|",
    ])
    for cid, n in sorted(cluster_dist_final.items(), key=lambda x: -x[1]):
        report.append(f"| `{cid}` | {n} |")
    report.append("")

report_path = f"{VIZ_DIR}/report.md"
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(report))


### Summary


In [ ]:
print(f"\n{'='*70}")
print(f"  ✅ VISUALIZATION COMPLETE — {VIZ_DIR}/")
print(f"  Figures: 8 PNG (300 dpi)")
print(f"  CSVs: main_metrics, eval_trajectory, failure_diagnosis,")
print(f"        bullet_lifts, test_ea_by_complexity")
print(f"  Report: {report_path}")
print(f"")
print(f"  Diagnostic test EA = {final_test_ea:.4f} (thesis 0.6806: "
       f"{'✅' if final_test_ea >= 0.6806 else '❌ -' + f'{(0.6806-final_test_ea)*100:.2f}pp'})")
print(f"  Diagnostic test PA = {final_test_pa:.4f} (thesis 0.6190: "
       f"{'✅' if final_test_pa >= 0.6190 else '❌ -' + f'{(0.6190-final_test_pa)*100:.2f}pp'})")
print(f"{'='*70}")
